# 05-02. 탐지 구간 분류 및 분석 대상 선별

**담당:** 정수진 · **Issue:** #65

[01](05_all_data_timeseries_jinny_01.ipynb)의 기준으로 탐지한 구간 가운데 **근거가 있어 확대하지 않아도 되는 것**을 걷어내고, 남은 것만 [03](05_all_data_timeseries_jinny_03.ipynb)으로 넘긴다. 지우지 않고 근거와 함께 남긴다. 항목마다 "보인 것 → 왜 이상으로 보기 어려운가 → 데이터 근거 → 처리" 순서로 본다.

## 분석 보고서 구성 및 읽는 흐름

| 파일 | 읽는 흐름 |
|---|---|
| **[01 분석 개요 및 판정 기준](05_all_data_timeseries_jinny_01.ipynb)** | 핵심 결과 요약 → 분석 대상 및 자료 범위 → 연간 흐름·월별 관측 구간 → 운전·정비 이력 → 품질 및 데이터 신뢰성 검토 → 판정 기준(운전 상태·평상시 대비 변동 수준·반복성 및 우연 변동 가능성) |
| **[02 탐지 구간 분류 및 분석 대상 선별](05_all_data_timeseries_jinny_02.ipynb)** | 탐지 구간 전체 현황 → 운전 상태별 분류(정지·승온) → 공정 위치·신호 특성 검토(Z3·세정 입구·소규모 CP·Z9 및 냉각) → 분석 제외·판단 보류·정밀 분석 대상 구분 및 근거 → 중점 분석 대상 우선순위 |
| **[03 주요 이상 의심 구간 상세 분석 및 비교](05_all_data_timeseries_jinny_03.ipynb)** | 핵심 결과 요약 → 제공 14 LOT의 CP 특성 및 검토 기준 → 주요 LOT(240069·240037·240117) 구간 확대 → 가스 분석계 셀 관련 신호 검토 → 동일 강종 비교 및 공정 조건 검토(Z7·Z8, 출력 Z4) → 정기수리 전후 변화 비교 → 주요 분석 결과 정리 |
| **[04 종합 판정 및 후속 검증 계획](05_all_data_timeseries_jinny_04.ipynb)** | 14 LOT × 공정별 종합 판정 → 주요 판단 근거 표 → 이상 원인 가설 및 불확실성 → 추가 확인 항목·검증 방법 → 후속 조치 및 판정 변경 조건 → 부록: 탐지 구간 전수 및 판정 근거 |
| **[05 상세 분석 근거 및 14 LOT 전수 자료](05_all_data_timeseries_jinny_05.ipynb)** | LOT별 전체 22개 신호 현황 → 공정 흐름 및 주요 구간 확대 → 강종별 전체 비교 → Z9 주기 특성 → 정비 이력 상세 → LOT별 선별 구간·비교 근거와 본문 판정 연결 |

대상은 CAL01의 제공된 14 LOT·22개 센서와 CAL01 정비/같은 WO의 부품교체, CR1 강종변경·정기수리다. 관측 범위는 실제 가동시간이나 소재 입·출측 시간이 아니다. 같은 강종·같은 경과 분은 비교 기준이며 같은 설정값·속도·두께를 보장하지 않는다. CP-Z4와 CP-Z4M의 단위, 물리적 정의와 산식은 미확인이다. 시간 겹침·선후관계만으로 고장 원인이나 정비 효과를 확정하지 않는다.

**이 파일에서 바로 보기:** **[결론](#funnel)** · [1. 정지·승온](#stop-restart) · [2. 탐지 구간 분류](#classify) · [3. Z3](#z3) · [4. 세정 입구](#cln) · [5. 작은 CP](#cp-small) · [6. Z9·냉각](#cycle) · [7. 판단 보류](#hold) · **[8. 남은 것](#remaining)** · [부록](#appendix-02)

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC는 ℃, OP는 %로 표시합니다(팀에서 사용하는 해석). 목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4와 CP-Z4M의 단위는 미확인이며 원본 기록값을 그대로 표시합니다.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
from IPython.display import display

available_fonts = []
for font in font_manager.fontManager.ttflist:
    available_fonts.append(font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if font_name in available_fonts:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False

os.makedirs(FIG_DIR, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
rcParams["figure.dpi"] = 110


In [ ]:
"""Shared presentation rules for the five CAL01 time-series notebooks.

Only artists and display labels are changed here. Sensor values, thresholds,
aggregation windows and axis limits belong to the notebooks.
"""


import html
import re
from pathlib import Path

from matplotlib import rcParams
from matplotlib.lines import Line2D

EXPORT_DPI = 180
EXPORT_PADDING = 0.20
INK = '#1F2937'
MUTED = '#64748B'

# A color always identifies the same member of the same domain. Different
# domains use separate legends. LOT colors contrast most within the same grade.
PROCESS_COLORS = {'세정': '#2563EB', '건조': '#D97706', '가열·균열': '#DC4C3E', '과시효': '#059669', '냉각': '#7C3AED'}
GRADE_COLORS = {'SGCC': '#B9D6D9', 'SPFH590': '#B4634E', 'SPCE': '#3F909A', 'SPCEN': '#8676AC', 'SPHC': '#A48258', 'DP590': '#2B527A', 'SPCD': '#3E75A8', 'SPCC': '#9AA7B5'}
LOT_COLORS = {240001: '#334155', 240024: '#2563EB', 240108: '#F97316', 240037: '#1D4ED8', 240121: '#D946EF', 240133: '#10B981', 240038: '#0891B2', 240114: '#DC2626', 240044: '#0F766E', 240061: '#D97706', 240069: '#7C3AED', 240091: '#DB2777', 240058: '#92400E', 240117: '#475569'}
MAINTENANCE_COLORS = {'BM': '#C81E3A', 'CBM': '#6D28D9', 'TBM': '#0E7490'}
MAINTENANCE_LABELS = {"BM": "BM · 고장 후 정비", "CBM": "CBM · 상태 기준 정비", "TBM": "TBM · 주기 기준 정비"}
MAINTENANCE_HATCHES = {"BM": "///", "CBM": "...", "TBM": "xxx"}
ZONE_COLORS = {1: '#475569', 2: '#B45309', 3: '#7C3AED', 4: '#2563EB', 5: '#F97316', 6: '#059669', 7: '#DB2777', 8: '#0891B2', 9: '#A16207'}
SENSOR_COLORS = {'CLN-IN': '#0F766E', 'CP-Z4': '#E11D48', 'CP-Z4M': '#64748B', 'TC-OUT1': '#4F46E5', 'TC-OUT2': '#CA8A04'}

# LOT is encoded by color. Measured OP uses a dashed line; TC/CP use solid lines.
LOT_LINESTYLES = {240001: '-', 240024: '-', 240037: '-', 240038: '-', 240044: '-', 240058: '-', 240061: '-', 240069: '-', 240091: '-', 240108: '-', 240114: '-', 240117: '-', 240121: '-', 240133: '-'}

# Full original explanations remain available immediately below each figure.
# Short figure text describes the target, comparison rule and reading purpose.
_HEADINGS = {
    "연간 공정 흐름": ("2024년 CAL01 관측·정비 흐름", "2024년 관측창과 정비 시각을 대조해 비교 범위 확인"),
    "LOT별 자료 상태 · 분석에 쓸 수 있는 구간과 빠진 곳": ("CAL01 14 LOT 관측 품질 비교", "LOT별 기록량과 결측을 대조해 분석 가능 구간 확인"),
    "측정 기록 공백·센서 NaN 비교": ("CAL01 측정 공백·센서 결측 비교", "같은 LOT의 TC·OP를 대조해 결측 종류 구분"),
    "운전 상태별 급변 — 정지·승온을 따로 떼어야 하는 이유": ("CAL01 운전 상태별 급변 크기", "신호별 안정 운전 기준으로 정지·승온의 급변 비교"),
    "정비 일정 전후 정지·재가동 — 설명됨의 근거": ("CAL01 정비 전후 정지·재가동", "14 LOT의 온도·출력을 일정과 대조해 전환 구간 확인"),
    "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳": ("CAL01 14 LOT의 CP 기준 이탈", "CP 편차(단위 미확인) ±0.05 이탈의 위치·횟수 비교"),
    "탐지된 구간이 어디로 갔나 — 근거가 있어 빼는 것과 03으로 넘기는 것": ("CAL01 탐지 구간의 검토 대상 선별", "탐지 구간을 설명 근거별로 나눠 상세 검토 대상 확인"),
    "안정 참고 구간 · 가열·균열 Z3 수준": ("CAL01 강종별 Z3 온도·출력 수준", "같은 강종의 안정 참고 중앙값으로 LOT 수준 비교"),
    "안정 참고 구간 · Z3 제어출력과 온도를 같이 보기": ("CAL01 LOT별 Z3 온도·출력 반응", "LOT별 안정 참고 구간에서 온도와 출력의 동행 확인"),
    "출력이 0%·100%에 붙을 때 Z3 온도는 어떻게 움직이나": ("CAL01 Z3 출력 한계·온도 반응", "출력 0%·100% 지속 구간의 온도 반응을 함께 확인"),
    "세정 입구 온도 (CLN-IN)": ("CAL01 LOT별 세정 입구 온도", "LOT 중앙값과 같은 강종의 흐름으로 세정 온도 비교"),
    "정비 전후 흐름 · LOT 사이 정비와 LOT별 상태 지표": ("CAL01 정비 시각·LOT 지표 비교", "관측 날짜 순 지표와 LOT 사이 정비의 시간 관계 확인"),
    "볼 것 고르기 — 평소 흔들림 대비 편차": ("CAL01 평소 변동 대비 검토 우선순위", "신호별 평소 변동으로 나눈 편차로 검토 대상 비교"),
    "CP의 평소 관계 — 14 LOT에서 먼저 확인": ("CAL01 14 LOT의 CP 동시 관계", "안정 참고 1초 변화 상관으로 CP와 센서의 동행 확인"),
    "어느 LOT이 CP의 평소 관계에서 벗어났나 — 14 LOT 비교": ("CAL01 LOT별 CP 변동·동조 비교", "안정 참고 구간의 이탈·변동·동조로 검토 LOT 비교"),
    "CP 관점에서 확인할 LOT · 같은 강종부터 비교": ("CAL01 강종별 CP 검토 LOT 비교", "같은 강종의 시간당 이탈 빈도와 크기로 후보 비교"),
    "가스 분석계 셀 교체와 CP 신호 상태": ("CAL01 분석계 셀 교체·CP 상태", "CP 계측 부품의 교체 시각과 시간별 신호 상태 대조"),
    "같은 강종끼리 대조한 LOT별 요약 · 과시효 Z7·Z8": ("CAL01 동일 강종의 Z7·Z8 비교", "같은 강종 다른 LOT 중앙값 대비 온도·출력 차이 확인"),
    "14 LOT × 공정 · 확인 이슈와 근거 한눈에 보기": ("CAL01 14 LOT 공정별 검토 근거", "공정별 비교 가능성과 근거로 추가 확인할 LOT 선별"),
    "과시효 Z7·Z8 · 같은 강종 LOT끼리 비교": ("CAL01 동일 강종의 과시효 온도 비교", "안정 참고 1분 중앙값 분포로 LOT 수준 차이 확인"),
}


def configure():
    rcParams.update({
        "font.size": 11.5, "axes.titlesize": 13, "axes.labelsize": 11.5,
        "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
        "savefig.dpi": EXPORT_DPI, "savefig.pad_inches": EXPORT_PADDING,
        "svg.fonttype": "none", "pdf.fonttype": 42,
    })


def sensor_color(sensor):
    if sensor.startswith(("TC-Z", "OP-Z")):
        return ZONE_COLORS[int(sensor.rsplit("Z", 1)[1])]
    return SENSOR_COLORS[sensor]


def lot_linestyle(lot):
    return '-'


def process_title_color(title, default=INK):
    return INK


def palette_preview(path):
    """Read the guide in order: signal type, zone color, then compared LOTs."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from IPython.display import display
    fig = plt.figure(figsize=(16, 10.2), facecolor="#F5F7FB")
    fig.text(.045, .955, "CAL01 시계열 · 색상과 선 표기", fontsize=25, weight="bold", color=INK, va="top")
    fig.text(.045, .909, "TC는 실선 · OP는 점선  |  같은 존은 같은 색  |  LOT 비교는 같은 강종 안에서 색으로 구분", fontsize=12, color=MUTED, va="top")

    def card(x, y, w, h, title, subtitle=""):
        fig.add_artist(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.008,rounding_size=0.012",
                       transform=fig.transFigure,fc="white",ec="#E2E8F0",lw=.8,zorder=0))
        ax=fig.add_axes([x+.014,y+.014,w-.028,h-.028],zorder=1)
        ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis("off")
        ax.text(0,.96,title,va="top",fontsize=14,weight="bold",color=INK)
        if subtitle:ax.text(0,.79,subtitle,va="top",fontsize=10.5,color=MUTED)
        return ax

    ax=card(.045,.661,.438,.207,"1  선 모양 = 측정 신호","TC·OP는 색을 바꾸지 않고 선 모양으로 구분")
    for y,s,label in [(.55,"TC-Z4","TC 온도 (℃) · 실선"),(.29,"OP-Z4","OP 제어출력 (%) · 점선")]:
        ax.plot([.015,.27],[y,y],color=ZONE_COLORS[4],lw=2.8,ls=signal_linestyle(s))
        ax.text(.32,y,label,va="center",fontsize=12.5,color=INK)
    ax.text(.015,.06,"CP는 실선 · 단위 미확인",fontsize=10.5,color=MUTED)

    ax=card(.504,.661,.451,.207,"2  선 색 = 존(ZONE)","같은 ZONE의 TC와 OP는 아래 두 선처럼 같은 색")
    for i,(z,col) in enumerate(ZONE_COLORS.items()):
        row,column=divmod(i,3);x=column*.337;y=.56-row*.20
        ax.text(x,y,f"Z{z}",va="center",fontsize=11.5,color=INK)
        ax.plot([x+.085,x+.175],[y+.027,y+.027],color=col,lw=2.1)
        ax.plot([x+.085,x+.175],[y-.027,y-.027],color=col,lw=2.1,ls="--")

    for x,w,title,colors in [(.045,.291,"공정 · 축 옆 구획색",PROCESS_COLORS),
                              (.357,.291,"강종 · 그룹 표식",{k:GRADE_COLORS[k] for k in ["SPCC","SPCD","SPCE","SPCEN","SPHC","SPFH590","DP590","SGCC"]}),
                              (.669,.286,"정비 · 실제 작업 구간",MAINTENANCE_COLORS)]:
        ax=card(x,.476,w,.151,title)
        columns=3 if len(colors)>5 else 2 if len(colors)>3 else 3
        for i,(label,col) in enumerate(colors.items()):
            row,column=divmod(i,columns);xx=column/columns;yy=.61-row*.235
            ax.plot([xx+.01,xx+.048],[yy,yy],color=col,lw=6,solid_capstyle="round")
            ax.text(xx+.073,yy,label,va="center",fontsize=9.8,color=INK)
        if title.startswith("정비"):
            ax.text(.01,.18,"BM 고장후  ·  CBM 상태기준  ·  TBM 주기기준",fontsize=9,color=MUTED)

    fig.text(.045,.437,"3  LOT 비교 · 아래 같은 강종 안에서 선 색을 대조",fontsize=14,weight="bold",color=INK)
    groups=[("SGCC",[240001]),("SPFH590",[240024,240108]),("SPCE",[240037,240121,240133]),
            ("SPCEN",[240038,240114]),("SPHC",[240044,240061,240069,240091]),
            ("DP590",[240058]),("SPCD",[240117])]
    for i,(grade,ids) in enumerate(groups):
        row,column=divmod(i,4);x=.045+column*.232;y=.237-row*.196
        ax=card(x,y,.213,.171,grade)
        for j,k in enumerate(ids):
            yy=.69-j*.175
            ax.plot([.015,.29],[yy,yy],color=LOT_COLORS[k],lw=2.6,ls="-")
            ax.text(.35,yy,str(k),va="center",fontsize=11.5,color=INK)
    ax=card(.741,.041,.213,.171,"비교 그림 읽는 법")
    ax.text(.01,.67,"대상 LOT  굵게\n대조 LOT  얇게\nTC 실선 / OP 점선",va="top",fontsize=11.5,color=MUTED,linespacing=1.65)
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=EXPORT_DPI,facecolor=fig.get_facecolor(),bbox_inches="tight",pad_inches=.18)
    fig.set_dpi(72);display(fig);plt.close(fig)



def signal_linestyle(signal):
    return "--" if str(signal).startswith("OP-") else "-"


def _signal_kind(label, axis_text=""):
    label=str(label)
    if re.search(r"\bOP-Z\d",label) or label.startswith("OP "):return "OP"
    if re.search(r"\bTC-(?:Z\d|OUT)",label) or label=="CLN-IN" or label.startswith("TC "):return "TC"
    if label.startswith("CP-"):return "CP"
    if "출력" in label and "온도" not in label:return "OP"
    # A LOT label carries color identity; its panel supplies the measurement.
    if "제어출력" in axis_text or "출력[%]" in axis_text or "[%]" in axis_text:return "OP"
    if "온도" in axis_text or "[℃]" in axis_text:return "TC"
    if "단위 미확인" in axis_text and "CP" in axis_text:return "CP"
    return None


def style_signal_lines(fig):
    """Color identifies a zone/LOT; a dashed measured curve identifies OP."""
    for ax in fig.axes:
        titles=" ".join([ax.title.get_text(),ax._left_title.get_text(),ax._right_title.get_text()])
        context=ax.get_ylabel()+" "+titles
        for line in ax.lines:
            if len(line.get_xdata())<3 or line.get_linestyle() in ["None","none","",None]:continue
            if line.get_transform()!=ax.transData:continue
            label=line.get_label();kind=_signal_kind(label,context)
            if kind:line.set_linestyle("--" if kind=="OP" else "-")
            if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",str(label)):
                line.set_color(sensor_color(label))
        # A restrained axis rail carries process context; titles stay dark.
        found=[p for p in PROCESS_COLORS if p in titles or (p=="가열·균열" and ("가열" in titles or "균열" in titles))]
        if len(found)==1 and ax.axison:
            ax.spines["left"].set_color(PROCESS_COLORS[found[0]])
            ax.spines["left"].set_alpha(.65)
            ax.spines["left"].set_linewidth(1.0)
        for title in [ax.title,ax._left_title,ax._right_title]:
            if title.get_text() and str(title.get_color()).lower() not in ["#c83745","#d62728","#e11d48"]:
                title.set_color(INK)
        legend=ax.get_legend()
        if legend is not None:_style_legend(legend,context)
    for legend in fig.legends:_style_legend(legend)


def _style_legend(legend, context=""):
    handles=getattr(legend,"legend_handles",getattr(legend,"legendHandles",[]))
    for handle,text in zip(handles,legend.get_texts()):
        if not isinstance(handle,Line2D) or handle.get_linestyle() in ["None","none","",None]:continue
        label=text.get_text()
        if any(word in label for word in ["기준","중앙값","경계","감지","정비","수리","범위"]):continue
        kind=_signal_kind(label,context)
        if kind:handle.set_linestyle("--" if kind=="OP" else "-")
        if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",label):
            handle.set_color(sensor_color(label))


def _heading(title, subtitle):
    if title in _HEADINGS:
        return _HEADINGS[title]
    patterns = [
        (r"CAL01 정비 이력 (\d+)건", lambda m: (f"CAL01 정비 {m[1]}건의 작업 흐름", "작업별 감지·착수·완료와 관측창을 대조해 해석 범위 확인")),
        (r"남은 짧은 변화 · (\d+)", lambda m: (f"LOT {m[1]} 탐지 구간과 반복 파형 대조", "같은 LOT의 1초 원본으로 탐지 구간의 동시 변화 확인")),
        (r"같은 강종 LOT 전체 비교 · (\w+) · (.+)", lambda m: (f"{m[1]} LOT별 공정 신호 비교", "같은 강종·같은 신호의 경과시간으로 LOT 간 차이 확인")),
        (r"같은 강종 LOT 비교 · (\w+) · 대상 (\d+) · Z4", lambda m: (f"LOT {m[2]} 동일 강종 Z4 비교", "안정 참고 시작을 맞춰 같은 강종의 온도·출력 비교")),
        (r"(\d+) 정비 전·중·후", lambda m: (f"LOT {m[1]} 정비 전후 신호 비교", "같은 LOT의 실제 작업 경계와 온도·출력 변화를 대조")),
        (r"(\d+) CP 변화·정비", lambda m: (f"LOT {m[1]} CP·정비 시각 대조", "같은 LOT의 CP 극값과 정비·교체 기록의 시간 관계 확인")),
        (r"(\d+) · 언제 CP", lambda m: (f"LOT {m[1]} CP 동조 변화 구간", "안정 참고의 시간별 변동과 1초 동조로 변화 시점 확인")),
        (r"CP 펄스 뒤 무엇이 달라졌나 · (.+)", lambda m: (f"CAL01 CP 펄스 전후 반응 {m[1]}", "같은 LOT의 펄스 없는 안정 참고 시각과 전후 반응 비교")),
    ]
    for pattern, make in patterns:
        match = re.search(pattern, title)
        if match:
            return make(match)
    return title, subtitle


def header(fig, title, subtitle="", handles=(), ncol=4):
    original_title, original_subtitle = title, subtitle
    title, subtitle = _heading(title, subtitle)
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=23, fontweight="bold", color=INK)
    sub = fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=12, color=MUTED)
    fig._report_subtitle = sub
    fig._report_original = {"title": original_title, "subtitle": original_subtitle}
    fig._report_details = []
    if handles:
        # Legend order is the caller's comparison order, never an alphabetic sort.
        labels = [h.get_label() for h in handles]
        fig.legend(handles=handles, labels=labels, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=11, handlelength=2.4,
                   columnspacing=1.5, labelspacing=.65)


def _panel_titles(fig, renderer):
    """Keep the identifying clause on the chart and retain the full caption."""
    for ax in fig.axes:
        title = ax._left_title if ax._left_title.get_text() else ax.title
        original = title.get_text()
        if title.get_color().lower() in [INK, "#17344d"]:
            title.set_color(process_title_color(original, title.get_color()))
        if not original or "\n" in original:
            continue
        if len(original) <= 66 and title.get_window_extent(renderer).width <= ax.bbox.width:
            continue
        # A long explanatory clause belongs below the figure. Do not cut a
        # label in the middle of a sensor name, quantity or comparison value.
        short = re.split(r" — | · (?=회색|위 |아래 |점 =|빨간|파란|평소|숫자|칸 =|세로)| \| ", original, maxsplit=1)[0]
        if short == original or len(short) > 66:
            continue
        fig._report_details.append(original)
        title.set_text(short)


def _legend_outside(fig, renderer):
    """Move an inside legend to a measured free title-row space when it fits."""
    moved = 0
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None or not leg.get_window_extent(renderer).overlaps(ax.bbox):
            continue
        handles, labels = ax.get_legend_handles_labels()
        if not labels:
            continue
        candidate = ax.legend(handles, labels, loc="lower right", bbox_to_anchor=(1, 1.025),
                              borderaxespad=0, ncol=min(4, len(labels)), frameon=False,
                              fontsize=10.5, handlelength=2, columnspacing=1.2)
        fig.canvas.draw()
        box = candidate.get_window_extent(renderer)
        obstacles = [a.bbox for a in fig.axes if a is not ax and a.bbox != ax.bbox]
        texts = list(fig.texts)
        for other in fig.axes:
            texts += [other._left_title, other._right_title, other.title, other.xaxis.label]
            texts += list(other.texts)
        obstacles += [t.get_window_extent(renderer) for t in texts if t.get_visible() and t.get_text()]
        if any(box.overlaps(b) for b in obstacles) or box.y1 > fig.bbox.y1:
            candidate.remove()
            ax.add_artist(leg)
            ax.legend_ = leg
        else:
            candidate._detail_legend = True
            moved += 1
    return moved


def prepare(fig):
    """Arrange the figure header in physical units, preserving data limits."""
    if getattr(fig, "_report_prepared", False):
        return
    fig._report_prepared = True
    style_signal_lines(fig)
    fig._report_details = getattr(fig, "_report_details", [])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    _panel_titles(fig, renderer)
    if hasattr(fig, "_report_subtitle"):
        height = fig.bbox.height
        gap = 9 * fig.dpi / 72
        title = fig._suptitle
        title.set_y(1 - .20 / fig.get_figheight())
        title.set_fontsize(min(25, title.get_fontsize()))
        fig.canvas.draw()
        sub = fig._report_subtitle
        sub.set_position((.035, (title.get_window_extent(renderer).y0 - gap) / height))
        sub.set_fontsize(12)
        sub.set_va("top")
        fig.canvas.draw()
        cursor = sub.get_window_extent(renderer).y0 - gap
        for legend in fig.legends:
            legend.set_bbox_to_anchor((.031, cursor / height), transform=fig.transFigure)
            fig.canvas.draw()
            cursor = legend.get_window_extent(renderer).y0 - gap
        # Existing charts already allocate a header. Grow it only if the
        # measured text requires additional room, including secondary axes.
        if fig.axes:
            positions = {ax: ax.get_position().frozen() for ax in fig.axes}
            low = min(p.y0 for p in positions.values())
            high = max(p.y1 for p in positions.values())
            visible_top = max((ax.get_tightbbox(renderer).y1 for ax in fig.axes if ax.get_visible()), default=high*height)
            extra = max(0, visible_top - high*height)
            target = (cursor - extra) / height
            target = min(target, high + .55 / fig.get_figheight())
            if abs(target-high) > .003:
                scale = (target-low) / (high-low)
                for ax, p in positions.items():
                    ax.set_position([p.x0, low+(p.y0-low)*scale, p.width, p.height*scale])
                for text in fig.texts:
                    if text in [title, sub]:
                        continue
                    x, y = text.get_position()
                    if low <= y <= high:
                        text.set_position((x, low+(y-low)*scale))
    fig.canvas.draw()
    fig._report_legends_moved = _legend_outside(fig, fig.canvas.get_renderer())


def details_html(fig):
    original = getattr(fig, "_report_original", {})
    details = [original.get("subtitle", ""), *getattr(fig, "_report_details", [])]
    details = list(dict.fromkeys(text for text in details if text))
    if not details:
        return ""
    body = "".join(f"<p>{html.escape(text)}</p>" for text in details)
    return '<details data-report-visuals="caption"><summary>그림 상세 기준·해석 펼치기</summary>' + body + "</details>"


# 각 노트북 안에서 완결되는 공통 그림 설정
from types import SimpleNamespace
report_viz = SimpleNamespace(
    EXPORT_DPI=EXPORT_DPI,
    EXPORT_PADDING=EXPORT_PADDING,
    PROCESS_COLORS=PROCESS_COLORS,
    GRADE_COLORS=GRADE_COLORS,
    LOT_COLORS=LOT_COLORS,
    MAINTENANCE_COLORS=MAINTENANCE_COLORS,
    MAINTENANCE_LABELS=MAINTENANCE_LABELS,
    MAINTENANCE_HATCHES=MAINTENANCE_HATCHES,
    ZONE_COLORS=ZONE_COLORS,
    SENSOR_COLORS=SENSOR_COLORS,
    LOT_LINESTYLES=LOT_LINESTYLES,
    header=header,
    prepare=prepare,
    details_html=details_html,
    sensor_color=sensor_color,
    lot_linestyle=lot_linestyle,
    process_title_color=process_title_color,
    palette_preview=palette_preview,
    signal_linestyle=signal_linestyle,
    style_signal_lines=style_signal_lines,
)
configure()


## 자료 읽기와 연결

기존 01과 같은 규칙으로 읽습니다. LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "02"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg = [frames[k] for k in ["부품", "정비", "교체"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = report_viz.LOT_COLORS.copy()
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": report_viz.MAINTENANCE_COLORS["BM"],
       "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"], "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None, opened=False):
    """계산 결과표는 접어 둔다(눌러서 펼침). 본문에는 그림과 글로 쓴 결론·판단표만 펼쳐 둔다."""
    from IPython.display import HTML
    shown = frame if columns is None else frame[columns]
    if isinstance(shown, pd.Series):
        shown = shown.to_frame()
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        body = shown.to_html()
    display(HTML(f"<details{' open' if opened else ''}><summary>표 펼쳐 보기 · {len(shown)}행</summary>{body}</details>"))

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":report_viz.MAINTENANCE_COLORS["BM"],"CBM":report_viz.MAINTENANCE_COLORS["CBM"],"TBM":report_viz.MAINTENANCE_COLORS["TBM"]})

## 그림 표시 규칙

01~05는 공통 색상·제목·범례 규칙을 사용합니다. 제목은 대상과 비교를 짧게 표시하고, 긴 정의와 해석은 그림 아래 **그림 상세 기준·해석 펼치기**에 둡니다.

- **범주 구분:** 공정은 패널 제목, 강종은 그룹 표식, LOT는 비교선, 정비유형은 작업 구간의 색으로 구분합니다. 같은 항목은 모든 노트북에서 같은 색을 씁니다.
- **색 외의 구분:** 같은 강종 LOT는 대비되는 고정 색으로 비교합니다. TC·CP는 실선, OP는 점선으로 표시하며, 같은 존의 TC·OP는 같은 색을 사용합니다. 검토 지점은 기존 (a)·(b)·(c) 또는 구간 번호와 확대 그림을 연결합니다.

- **축과 단위:** 실제 시각과 경과 분을 구분합니다. TC는 ℃, OP는 %로 표시합니다(팀 해석). CP의 단위는 미확인이며 원본 기록값을 사용합니다.
- **표시:** 저장된 그림과 바로 아래 해석을 함께 읽습니다.

이 노트북은 결론, 수준 변동, 근거 범위, 출력 응답, 세정 입구의 그림 10개를 저장한다. 같은 폴더의 `02_LOT`, `03_강종`, `04_정비전후`는 이전 분석 보존본이며 이 노트북 실행으로 갱신되지 않는다. 현행 전수 근거는 [05](05_all_data_timeseries_jinny_05.ipynb)를 따른다.


In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M (단위 미확인)"
    if signal == "CP-Z4":
        return "CP-Z4 (단위 미확인)"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    return report_viz.header(fig, title, subtitle, handles, ncol)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}", ls=report_viz.lot_linestyle(int(k))) for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.2인치) 둔다.
PAD = report_viz.EXPORT_PADDING
rcParams["savefig.pad_inches"] = PAD

def finish(fig, name, svg=False):
    """공통 표시 규칙으로 현재 그림을 저장하고 본문에 표시한다."""
    fig.set_dpi(report_viz.EXPORT_DPI)
    report_viz.prepare(fig)
    compact_explanations(fig)
    fig.canvas.draw()
    # 여백을 잘라 저장할 때 그림 위쪽 범례만 따로 밀리지 않도록, 범례 위치를 그림 좌표에 다시 묶는다.
    for legend in fig.legends:
        anchor = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(anchor, transform=fig.transFigure)
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=report_viz.EXPORT_DPI, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(180dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display(fig)
    caption = report_viz.details_html(fig)
    if caption:
        from IPython.display import HTML
        display(HTML(caption))
    plt.close(fig)

In [ ]:
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

## 상세 시계열 준비

1초 기대 간격의 품질, 1분 집계, 사후 형태 상태(초기 승온·종료 전환·안정 참고), 같은 강종·안정 참고·같은 경과 분 비교, 실제 작업 교집합을 계산한다. 기존 01~08과 같은 코드다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8, ls=report_viz.lot_linestyle(k))
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")

In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP의 단위는 미확인이다. 자체 참고 중앙값과 차이를 탐색하며 정상 한계로 해석하지 않는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])

In [ ]:
# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])


In [ ]:
hypotheses = []
def verdict(hypothesis, result, evidence, reason, use, limit="운전 설정·인과·실제 고장 정보 부족", source="기존 03"):
    hypotheses.append({"가설": hypothesis, "판정": result, "검증 근거": evidence, "판정 이유": reason,
                       "08 활용": use, "남은 한계": limit, "출처": source})

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a * 60)), min(r.END, r.START + pd.Timedelta(seconds=int(b * 60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def clock_axis(ax, day):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    style(ax, f"실제 측정 시각[{day:%m/%d} 시:분]")

def clock_top(ax, start, label="같은 LOT 실제 시각[시:분]"):
    """경과 분 축 위에 대상 LOT의 실제 시각을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda m: base + m / 1440, lambda d: (d - base) * 1440))
    top.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    return top

def mean30(k, cols):
    """팀 자료와 같은 집계: 센서별 유효값 30개 이상인 1분 평균."""
    g = raw[k].set_index("MEAS_DT")[cols]
    return g.resample("1min").mean().where(g.resample("1min").count().ge(30))

def sustained_from(series, limit):
    """기준보다 낮은 상태가 관측 끝까지 이어지는 구간의 시작 시각."""
    s = series.dropna(); low = s.lt(limit)
    if not low.iloc[-1]:
        return None
    i = len(low) - 1
    while i > 0 and low.iloc[i - 1]:
        i -= 1
    return low.index[i]

In [ ]:
# ------------------------------------------------------------------------------------------------
# 예정된 변화와 탐지 구간 분류용 기준 ①~④ (01~04가 같은 코드로 계산한다 · 최종 확인 대상은 04에서 정한다)
# ------------------------------------------------------------------------------------------------
# 예정된 변화: 계획수리 직전의 종료 전환(끄면 출력·온도가 함께 내려감), 직후의 초기 승온(켜면 함께 올라감)
relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "예정된 정지"), ("초기 승온", "재가동 승온")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())

# ① 출력과 온도가 반대로 움직인 구간: 출력 0%(≤2%)가 5분 이상인데 온도가 내려가지 않음 / 출력 100%(≥98%)가 5분 이상인데 온도가 오르지 않음
opposite_rows = []
for k in lots.LOT_NO.astype(int):
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9))
        tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on, bad in [("출력 0%인데 온도 안 내려감", op.le(2), lambda d: d >= 0),
                              ("출력 100%인데 온도 안 오름", op.ge(98), lambda d: d <= 0)]:
            for a, b in runs(on.fillna(False), 5):
                change = tc.iloc[min(b, len(tc) - 1)] - tc.iloc[a]
                if bad(change):
                    opposite_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "ZONE": f"Z{z}", "종류": kind,
                                          "START_MIN": a, "END_MIN": b, "길이[분]": b - a, "온도 변화[℃]": change,
                                          "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=a),
                                          "예정된 변화 중": in_expected(k, a)})
opposite = pd.DataFrame(opposite_rows)

# ②·③ 같은 강종 대비 LOT 전체 수준 차이: 안정 참고 1분 중앙값의 LOT 중앙값 - 같은 강종 다른 LOT 중앙값들의 중앙값
level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(states[k].eq("안정 참고") & quality[k][s].ge(.9)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN
overage = level_gap.loc[level_gap.SENSOR.isin(["TC-Z7", "TC-Z8"]) & level_gap.GAP.ge(2.5)]       # ② 과시효 Z7·Z8이 같은 강종보다 2.5℃ 이상 높은 LOT
OVERAGE_LOTS = sorted(overage.LOT.unique())

# ③ 세정 입구 온도(CLN-IN) LOT 중앙값을 정렬해 가장 큰 간격에서 두 묶음으로 나눈다.
cln = level_gap.loc[level_gap.SENSOR.eq("CLN-IN")].set_index("LOT").LOT_MEDIAN.sort_values()
cut = cln.diff().idxmax()                                   # 큰 묶음의 첫 LOT
CLN_SPLIT = (cln.loc[:cut].iloc[-2] + cln.loc[cut]) / 2
cln_group = pd.Series(np.where(cln.ge(CLN_SPLIT), f"{CLN_SPLIT:.1f}℃ 이상", f"{CLN_SPLIT:.1f}℃ 미만"), index=cln.index)

# ④ CP 변화: 자체 안정 참고 기준과 가장 크게 벗어난 펄스 (예정된 정지 중의 CP 하강은 제외)
cp_main = cp_candidates.loc[[not in_expected(int(r.LOT), r.PEAK_MIN) for r in cp_candidates.itertuples()]]
CP_TOP = cp_main.loc[cp_main.DELTA.abs().idxmax()]

# ① 대표 사례: 출력 0%가 5분 이상인데 온도가 가장 크게 오른 곳
z0 = opposite.loc[opposite["종류"].str.startswith("출력 0%") & ~opposite["예정된 변화 중"]]
Z3_TOP = z0.loc[z0["온도 변화[℃]"].idxmax()]

# 03에서 자세히 본 LOT과 이유 (그림에서 빨간색으로 표시한다)
SELECTED = {}
for k in OVERAGE_LOTS:
    SELECTED.setdefault(int(k), []).append("② Z7·Z8")
SELECTED.setdefault(int(CP_TOP.LOT), []).append("④ CP")
SELECTED.setdefault(int(Z3_TOP.LOT), []).append("① Z3")
SELECT_COLOR = "#d62728"

MARK = {"①": "#c0392b", "②": "#8e44ad", "③": "#16a085", "④": "#d35400"}
MARK_NAME = {"①": "출력·온도 반대 움직임", "②": "과시효 Z7·Z8 수준 차이", "③": "세정 입구 온도 두 수준", "④": "가장 큰 CP 변화"}
EXPECTED_COLOR = "#8d99a6"

In [ ]:
from matplotlib.patches import Rectangle
GRADE_COLOR=report_viz.GRADE_COLORS.copy()
ZONE_COLOR=report_viz.ZONE_COLORS.copy()
ZONES=[3,4,5,6]
SELECT_COLOR="#c83745"
MARK_NAME["①"]="출력·온도 응답 확인"
PROCESS["TC-Z9"]="Z9"
def sensor_color(s):
    return report_viz.sensor_color(s)
def sensor_scale(s):return 1e10 if s=="CP-Z4M" else 1
def shown_unit(s):return "CP-Z4M (단위 미확인·10⁻¹⁰)" if s=="CP-Z4M" else unit(s)
def when(k,a,b):
    start=lot_meta.loc[k,"START"]
    return f"{start+pd.Timedelta(minutes=float(a)):%m/%d %H:%M:%S}–{start+pd.Timedelta(minutes=float(b)):%m/%d %H:%M:%S}"
def clock_axis(ax,day=None):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    style(ax,"실제 측정 날짜·시각[월/일 시:분]")
def raw_window(lot,a,b):
    r=lot_meta.loc[lot]
    times=pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)),min(r.END,r.START+pd.Timedelta(seconds=int(b*60))),freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)
def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"
def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]
def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")

# 모든 후보를 목록에 남기고, 긴 수준 차이와 짧은 변화를 구분해 표시한다.
review_rows=[]
for e in episodes.itertuples():
    level=level_gap.loc[level_gap.LOT.eq(e.LOT)&level_gap.SENSOR.eq(e.SENSOR),"GAP"].iloc[0]
    long=bool(e.DURATION_MIN>=30)
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":e.SENSOR,"KIND":"같은 강종 차이","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"최대 {e.MAX_ABS_DELTA_C:.2f}℃ · 대조 {e.MIN_PEERS} LOT 이상","LOCAL":not long})
for e in opposite.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":f"TC-{e.ZONE}","KIND":"출력·온도 응답","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"{e.종류} · TC 변화 {e._8:+.2f}℃","LOCAL":True})
for e in cp_candidates.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":"CP-Z4","KIND":"CP 변화","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"자체 참고 대비 {e.DELTA:+.3f} · 극값 {e.PEAK_VALUE:.4g}","LOCAL":e.PULSE_WIDTH_MIN<30})
review_catalogue=pd.DataFrame(review_rows).sort_values(["LOT","A","SENSOR"]).reset_index(drop=True)
review_catalogue["REVIEW_ID"]=[f"R{i+1:03d}" for i in range(len(review_catalogue))]
review_catalogue["START"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.A,unit="min")
review_catalogue["END"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.B,unit="min")

def lot_windows(k):
    """겹친 국소 변화의 문맥을 합친다. 긴 수준 차이는 전체 파형·비교 표에서 확인한다."""
    rows=review_catalogue.loc[review_catalogue.LOT.eq(k)&review_catalogue.LOCAL]
    result=[]
    for r in rows.sort_values('A').itertuples():
        a=max(0,r.A-5);b=min(lot_meta.loc[k,"MINUTES"],r.B+5)
        if result and a<=result[-1]["B"]+3:
            result[-1]["B"]=max(result[-1]["B"],b);result[-1]["IDS"].append(r.REVIEW_ID);result[-1]["SENSORS"].add(r.SENSOR)
        else:result.append({"A":a,"B":b,"IDS":[r.REVIEW_ID],"SENSORS":{r.SENSOR}})
    for i,r in enumerate(result):r["WINDOW_ID"]=f"{k}-W{i+1:02d}"
    return result
overview_coverage={};window_coverage={};maintenance_stage_rows=[];window_audit=[]

## 준비 — 01의 계산 다시 하기

자료·탐지·운전 상태 계산은 [01](05_all_data_timeseries_jinny_01.ipynb)과 같다. 그림·표는 01에서 보므로 여기서는 저장·표시하지 않는다(숨은 셀).

In [ ]:
# 01과 같은 계산을 다시 한다. 그림·표는 01에서 보므로 여기서는 저장·표시하지 않는다.
_finish, _note, _table = finish, note, table
finish = lambda fig, name, svg=False: plt.close(fig)
note = table = lambda *args, **kwargs: None

In [ ]:
roles = {"온도":("LOT_NO·MEAS_DT","22개 공정 신호·14 LOT 관측"),"조업":("PLANT_CD·EVT_DT","CR1 강종변경·이벤트"),
         "수리":("PLANT_CD·STRT_DT·END_DT","계획 날짜·범위; 실제 작업 완료 보증 아님"),
         "설비":("EQP_TAG","CAL01 설비 범위 확인"),"부품":("PART_CD","부품명·형식; 표준 수명은 실제 잔여수명 아님"),
         "정비":("EQP_TAG·WO_NO·세 시각","감지와 작업 시작·완료를 구분"),"교체":("WO_NO·PART_CD","정비 WO에 연결된 교체")}
table(pd.DataFrame([{"자료":k,"원본 행수":len(v),"연결 기준":roles[k][0],"확인할 내용":roles[k][1]} for k,v in frames.items()]))
table(eqp.loc[eqp.EQP_TAG.isin([CAL_TAG]),["EQP_TAG","EQP_NM","PARENT_TAG","INSTALL_DT","CBM_TGT_YN","DATA_AVL_YN"]])
table(context)
table(cal_parts[["WO_NO","END_DT","MNT_TYPE","PART_CD","PART_NM","QTY","CHG_RSN","REMARK"]])
note(f"**결과:** 온도 {len(temp):,}행·{len(lots)} LOT·{temp.MEAS_DT.dt.normalize().nunique()}일이다. CAL01 정비 {len(cal_work)} WO, 연결된 교체 {len(cal_parts)}행이다. 실제 온도 관측과 작업이 겹치는 LOT은 {context.loc[context.CAL_WORK.gt(0),'LOT'].tolist()}이다. 관측이 드물어 연속 생산 흐름 전체를 복원할 수는 없다.")

In [ ]:
# LOT 안의 연속 시각 구간을 확인한다. 60초를 넘는 공백이 있으면 별도 세그먼트로 나눈다.
GAP_SPLIT_S = 60
segment_rows = []
for r in lots.itertuples():
    t = temp.loc[temp.LOT_NO.eq(r.LOT_NO), "MEAS_DT"].sort_values()
    gap = t.diff().dt.total_seconds()
    segment_rows.append({"LOT": r.LOT_NO, "강종": r.GRADE, "관측 시작": r.START, "관측 끝": r.END,
                         "기록[행]": len(t), "최대 시각 간격[초]": gap.max(),
                         "1초 초과 간격[개]": int(gap.gt(1).sum()),
                         "60초 초과 공백[개]": int(gap.gt(GAP_SPLIT_S).sum()),
                         "세그먼트[개]": int(gap.gt(GAP_SPLIT_S).sum()) + 1})
segments = pd.DataFrame(segment_rows)
table(segments)
note(f"**결과:** 14 LOT 모두 세그먼트 1개다. LOT 안의 최대 시각 간격은 {segments['최대 시각 간격[초]'].max():.0f}초이고 60초를 넘는 공백은 없다. "
     "따라서 이후 분석의 관측 세그먼트는 LOT과 같다. 1초를 넘는 짧은 간격은 02의 '1. 데이터 상태'에서 '측정 기록 공백'으로 따로 확인한다.")

In [ ]:
# 각 LOT 시작과 같은 시각의 CR1 강종변경, 관측 종료 뒤 다음 강종변경까지의 시간을 함께 본다.
change_rows = []
for r in lots.itertuples():
    e = grade_evt.loc[grade_evt.EVT_DT.eq(r.START)].iloc[0]
    next_change = grade_evt.loc[grade_evt.EVT_DT.gt(r.END), "EVT_DT"].min()
    inside = grade_evt.EVT_DT.gt(r.START) & grade_evt.EVT_DT.le(r.END)
    change_rows.append({"LOT": r.LOT_NO, "강종변경 시각": e.EVT_DT, "이전 강종": e.BEF_VAL, "변경 후 강종": e.AFT_VAL,
                        "관측 중 추가 변경[건]": int(inside.sum()),
                        "관측 종료→다음 변경[시간]": (next_change - r.END).total_seconds() / 3600})
grade_link = pd.DataFrame(change_rows)
table(grade_link)
table(event.groupby(["PLANT_CD", "EVT_TYPE"]).size().unstack(fill_value=0))
note(f"**결과:** 14 LOT 모두 CR1 강종변경 시각에 정확히 시작하고, 관측 중 추가 강종변경은 없다. 관측 종료부터 다음 강종변경까지는 최소 {grade_link['관측 종료→다음 변경[시간]'].min():.1f}시간이다. "
     "즉 **같은 강종변경의 '변경 전' 수준은 온도 자료에 없다.** CR1 이벤트는 강종변경뿐이며, 조업률변경·원료배합변경은 BF·SP 설비의 기록이라 CAL01 조건 변수로 쓰지 않는다.")
note("**2단계 재정의:** '강종변경 전후 기준 수준 비교'는 같은 변경의 전후가 아니라, **변경 직후 과도 구간이 안정 수준에 도달하는 데 걸린 시간과 수준 차이를 이전 강종별로 비교**하는 것으로 바꾼다. 비교 그림은 아래 '9. 강종변경 직후 과도 구간'에 있다.")

In [ ]:
focus = set(context.loc[context.CAL_WORK.gt(0)|context.CAL_DETECT.gt(0)|context.PLAN_RELATION.ne(""),"LOT"])
# 행과 주석 사이에 여백을 확보하되, 연간 날짜축의 가로 폭은 유지한다.
fig, ax = plt.subplots(figsize=(30,19.5))
fig.subplots_adjust(left=.14,right=.977,top=.83,bottom=.12)
rows = {"LOT":9,"GRADE":8,"BM":7,"CBM":6,"TBM":5,"DETECT":4,"INS-006":3,"SEL-006":2,"SEL-009":1,"PLAN":0}
for label,y in rows.items():
    color = COL.get(label,COL["PART"])
    ax.axhspan(y-.09,y+.09,color=color,alpha=.055,lw=0)
for p in plans.itertuples():
    ax.axvspan(p.STRT_DT,p.END_EXCL,color=COL["PLAN"],alpha=.65,lw=0)
    ax.plot([p.STRT_DT,p.END_EXCL],[0,0],color=COL["LOT"],lw=5,solid_capstyle="butt")
for r in grade_evt.itertuples():
    ax.plot(r.EVT_DT,8,marker="|",color=COL["GRADE"],ms=8,mew=.9)
for t in cal_work.itertuples():
    y = rows[t.MNT_TYPE]
    ax.plot([t.STRT_DT,t.END_DT],[y,y],color=COL[t.MNT_TYPE],lw=2)
    ax.plot(t.STRT_DT,y,"x",color=COL[t.MNT_TYPE],ms=6)
    ax.plot(t.DETC_DT,4,"v",color=COL["DETECT"],ms=5,alpha=.75)
# 동일 WO의 정비와 부품교체 연결은 LOT 시간 겹침과 별도로 유지한다.
# 한 WO에 부품이 둘이어도 세로 점선 하나를 쓰고 해당 부품 행에 마름모를 찍는다.
annual_part_links = []
for wo, group in cal_parts.groupby("WO_NO",sort=False):
    shown = group.loc[group.PART_CD.isin(rows)]
    if shown.empty:
        continue
    maintenance = shown.MNT_TYPE.iloc[0]
    completed = shown.END_DT.iloc[0]
    bottom = min(rows[code] for code in shown.PART_CD)
    ax.plot([completed,completed],[rows[maintenance],bottom],
            color=COL["PART"],lw=1.8,ls="--",alpha=.85,zorder=1.8,
            label=f"_wo_part_link_{wo}")
    for p in shown.itertuples():
        ax.plot(p.END_DT,rows[p.PART_CD],"D",color=COL["PART"],ms=7,zorder=5)
    annual_part_links.append({
        "WO":wo,"정비 종류":maintenance,"WO 완료 시각":completed,
        "연결 부품":" · ".join(f"{p.PART_NM} [{p.PART_CD}] × {p.QTY}" for p in shown.itertuples()),
        "교체 기록[행]":len(shown),"REMARK":shown.REMARK.fillna("미기재").iloc[0]})
annual_part_summary = pd.DataFrame(annual_part_links)
work_by_wo = cal_work.set_index("WO_NO")
labels = []
for i,r in enumerate(lots.itertuples()):
    color = COL["FOCUS"] if r.LOT_NO in focus else COL["LOT"]
    ax.plot([r.START,r.END],[9,9],color=color,lw=3)
    ax.plot(r.START,9,"o",color=color,ms=6)
    y = 10.4 if i%2==0 else 12.3
    label_text = f"{r.LOT_NO}\n{r.GRADE}\n{r.START:%m/%d}"
    label = ax.text(r.START,y,label_text,ha="center",va="center",fontsize=14.5,color=color,
                    bbox=dict(boxstyle="round,pad=.30",fc="white",ec=color,lw=.65))
    labels.append((label,r.START,y,color))

# 한 LOT당 세로 안내선 하나를 사용한다.
# 빈 원만 LOT 관측창과 실제로 겹친 기록을 뜻하며, 지나가는 선만으로는 겹침을 뜻하지 않는다.
# 원은 실제 사건/교집합 대표시각에 찍고, 정확한 시간·구간은 아래 표에 보존한다.
annual_links, annual_overlap_rows, annual_guides = [], [], []

def clock_union_minutes(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return sum((end-start).total_seconds()/60 for start,end in merged)

def mark_lot_record(lot, category, at_time, start, end, key=""):
    color = COL[category]
    # 원의 위치는 실제 시각 그대로 두어, 서로 다른 사건을 같은 순간으로 옮기지 않는다.
    ax.plot(at_time, rows[category], "o", mfc="white", mec=color, ms=9, mew=2, zorder=5)
    annual_links.append({"LOT":lot, "연결 기록":category, "키":key,
                         "겹침 시작":start, "겹침 끝":end, "빈 원 표시 시각":at_time,
                         "구간 교집합[분]":(end-start).total_seconds()/60})

for r in lots.itertuples():
    k = int(r.LOT_NO)
    ge = grade_evt.loc[grade_evt.EVT_DT.between(r.START,r.END)]
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START,r.END)]
    for event_row in ge.itertuples():
        at_time = event_row.EVT_DT
        mark_lot_record(k, "GRADE", at_time, at_time, at_time,
                        key=f"{event_row.BEF_VAL}→{event_row.AFT_VAL}")
    work_intervals = []
    for task in tasks.itertuples():
        start,end = max(task.STRT_DT,r.START),min(task.END_DT,r.END)
        at_time = start + (end-start)/2
        mark_lot_record(k, task.MNT_TYPE, at_time, start, end, key=task.WO_NO)
        work_intervals.append((start,end))
    for task in detected.itertuples():
        at_time = task.DETC_DT
        mark_lot_record(k, "DETECT", at_time, at_time, at_time, key=task.WO_NO)
    annual_overlap_rows.append({
        "LOT":k, "강종":r.GRADE, "온도 관측 시작":r.START, "온도 관측 끝":r.END,
        "강종변경 동시 기록[건]":len(ge),
        "실제 CAL01 작업[WO]":len(tasks),
        "실제 작업 교집합 합집합[분]":clock_union_minutes(work_intervals),
        "관측 중 감지 기록[건]":len(detected)})
annual_overlap_summary = pd.DataFrame(annual_overlap_rows)
annual_interval_links = pd.DataFrame(annual_links)

# 연간 날짜축에서는 한 LOT의 짧은 관측창을 같은 세로 위치에서 읽도록 묶는다.
# 아래쪽 끝은 마지막으로 겹친 기록 행이며, 겹치지 않은 행에는 빈 원을 그리지 않는다.
GUIDE_COLOR = "#607080"
for r in lots.itertuples():
    k = int(r.LOT_NO)
    linked = annual_interval_links.loc[annual_interval_links.LOT.eq(k)]
    if linked.empty:
        continue
    bottom_y = min(rows[category] for category in linked["연결 기록"])
    guide, = ax.plot([r.START,r.START], [rows["LOT"],bottom_y],
                     color=GUIDE_COLOR,lw=2.2,alpha=.75,zorder=2,
                     label=f"_lot_guide_{k}",solid_capstyle="butt")
    ax.plot(r.START,rows["LOT"],"o",mfc="none",mec=GUIDE_COLOR,ms=10,mew=1.5,zorder=5)
    annual_guides.append({"LOT":k,"안내선 기준":r.START,"끝 행":bottom_y,
                          "겹친 기록 종류":",".join(linked["연결 기록"].drop_duplicates())})


# 글상자 폭을 측정하여 같은 층에서 겹치는 라벨만 조금 이동한다.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for y in [10.4,12.3]:
    previous = None
    for label,x,level,color in labels:
        if level != y:
            continue
        box = text_box(label,renderer)
        if previous is not None and box.x0 < previous+7:
            delta = previous+7-box.x0
            screen = ax.transData.transform((mdates.date2num(x),y))
            label.set_x(ax.transData.inverted().transform((screen[0]+delta,screen[1]))[0])
            fig.canvas.draw()
            box = text_box(label,renderer)
        previous = box.x1
for label,x,y,color in labels:
    ax.plot([x,label.get_position()[0]],[9,y-.45],color=color,alpha=.45,lw=.7)
counts = cal_work.MNT_TYPE.value_counts()
axis_labels = ["CR1 계획수리 · 2회","O링 세트 · SEL-009","오일씰 · SEL-006","가스 분석계 셀 · INS-006",
               "정비 감지 [DETC_DT]",f"TBM 시간기반 · {counts['TBM']} WO",
               f"CBM 상태기반 · {counts['CBM']} WO",f"BM 사후 · {counts['BM']} WO",
               f"강종변경 · {len(grade_evt)}건",f"온도 관측 · {len(lots)} LOT"]
ax.set_yticks(range(10),axis_labels)
axis_colors = [COL["LOT"],COL["PART"],COL["PART"],COL["PART"],COL["DETECT"],
               COL["TBM"],COL["CBM"],COL["BM"],COL["GRADE"],COL["LOT"]]
for label,color in zip(ax.get_yticklabels(),axis_colors):
    label.set_color(color)
    label.set_fontweight("bold")
    label.set_fontsize(14.5)
ax.set_ylim(-.65,13.8)
ax.set_xlim(YEAR_START-pd.Timedelta(days=2),YEAR_END+pd.Timedelta(days=3))
ax.set_xticks(pd.date_range("2024-01-01","2024-12-01",freq="MS"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
style(ax,"2024년 기록 시각[월]")
ax.set_ylabel("")
handles = [Line2D([0],[0],color=COL["FOCUS"],marker="o",label="작업 겹침·감지 포함·수리 전후 LOT"),
           Line2D([0],[0],color=COL["LOT"],marker="o",label="기타 온도 관측 LOT"),
           Line2D([0],[0],color=COL["GRADE"],marker="|",label="CR1 강종변경"),
           Line2D([0],[0],color=GUIDE_COLOR,lw=2.2,label="LOT별 세로 안내선"),
           Line2D([0],[0],color="#34495e",marker="o",mfc="white",ls="None",ms=8,label="빈 원 · LOT 관측창과 겹친 기록"),
           *[Line2D([0],[0],color=COL[k],marker="x",label=k+" 작업") for k in ["BM","CBM","TBM"]],
           Line2D([0],[0],color=COL["DETECT"],marker="v",label="DETC_DT 감지 기록 · 방법 미제공"),
           Line2D([0],[0],color=COL["PART"],marker="D",ls="--",lw=1.8,label="동일 WO 정비→부품교체 · 완료 기준"),
           Patch(facecolor=COL["PLAN"],label="계획수리 날짜")]
# 연간 축에는 주요 REMARK만 표시한다. 전체 WO 내용은 월별 상세 그림에서 확인한다.
for wo,x,y,align,prefix in [
    ("WO-2024-00960","2024-06-17",5.5,"right","6월 수리 TBM 6건 · 대표 REMARK"),
    ("WO-2024-01094","2024-07-10",7.5,"right","07/08 BM"),
    ("WO-2024-01859","2024-11-07",5.5,"right","수리 TBM 5건 이후 · 11/13 TBM"),
    ("WO-2024-01854","2024-11-09",6.5,"right","11/13 CBM"),
    ("WO-2024-02105","2024-12-27",6.5,"right","12/28 CBM · 셀 3개 교체 연결")]:
    t=work_by_wo.loc[wo]
    ax.annotate(f"{prefix}\n{str(t.REMARK).strip()}",xy=(t.STRT_DT,rows[t.MNT_TYPE]),
                xytext=(pd.Timestamp(x),y),ha=align,va="center",fontsize=14,color=COL[t.MNT_TYPE],
                bbox=dict(boxstyle="round,pad=.42",fc="white",ec=COL[t.MNT_TYPE],lw=.8),
                arrowprops=dict(arrowstyle="->",color=COL[t.MNT_TYPE],lw=.9),zorder=8)
for p in plans.itertuples():
    scope = p.SCOPE
    ax.annotate(f"{p.STRT_DT:%m/%d}–{p.END_DT:%m/%d}\n{scope}",xy=(p.STRT_DT,0),
                xytext=(p.END_EXCL+pd.Timedelta(days=3),.5),ha="left",va="center",fontsize=14,color=COL["LOT"],
                bbox=dict(boxstyle="round,pad=.42",fc="white",ec=COL["LOT"],lw=.8),
                arrowprops=dict(arrowstyle="->",color=COL["LOT"],lw=.9),zorder=8)
for y,text in [
    (.081,"빈 원: 강종변경 14 LOT · 실제 작업 240117(TBM·CBM) · 감지 240117·240133"),
    (.053,"240117: 실제 작업 합집합 304분(TBM 115+CBM 230-동시 41)"),
    (.025,"선 구분: 빈 원이 있는 행만 LOT 관측창과 겹침 · 보라 점선은 같은 WO의 정비→부품교체 | 7월 BM·12월 작업은 관측 종료 뒤")]:
    fig.text(.14,y,text,ha="left",va="center",fontsize=14.2,color="#34495e")
header(fig,"연간 공정 흐름","TIME AXIS · 실제 측정 날짜·시각 | 회색 직선·빈 원=LOT 관측창 겹침 · 보라 점선·◆=동일 WO 정비→부품교체",handles,ncol=4)
ax.tick_params(axis="x",labelsize=14)
ax.xaxis.label.set_fontsize(14.5)
fig._suptitle.set_fontsize(24)
fig.texts[-1].set_fontsize(14)
# 제목·설명·범례는 위쪽에 모으고, 본문과 범례 사이에 별도 여백을 둔다.
fig.texts[-1].set_position((.035,.94))
fig.legends[0].set_bbox_to_anchor((.031,.907))
for text in fig.legends[0].get_texts(): text.set_fontsize(13)
finish(fig,"01_연간흐름/01_연간_공정흐름.png",svg=True)
table(annual_overlap_summary)
table(annual_interval_links)
table(annual_part_summary)
note("**안내선과 빈 원 읽기:** 한 LOT마다 회색 세로 안내선 하나를 쓰고, 관측창과 겹친 기록 행에만 빈 원을 표시한다. 파랑은 강종변경, 정비유형 색은 실제 TBM·CBM 작업 교집합, 남색은 관측 중 DETC_DT 감지 시각이다. 원이 없는 행은 선이 지나가더라도 겹침으로 읽지 않는다. 안내선은 LOT 시작 위치에서 기록을 묶는 읽기 기준이며, 빈 원은 실제 사건 또는 교집합 대표시각에 남긴다. 따라서 같은 LOT의 빈 원들이 모두 같은 순간에 발생했다는 뜻은 아니다. 보라 점선은 동일 WO의 BM·CBM·TBM 정비와 부품교체 기록을 연결하고, ◆는 해당 부품 행의 기록이다. 이 점선은 LOT 시간 겹침선과 구분한다. 부품 위치는 WO 완료 시각 기준이며 실제 교체 순간을 뜻하지 않는다. 정확한 구간·분·WO·부품·REMARK는 위 표와 월별 상세에서 확인한다.")
note(f"**그래프에서 확인:** 온도·실제 CAL01 작업이 겹친 LOT은 240117이다. 주요 REMARK는 6월 TBM의 「1호기 재발, SD작업 수행」, 7월 BM의 「1호기 재발, 절연 저하」, 11월 TBM의 「계획 점검 수행」와 CBM의 「마모량 초과」, 12월 CBM의 「마모량 초과」다. 6월·11월 계획수리 범위와 관측 전후, 7월 후속 BM, 12월 감지 뒤 작업·교체를 구분한다. 감지 방법·실제 교체 시각·고장 원인은 이 연간 그림으로 확인되지 않는다. 모든 WO의 원문과 정확한 시각은 뒤의 월별 상세 그림·표에서 확인한다.")

In [ ]:
from matplotlib.patches import Rectangle

months = pd.Index(range(1,13),name="MONTH")
monthly = pd.DataFrame(index=months)
monthly["온도 LOT[개]"] = lots.groupby(lots.START.dt.month).size().reindex(months,fill_value=0)
monthly["온도 관측창 합계[시간]"] = lots.groupby(lots.START.dt.month).MINUTES.sum().reindex(months,fill_value=0)/60
monthly["CAL01 작업[WO]"] = cal_work.groupby(cal_work.STRT_DT.dt.month).size().reindex(months,fill_value=0)
monthly["CR1 강종변경[건]"] = grade_evt.groupby(grade_evt.EVT_DT.dt.month).size().reindex(months,fill_value=0)
for kind in ["BM","CBM","TBM"]:
    work = cal_work.loc[cal_work.MNT_TYPE.eq(kind)]
    monthly[f"{kind} 작업[WO]"] = work.groupby(work.STRT_DT.dt.month).size().reindex(months,fill_value=0)

# 작업 구간을 LOT 관측창으로 자르고, 여러 WO가 동시에 진행된 시간은 한 번만 센다.
def observed_work_minutes(start,end,work):
    intervals = sorted((max(start,t.STRT_DT),min(end,t.END_DT)) for t in work.itertuples())
    total = 0.0
    current_end = start
    for left,right in intervals:
        left = max(left,current_end)
        if right > left:
            total += (right-left).total_seconds()/60
            current_end = right
    return total

links = []
for r in lots.itertuples():
    work = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    links.append({"LOT":r.LOT_NO,"MONTH":r.START.month,
                  "온도·작업 겹친 LOT[개]":int(len(work)>0),"온도·작업 겹친 WO[개]":len(work),
                  "온도·작업 겹침[분]":observed_work_minutes(r.START,r.END,work)})
monthly_links = pd.DataFrame(links)
for column in monthly_links.columns[2:]:
    monthly[column] = monthly_links.groupby("MONTH")[column].sum().reindex(months,fill_value=0)

# 월별 건수 대신 실제 관측창과 정비 REMARK를 보여준다.
# START–END는 제공된 기록의 범위이며, 실제 가동시간을 뜻하지 않는다.
GRADE_COLOR_MONTHLY = report_viz.GRADE_COLORS.copy()


def monthly_clock_range(start, end, seconds=False):
    clock = "%H:%M:%S" if seconds else "%H:%M"
    left = start.strftime("%m/%d " + clock)
    right = end.strftime(clock if start.date() == end.date() else "%m/%d " + clock)
    return f"{left}–{right}"


def monthly_part_records(wo_no):
    records = cal_parts.loc[cal_parts.WO_NO.eq(wo_no)]
    return [f"{r.PART_NM} [{r.PART_CD}] ×{r.QTY:g} ({r.CHG_RSN})"
            for r in records.itertuples()]


# 같은 월이라는 이유로 연결하지 않는다. 작업 구간과 LOT 관측창을 직접 대조한다.
work_observation_links = []
detection_observation_links = []
for task in cal_work.itertuples():
    for lot in lots.itertuples():
        left, right = max(task.STRT_DT, lot.START), min(task.END_DT, lot.END)
        if right > left:
            work_observation_links.append({
                "WO_NO":task.WO_NO, "LOT":int(lot.LOT_NO), "TYPE":task.MNT_TYPE,
                "START":left, "END":right, "MINUTES":(right-left).total_seconds()/60})
        if lot.START <= task.DETC_DT <= lot.END:
            detection_observation_links.append({
                "WO_NO":task.WO_NO, "LOT":int(lot.LOT_NO), "DETC_DT":task.DETC_DT,
                "WORK_START":task.STRT_DT})
work_observation_links = pd.DataFrame(work_observation_links)
detection_observation_links = pd.DataFrame(detection_observation_links)

# 각 월의 내용 길이에 맞춰 높이를 잡는다. 숫자 집계와 해석 문장은 그림에서 뺀다.
month_cards = []
for month in months:
    records = []
    month_lots = lots.loc[lots.START.dt.month.eq(month)]
    month_work = cal_work.loc[cal_work.STRT_DT.dt.month.eq(month)]
    month_plans = plans.loc[plans.STRT_DT.dt.month.eq(month)]
    for plan in month_plans.itertuples():
        records.append(("plan", f"계획수리 {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}", "#087f9c"))
    records.append(("section", "온도 · 제공된 LOT 관측창", "#17344d"))
    for lot in month_lots.itertuples():
        records.append(("lot", f"{lot.LOT_NO} {lot.GRADE} · {monthly_clock_range(lot.START,lot.END)}",
                        GRADE_COLOR_MONTHLY[lot.GRADE]))
    if month_lots.empty:
        records.append(("empty", "제공된 온도 기록 없음", "#6c7c89"))
    records.append(("space", "", ""))
    records.append(("section", "정비 · CAL01 작업 기록", "#17344d"))
    for task in month_work.itertuples():
        remark = str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재"
        short_wo = task.WO_NO.rsplit("-",1)[-1]
        records.append(("work", f"{task.STRT_DT:%m/%d} {task.MNT_TYPE} {short_wo} · {remark}",
                        COL[task.MNT_TYPE]))
        for part_record in monthly_part_records(task.WO_NO):
            records.append(("part", "교체: " + part_record, COL["PART"]))
        observed = work_observation_links.loc[work_observation_links.WO_NO.eq(task.WO_NO)]
        for link in observed.itertuples():
            records.append(("focus", f"LOT {link.LOT} 관측 중 · {link.START:%H:%M}–{link.END:%H:%M} · {link.MINUTES:g}분",
                            "#7540a8"))
        detected = detection_observation_links.loc[detection_observation_links.WO_NO.eq(task.WO_NO)]
        if observed.empty:
            for link in detected.itertuples():
                records.append(("focus", f"LOT {link.LOT} 감지 시각 관측 · {link.DETC_DT:%H:%M}", "#34495e"))
    if month_work.empty:
        records.append(("empty", "제공된 정비 작업 기록 없음", "#6c7c89"))
    card_height = 48 + sum(10 if kind=="space" else 23 for kind,_,_ in records)
    month_cards.append({"month":month,"records":records,"height":card_height})

row_heights = [max(card["height"] for card in month_cards[row*3:(row+1)*3]) for row in range(4)]
header_height, row_gap, margin_bottom = 90, 16, 16
figure_height = (header_height + sum(row_heights) + 3*row_gap + margin_bottom)/72
fig = plt.figure(figsize=(24,figure_height),facecolor="white")
fig.suptitle("월별 관측창·정비 작업 내용",x=.027,y=1-18/(72*figure_height),
             ha="left",va="top",fontsize=23,fontweight="bold",color="#17344d")
fig.text(.027,1-52/(72*figure_height),
         "온도: LOT별 관측 날짜·시각  |  정비: 작업일·유형·WO·REMARK  |  교체 부품은 기록이 있는 작업에만 표시",
         ha="left",va="top",fontsize=12.8,color="#526d80")
fig.text(.973,1-25/(72*figure_height),"2024년 · 실제 기록 시각",ha="right",va="top",
         fontsize=12.5,color="#526d80")
row_top = header_height
left_margin, right_margin, column_gap = .027, .027, .019
card_width = (1-left_margin-right_margin-2*column_gap)/3
for row in range(4):
    for column in range(3):
        card = month_cards[row*3+column]
        height = row_heights[row]
        ax = fig.add_axes([left_margin+column*(card_width+column_gap),
                           1-(row_top+height)/(72*figure_height),
                           card_width,height/(72*figure_height)])
        ax.set(xlim=(0,1),ylim=(height,0),xticks=[],yticks=[])
        ax.spines[:].set_visible(False)
        focused = card["month"]==11
        ax.add_patch(Rectangle((0,0),1,height,facecolor="#fafcfd",
                               edgecolor="#7540a8" if focused else "#dce5eb",lw=1.4))
        if card["month"] in [6,11]:
            ax.add_patch(Rectangle((0,0),1,35,facecolor=COL["PLAN"],alpha=.35,edgecolor="none"))
        ax.text(.025,12,f"{card['month']:02d}월",ha="left",va="top",fontsize=17,
                color="#17344d",fontweight="bold")
        cursor = 45
        for kind,text,color in card["records"]:
            if kind=="space":
                cursor += 10
                continue
            fontsize = 11.6 if kind=="part" else 12.4 if kind=="focus" else 12.8
            x = .045 if kind in ["part","focus"] else .025
            artist = ax.text(x,cursor,text,ha="left",va="top",fontsize=fontsize,color=color,
                             fontweight="bold" if kind in ["section","focus"] else "normal")
            artist._monthly_card = card["month"]
            cursor += 23
    row_top += height + row_gap

finish(fig,"01_연간흐름/02_월별_기록비교.png")

# 정확한 초 단위의 관측창과 WO별 작업 시간을 표로 보존한다.
observation_rows = []
for lot in lots.itertuples():
    work_links = work_observation_links.loc[work_observation_links.LOT.eq(lot.LOT_NO)]
    detection_links = detection_observation_links.loc[detection_observation_links.LOT.eq(lot.LOT_NO)]
    observation_rows.append({
        "월":f"{lot.START.month:02d}월","LOT":int(lot.LOT_NO),"강종":lot.GRADE,
        "첫 측정":lot.START,"마지막 측정":lot.END,"제공된 측정 기록[행]":lot.N_ROWS,
        "관측 중 작업":", ".join(work_links.WO_NO),
        "관측 중 감지 기록":", ".join(detection_links.WO_NO)})
observation_windows = pd.DataFrame(observation_rows)
maintenance_rows = []
for task in cal_work.itertuples():
    observed = work_observation_links.loc[work_observation_links.WO_NO.eq(task.WO_NO)]
    maintenance_rows.append({
        "월":f"{task.STRT_DT.month:02d}월","유형":task.MNT_TYPE,"WO_NO":task.WO_NO,
        "감지 기록 시각":task.DETC_DT,"작업 시작":task.STRT_DT,"작업 완료":task.END_DT,
        "기록된 작업 내용":str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재",
        "기록된 교체 부품":"; ".join(monthly_part_records(task.WO_NO)),
        "온도 관측 중 작업":"; ".join(
            f"LOT {r.LOT} · {monthly_clock_range(r.START,r.END,seconds=True)} · {r.MINUTES:g}분"
            for r in observed.itertuples())})
monthly_maintenance_details = pd.DataFrame(maintenance_rows)
table(observation_windows)
table(monthly_maintenance_details)

In [ ]:
# CAL01 작업 44건을 '그림이 들어간 표'로 보인다: 한 줄 = 작업 1건, 월별로 묶고 칸을 맞춘다.
# 시간 막대는 모양만(6시간 넘는 대기는 //로 끊음) · 정확한 시간은 숫자 칸에서 읽는다.
PAD_H = 72                                              # 가까운 LOT으로 볼 시간
TYPE_C = {"BM": report_viz.MAINTENANCE_COLORS["BM"], "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"]}
w = cal_work.sort_values("DETC_DT").reset_index(drop=True)
w["감지→착수[h]"] = (w.STRT_DT - w.DETC_DT).dt.total_seconds() / 3600
w["작업 시간[h]"] = (w.END_DT - w.STRT_DT).dt.total_seconds() / 3600
parts_of = cal_parts.groupby("WO_NO").apply(lambda d: "\n".join(f"{r.PART_NM} ×{int(r.QTY)} ({r.CHG_RSN})" for r in d.itertuples()), include_groups=False)
w["교체 부품"] = w.WO_NO.map(parts_of).fillna("")
def near_lot(t):
    """작업 기간과 LOT 관측의 시간 관계(겹침 · 관측 끝 뒤 · 관측 시작 전, 72시간 안)."""
    out = []
    for k, r in lot_meta.iterrows():
        if t.DETC_DT <= r.END and t.END_DT >= r.START: out.append(f"{k} 관측 중")
        elif pd.Timedelta(0) <= t.DETC_DT - r.END <= pd.Timedelta(hours=PAD_H): out.append(f"{k} 관측 끝 {(t.DETC_DT - r.END).total_seconds() / 3600:.0f}h 뒤")
        elif pd.Timedelta(0) <= r.START - t.END_DT <= pd.Timedelta(hours=PAD_H): out.append(f"{k} 관측 시작 {(r.START - t.END_DT).total_seconds() / 3600:.0f}h 전")
    return "\n".join(out)
def temp_record(t):
    """그 작업을 온도로 확인할 수 있는가: 작업 중 기록 / 감지 때만 / 앞뒤 LOT이 72시간 안."""
    if any(t.STRT_DT < r.END and t.END_DT > r.START for _, r in lot_meta.iterrows()): return "작업 중 기록"
    if any(r.START <= t.DETC_DT <= r.END for _, r in lot_meta.iterrows()): return "감지 때만"
    before = lot_meta.END.le(t.DETC_DT) & (t.DETC_DT - lot_meta.END).le(pd.Timedelta(hours=PAD_H))
    after = lot_meta.START.ge(t.END_DT) & (lot_meta.START - t.END_DT).le(pd.Timedelta(hours=PAD_H))
    return "앞뒤 LOT" if before.any() and after.any() else ""
w["가까운 LOT"] = [near_lot(t) for t in w.itertuples()]
w["온도 기록"] = [temp_record(t) for t in w.itertuples()]
plan_of = lambda t: next((p for p in plans.itertuples() if p.STRT_DT <= t < p.END_EXCL), None)
hours = lambda h: f"{h / 24:.1f}일" if h >= 24 else f"{h:.1f}h"

# 줄 목록: 월 머리 줄 + 작업 줄
rows = []
for m, d in w.groupby(w.DETC_DT.dt.month):
    ps = [p for p in plans.itertuples() if p.STRT_DT.month == m]
    rows.append(("month", f"{m}월 · {len(d)}건" + "".join(f"   ·   CR1 정기수리 {p.STRT_DT:%m/%d}~{p.END_DT:%m/%d} ({p.SCOPE})" for p in ps)))
    rows += [("wo", t) for _, t in d.iterrows()]
COLS = [(.0, "감지 시각"), (.072, "종류"), (.112, "WO"), (.165, "◆ 감지 · 점선 = 착수까지 대기 · 막대 = 작업"), (.35, "감지→착수"), (.405, "작업 시간"), (.452, "설비 정지"),
        (.5, "REMARK · 고장 유형"), (.645, "교체 부품"), (.79, "온도 기록"), (.86, "가까운 LOT (±72시간)")]
G0, GW, HMAX = .165, .165, 16                            # 시간 막대 칸 시작·폭, 칸 안에서 보이는 최대 시간
gx = lambda h: G0 + GW * min(h, HMAX) / HMAX
H = .5 * len(rows) + 4.0
fig = plt.figure(figsize=(30, H)); ax = fig.add_axes([.02, 1.0 / H, .97, 1 - 3.5 / H]); ax.axis("off")   # 위 2.5인치는 제목·범례, 아래 1인치는 계산식
ax.set_xlim(0, 1); ax.set_ylim(len(rows) - .5, -1.2)
for x, name in COLS:
    ax.text(x, -.95, name, fontsize=13, fontweight="bold", color="#526d80", va="center")
ax.plot([0, 1], [-.55, -.55], color="#9fb3c3", lw=1.2)
for i, (kind, t) in enumerate(rows):
    if kind == "month":
        ax.add_patch(Rectangle((0, i - .42), 1, .84, color="#e9eff4", lw=0, zorder=0))
        ax.text(.004, i, t, fontsize=13.5, fontweight="bold", color="#17344d", va="center")
        continue
    c = TYPE_C.get(t.MNT_TYPE, "#7c8995")
    if plan_of(t.STRT_DT) is not None:
        ax.add_patch(Rectangle((0, i - .45), 1, .9, color="#f3f7fa", lw=0, zorder=0))
    ax.add_patch(Rectangle((0, i - .4), .003, .8, color=c, lw=0))
    ax.text(.008, i, f"{t.DETC_DT:%m/%d %H:%M}", fontsize=12.5, va="center", color="#17344d")
    ax.text(.072, i, t.MNT_TYPE, fontsize=12, va="center", color="white", fontweight="bold", bbox=dict(boxstyle="round,pad=.25", fc=c, ec="none"))
    ax.text(.112, i, t.WO_NO.replace("WO-2024-", ""), fontsize=12, va="center", color="#526d80")
    wait, work = t["감지→착수[h]"], t["작업 시간[h]"]
    wait_s = wait if wait <= 6 else 7                           # 6시간 넘는 대기는 끊어 그림(//)
    ax.plot([gx(0), gx(wait_s)], [i, i], color=c, ls=":", lw=2)
    if wait > 6:
        ax.text(gx(6.5), i, "//", fontsize=13, color=c, ha="center", va="center", fontweight="bold", bbox=dict(boxstyle="square,pad=.05", fc="white", ec="none"))
    ax.add_patch(Rectangle((gx(wait_s), i - .17), gx(wait_s + min(work, HMAX - wait_s)) - gx(wait_s), .34, color=c, lw=0))
    ax.plot(gx(0), i, "D", ms=7, color=c)
    ax.text(.35, i, hours(wait), fontsize=12.5, va="center", color=c if wait >= 24 else "#17344d", fontweight="bold" if wait >= 24 else "normal")
    ax.text(.405, i, hours(work), fontsize=12.5, va="center", color="#17344d")
    ax.text(.452, i, f"{t.DOWN_MIN:.0f}분", fontsize=12.5, va="center", color="#17344d" if t.DOWN_MIN else "#9aa6ae")
    rem = (str(t.REMARK).strip() if pd.notna(t.REMARK) else "내용 미기재") + (f" · {t.FAIL_MODE}" if pd.notna(t.FAIL_MODE) else "")
    ax.text(.5, i, rem, fontsize=12.5, va="center", color="#17344d" if pd.notna(t.REMARK) else "#9aa6ae")
    if t["교체 부품"]:
        ax.text(.645, i, t["교체 부품"], fontsize=11.5, va="center", color=SELECT_COLOR, fontweight="bold", linespacing=1.15)
    if t["온도 기록"]:
        ax.text(.79, i, t["온도 기록"], fontsize=12, va="center", color="#17344d" if t["온도 기록"] == "작업 중 기록" else "#526d80",
                fontweight="bold" if t["온도 기록"] == "작업 중 기록" else "normal")
    if t["가까운 LOT"]:
        ax.text(.86, i, t["가까운 LOT"], fontsize=11.5, va="center", color="#17344d", linespacing=1.15)
n_part = int(w["교체 부품"].ne("").sum()); n_long = int(w["감지→착수[h]"].ge(24).sum()); n_rec = int(w["온도 기록"].eq("작업 중 기록").sum())
header(fig, f"CAL01 정비 이력 {len(w)}건 · 감지 → 착수 → 완료와 작업 내용",
       f"한 줄 = 작업 1건(감지 순, 월별 묶음) · 감지→착수 중앙값 {w['감지→착수[h]'].median():.1f}h, 1일 넘게 기다린 작업 {n_long}건 · 부품 교체 {n_part}건(빨강) · 작업 중 온도 기록 {n_rec}건 · 연한 바탕 = CR1 정기수리 기간",
       [Patch(facecolor=TYPE_C[k], label=f"{k} {int(w.MNT_TYPE.eq(k).sum())}건") for k in TYPE_C], ncol=3)
compact_header(fig, .975 - .6 / H, .975 - 1.1 / H, title_size=25, sub_size=14, legend_size=13)
fig.text(.02, .25 / H, "감지→착수 = 작업 시작(STRT_DT) - 감지(DETC_DT) · 작업 시간 = 완료(END_DT) - 작업 시작 · 설비 정지 = 기록된 정지 시간(DOWN_MIN) · "
         "온도 기록 = 작업하는 동안 LOT 온도 기록이 있음(작업 중 기록) / 감지 시각에만 있음(감지 때만) / 앞뒤 LOT이 모두 72시간 안(앞뒤 LOT)",
         ha="left", va="bottom", fontsize=12.5, color="#526d80")
axis_tag(fig, "작업 1건 = 1줄 · 감지 순서")
finish(fig, "01_연간흐름/03_정비이력_감지착수완료.png")
table(w[["WO_NO", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "감지→착수[h]", "작업 시간[h]", "DOWN_MIN", "FAIL_MODE", "REMARK", "교체 부품", "온도 기록", "가까운 LOT"]]
      .rename(columns={"DOWN_MIN": "설비 정지[분]"}).round({"감지→착수[h]": 1, "작업 시간[h]": 1}))


In [ ]:
# LOT별 자료 상태: 왼쪽 = 관측 전체(경과 분, 색 = 사후 상태), 오른쪽 = LOT끼리 비교하는 작은 막대 4칸
#   세로 눈금 = 1분 기록이 90% 미만이라 그 분 값을 쓰지 않은 곳 · ▼ = 기록이 5초 이상 끊긴 곳(숫자 = 끊긴 초, 평소 간격은 1초)
STATE_C = {"초기 승온": "#f2d3a8", "안정 참고": "#d5e4ef", "종료 전환": "#d9d2ee"}
GAP_C, NAN_C, JUMP_C = "#4a5a68", "#d55e00", "#17344d"
ids = lots.sort_values("START").LOT_NO.astype(int).tolist()
info = []
for k in ids:
    g = raw[k].sort_values("MEAS_DT"); dt = g.MEAS_DT.diff().dt.total_seconds()
    expected_s = (g.MEAS_DT.iloc[-1] - g.MEAS_DT.iloc[0]).total_seconds() + 1
    low = quality[k][SIGNALS].lt(.9)
    jump = dt.ge(3)                                          # 기록 간격 3초 이상(기록 2개 이상 연속으로 빠짐)
    at = ((g.MEAS_DT - lot_meta.loc[k, "START"]).dt.total_seconds() / 60)[jump]
    info.append({"공백[%]": 100 * (1 - len(g) / expected_s), "간격 위치": at.to_numpy(), "간격[초]": dt[jump].to_numpy(),
                 "공백 분": low.all(axis=1), "NaN 분": low.any(axis=1) & ~low.all(axis=1), "안정 참고[분]": int(states[k].eq("안정 참고").sum())})
H = .9 * len(ids) + 3.4
fig = plt.figure(figsize=(30, H)); top, height = 1 - 2.75 / H, 1 - 3.65 / H
ax = fig.add_axes([.12, .9 / H, .53, height])
for i, (k, d) in enumerate(zip(ids, info)):
    st = states[k]
    for _, run in st.groupby(st.ne(st.shift()).cumsum()):
        ax.barh(i, len(run), left=run.index[0], height=.56, color=STATE_C.get(run.iloc[0], "#eef1f3"), lw=0, zorder=1)
    for col, mask in [(GAP_C, d["공백 분"]), (NAN_C, d["NaN 분"])]:
        ax.vlines(mask.index[mask.to_numpy()] + .5, i - .28, i + .28, color=col, lw=1.6, zorder=3)
    for x, sec in zip(d["간격 위치"], d["간격[초]"]):
        if sec >= 5:                                          # 5초 이상 끊긴 곳만 ▼ + 초
            ax.plot(x, i - .38, "v", color=JUMP_C, ms=11, zorder=4)
            ax.text(x + 4, i - .4, f"{sec:.0f}초", fontsize=12, color=JUMP_C, va="center", fontweight="bold")
ax.set_yticks(range(len(ids)), [f"{k} · {lot_meta.loc[k, 'GRADE']} · {lot_meta.loc[k, 'START']:%m/%d}" for k in ids])
ax.set_ylim(len(ids) - .5, -.7); ax.set_xlim(0, max(len(states[k]) for k in ids) + 25)
style(ax, "관측 시작 후 경과시간[분]"); ax.tick_params(labelsize=13)
ax.set_title("관측 전체 · 색 = 상태 · 세로 눈금 = 값을 쓰지 않은 분 · ▼ = 기록이 5초 이상 끊긴 곳(숫자 = 초)", loc="left", fontsize=14.5, pad=8)
# 오른쪽: LOT끼리 비교하는 작은 막대 4칸 (숫자는 막대 끝)
COLS = [("공백[%]", [d["공백[%]"] for d in info], "{:.1f}%", "#62788a"),
        ("3초 이상 간격[번]", [len(d["간격[초]"]) for d in info], "{:.0f}", JUMP_C),
        ("값을 쓰지 않은 분", [int(d["공백 분"].sum() + d["NaN 분"].sum()) for d in info], "{:.0f}", GAP_C),
        ("안정 참고[분]", [d["안정 참고[분]"] for d in info], "{:.0f}", "#7fa3bf")]
for j, (name, vals, fmt, col) in enumerate(COLS):
    bx = fig.add_axes([.675 + j * .078, .9 / H, .062, height], sharey=ax); bx.axis("off")
    bx.barh(range(len(ids)), vals, height=.45, color=col)
    for i, v in enumerate(vals):
        bx.text(v + max(vals) * .04, i, fmt.format(v), va="center", fontsize=12, color="#17344d")
    for i, d in enumerate(info):   # 덧붙임: 그중 5초 이상 횟수 · 그중 개별 센서 NaN 분
        n5, nn = int((d["간격[초]"] >= 5).sum()), int(d["NaN 분"].sum())
        if name.startswith("3초") and n5:
            bx.text(max(vals) * 1.3, i, f"(5초 이상 {n5})", va="center", fontsize=11.5, color=JUMP_C, fontweight="bold")
        if name.startswith("값을") and nn:
            bx.text(vals[i] + max(vals) * .38, i, f"(NaN {nn})", va="center", fontsize=11.5, color=NAN_C)
    bx.set_xlim(0, max(vals) * (3 if name.startswith("3초") else 1.75))   # 3초 칸은 덧붙임 글자 자리를 더 둔다
    fig.text(.675 + j * .078, top + .35 / H, name, fontsize=13, fontweight="bold", color="#526d80", va="bottom")
header(fig, "LOT별 자료 상태 · 분석에 쓸 수 있는 구간과 빠진 곳",
       "1분 안 기록이 90% 미만인 분은 그 분 값을 쓰지 않는다(보간하지 않음) · 진회색 = 모든 센서가 함께 빈 측정 기록 공백 · 주황 = 시각은 있으나 일부 센서 값이 빈 NaN · ▼ = 기록이 5초 이상 끊긴 곳(평소 1초 간격)",
       [Patch(facecolor=c, label=f"{n}") for n, c in STATE_C.items()] +
       [Line2D([0], [0], color=GAP_C, lw=2.5, label="측정 기록 공백 분"), Line2D([0], [0], color=NAN_C, lw=2.5, label="개별 센서 NaN 분"),
        Line2D([0], [0], marker="v", ls="None", color=JUMP_C, ms=9, label="5초 이상 끊긴 곳 (숫자 = 초)")], ncol=6)
compact_header(fig, .975 - .6 / H, .975 - 1.1 / H, title_size=25, sub_size=14, legend_size=13)
axis_tag(fig, "TIME AXIS · 관측 시작 후 경과시간[분]")
finish(fig, "01_데이터상태/01_관측품질.png")
# 값 점검: 중복 시각 · 값 범위 · 같은 값이 이어진 최장 시간(1초 원본)
check = []
for s in SIGNALS:
    allv = pd.concat([raw[k][s] for k in ids]); run_max, run_lot = 0, None
    for k in ids:
        v = raw[k].sort_values("MEAS_DT")[s]; n = v.groupby(v.ne(v.shift()).cumsum()).transform("size").where(v.notna())
        if n.max() > run_max: run_max, run_lot = int(n.max()), k
    check.append({"신호": s, "최소": allv.min(), "최대": allv.max(), "같은 값 최장 연속[초]": run_max, "그 LOT": run_lot})
check = pd.DataFrame(check)
table(check.assign(최소=check.최소.map(lambda v: f"{v:.3g}"), 최대=check.최대.map(lambda v: f"{v:.4g}")))
all_gap = np.concatenate([d["간격[초]"] for d in info])
note(f"- 중복 시각: {sum(int(raw[k].MEAS_DT.duplicated().sum()) for k in ids)}건. 측정 기록 공백은 LOT마다 {min(d['공백[%]'] for d in info):.1f}~{max(d['공백[%]'] for d in info):.1f}%이고, 대부분 2초 간격(기록 1개 빠짐)이다.\n"
     f"- 기록 간격 3초 이상은 14 LOT 모두 합쳐 {len(all_gap)}번이고 그중 5초 이상은 {int((all_gap >= 5).sum())}번(▼), 가장 긴 것은 {all_gap.max():.0f}초다. "
     f"10초 이상 끊긴 {int((all_gap >= 10).sum())}곳은 그 분 기록이 90% 아래로 내려가 그 분 값을 쓰지 않고(▼ 아래 세로 눈금), 5~6초는 그 분을 그대로 쓴다.\n"
     f"- 값 범위: 센서별 기록값 범위를 표로 확인했다. CP의 물리 단위와 정상 범위는 미확인이다. 출력 0%는 정지·포화 구간이다. CP-Z4M 기록값은 1e-10 크기여서 표시할 때 배율을 붙인다.\n"
     f"- 같은 값 연속: 온도는 최장 {check.loc[check.신호.str.startswith('TC') | check.신호.eq('CLN-IN'), '같은 값 최장 연속[초]'].max()}초라 멈춘 센서는 없다. OP-Z3의 {check.set_index('신호').loc['OP-Z3', '같은 값 최장 연속[초]']}초는 출력이 0%·100%에 붙은 구간(03 2절)이라 센서 멈춤이 아니다.")
# 다음 그림(원본 예시)에 쓸 색과 개별 센서 NaN이 가장 많은 2 LOT
QCOL = {"TIME": "#62788a", "NAN": "#d55e00", "TC": report_viz.sensor_color("TC-Z3"), "OP": report_viz.sensor_color("OP-Z3")}
quality_summary["SENSOR_NAN_PCT"] = quality_summary.SENSOR_NAN / quality_summary.EXPECTED_SECONDS * 100
focus_lots = quality_summary.nlargest(2, "SENSOR_NAN_PCT").LOT.astype(int).tolist()

In [ ]:
_below = []
# 표시용 1초 격자에 놓아 누락 시각에서 선을 끊는다. 값을 보간하거나 0으로 채우지 않는다.
def second_intervals(mask):
    starts=mask & ~mask.shift(1,fill_value=False)
    ends=mask & ~mask.shift(-1,fill_value=False)
    return list(zip(mask.index[starts],mask.index[ends]+pd.Timedelta(seconds=1)))

second_quality={}
for k in focus_lots+[240069]:
    r=lot_meta.loc[k];g=raw[k].set_index("MEAS_DT")
    grid=pd.date_range(r.START,r.END,freq="s")
    present=pd.Series(grid.isin(g.index),index=grid)
    shown=g[SIGNALS].reindex(grid)
    absent=~present
    for s in ["TC-Z3","OP-Z3"]:
        sensor_nan=present & shown[s].isna()
    second_quality[k]=(shown,present)

# 개별 센서 NaN과 별도로 고른다: 240069의 가장 긴 측정 기록 공백.
shown,present=second_quality[240069]
gap=max(second_intervals(~present),key=lambda z:z[1]-z[0])
cases=[(k,lot_meta.loc[k,"START"],lot_meta.loc[k,"START"]+pd.Timedelta(minutes=5),"개별 센서 NaN · 첫 5분") for k in focus_lots]
cases.append((240069,gap[0]-pd.Timedelta(seconds=10),gap[1]+pd.Timedelta(seconds=10),"측정 기록 공백 · 최장 공백 주변"))
note("**그림 목적 — 측정 기록 공백·센서 NaN 비교:** 왼쪽 TC-Z3와 오른쪽 OP-Z3를 같은 실제 시각에서 대조한다. 위 두 LOT은 시각 기록이 있으나 OP 값이 NaN인 경우, 마지막 LOT은 모든 센서의 측정 기록이 함께 비는 경우다. 두 종류의 빈 선과 실제 누락 개수를 구분한다.")
fig,axs=plt.subplots(3,2,figsize=(24,15))
fig.subplots_adjust(left=.10,right=.97,top=.77,bottom=0.07,wspace=.30,hspace=.65)
for i,(k,a,b,reason) in enumerate(cases):
    shown,present=second_quality[k]
    values=shown.loc[a:b];has_row=present.loc[a:b]
    for j,(s,key) in enumerate([("TC-Z3","TC"),("OP-Z3","OP")]):
        ax=axs[i,j]
        # 표시 음영은 마지막 관측점부터 다음 관측점까지 잇는다.
        # 실제 누락 수는 [start, end) 안의 예상 1초 시각 수로 계산한다.
        for start,end in second_intervals(~present):
            shade_start=max(a,start-pd.Timedelta(seconds=1))
            shade_end=min(b,end)
            if shade_start < shade_end:
                ax.axvspan(shade_start,shade_end,color=QCOL["TIME"],alpha=.45,lw=0,zorder=0)
        for start,end in second_intervals(has_row & values[s].isna()):
            ax.axvspan(start,end,color=QCOL["NAN"],alpha=.20,lw=0)
        ax.plot(values.index,values[s],color=QCOL[key],ls=report_viz.signal_linestyle(s),label=s,lw=1.8)
        ax.set_xlim(a,b)
        ax.set_title(f"{k} · {lot_meta.loc[k,'GRADE']} · 가열·균열 [{s}] · {a:%m/%d}",loc="left",pad=12)
        ax.set_ylabel(unit(s),labelpad=10);style(ax,"실제 관측 시각[시:분:초]")
        ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=2) if i<2 else mdates.SecondLocator(interval=10))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.yaxis.set_major_locator(MaxNLocator(4))
        if s=="OP-Z3":
            ax.set_ylim(-3,103);ax.set_yticks([0,25,50,75,100])
        else:
            ax.set_ylim(np.floor(values[s].min())-1,np.ceil(values[s].max())+1)
        if i<2 and s=="OP-Z3":
            missing=values.index[has_row & values[s].isna()]
            duration=int((missing.max()-missing.min()).total_seconds()+1)
            midpoint=missing.min()+(missing.max()-missing.min())/2
            ax.text(midpoint,.50,f"개별 센서 NaN · {len(missing)}개\n처음~마지막 NaN 범위 {duration}초",
                    transform=ax.get_xaxis_transform(),ha="center",va="center",fontsize=11.5,color=QCOL["NAN"])
        elif i==2:
            missing_count=int((gap[1]-gap[0]).total_seconds())
            prev_time,next_time=gap[0]-pd.Timedelta(seconds=1),gap[1]
            midpoint=prev_time+(next_time-prev_time)/2
            ax.text(midpoint,.50,f"측정 기록 공백 · {missing_count}초",
                    transform=ax.get_xaxis_transform(),ha="center",va="center",fontsize=11.5,color=QCOL["TIME"])
header(fig,"측정 기록 공백·센서 NaN 비교","TC 왼쪽 · OP 오른쪽; 회색=모든 센서의 측정 기록 공백 · 주황=시각 기록이 있는 개별 센서 NaN · 빈 선은 값 0이 아님",
       [Line2D([0],[0],color=QCOL["TC"],lw=2,label="가열·균열 [TC-Z3] 원본"),
        Line2D([0],[0],color=QCOL["OP"],lw=2,label="가열·균열 [OP-Z3] 원본"),
        Patch(facecolor=QCOL["TIME"],alpha=.35,label="측정 기록 공백"),
        Patch(facecolor=QCOL["NAN"],alpha=.3,label="개별 센서 NaN")],ncol=4)
_below.append("확인: 회색은 모든 센서의 측정 기록 공백, 주황은 시각 기록이 있지만 해당 센서 값이 NaN인 구간입니다. 빈 선은 값 0·급락을 뜻하지 않습니다.")
gap_missing_s=int((gap[1]-gap[0]).total_seconds())
gap_display_s=int((gap[1]-(gap[0]-pd.Timedelta(seconds=1))).total_seconds())
_below.append(f"표시: 회색은 앞뒤 관측점까지 연결합니다. 240069의 음영 폭 {gap_display_s}초와 실제 누락 {gap_missing_s}개(1초 간격 예상 시각)는 구분합니다.")
axis_tag(fig, TAG_REAL)
finish(fig,"01_데이터상태/02_기록공백_센서결측_원본.png")
note("\n".join("- " + t.strip() for t in _below))
for k in focus_lots:
    g=raw[k];missing=g.loc[g["OP-Z3"].isna(),"MEAS_DT"]
    envelope_s=(missing.max()-missing.min()).total_seconds()+1
    nan_s=int(len(missing))
    common_s=int((~second_quality[k][1].loc[missing.min():missing.max()]).sum())
    note(f"**그림 결과 — {k}:** OP-Z3의 초기 결측은 {missing.min():%m/%d %H:%M:%S}–{missing.max():%H:%M:%S}, 최초~마지막 NaN을 포함한 {envelope_s:.0f}초 범위에 있다. 이 범위에서 OP 센서 NaN {nan_s}개와 측정 기록 공백 {common_s}초를 구분한다. 같은 범위 TC-Z3에는 {int(second_quality[k][0].loc[missing.min():missing.max(),'TC-Z3'].count())}개 유효 값이 있다.")
note(f"**그림 결과 — 240069:** {gap[0]:%m/%d %H:%M:%S}–{gap[1]-pd.Timedelta(seconds=1):%H:%M:%S}의 {(gap[1]-gap[0]).total_seconds():.0f}개 예상 시각(1초 간격)에 측정 기록이 없다. 모든 22개 센서의 기록이 함께 비어 TC와 OP 선이 끊긴다. 회색 음영은 앞 관측 {gap[0]-pd.Timedelta(seconds=1):%H:%M:%S}부터 다음 관측 {gap[1]:%H:%M:%S}까지 이어 흰 여백을 없앴다. 측정 기록 공백은 개별 센서 NaN과 구분하며, 온도·출력 급변이나 설비 이상으로 판정하지 않는다. 통신·수집·센서 중 어느 원인인지는 CSV만으로 확인되지 않는다.")
note("**분석 결정:** 두 OP-Z3 초기 결측은 센서·시간 범위를 표시하고 TC↔OP 비교 품질 조건에 반영한다. 측정 기록 공백은 전체 LOT별 품질 마스크로 계속 확인한다. 두 종류 모두 보간·0 채움 없이 원본으로 유지하며, 유효율이 높다는 이유로 정상 라벨을 붙이지 않는다.")

In [ ]:
# 원칙 2: 수준은 같은 강종 안에서만 비교한다. 세 조건 모두 데이터에서 정한다(따로 정한 문턱 숫자 없음).
#  (가) 신호가 강종을 따를 것: 강종으로 설명되는 정도 > 우연 수준 (강종 수 - 1) / (LOT 수 - 1)
#  (나) 혼자 떨어질 것: 가장 가까운 같은 강종 LOT과의 차이 > 다른 LOT끼리의 폭 (같은 강종 3 LOT 이상)
#  (다) 차이가 클 것: 그 차이 > 그 신호가 LOT 안에서 평소 흔들리는 폭 (10~90%, 14 LOT 중앙값)
stable = lambda k, s: profile[k][(s, "median")].where(quality[k][s].ge(.9) & states[k].eq("안정 참고")).dropna()
ids = lots.sort_values("START").LOT_NO.astype(int).tolist(); grade_of = lot_meta.loc[ids, "GRADE"]
level_med = pd.DataFrame({s: {k: stable(k, s).median() for k in ids} for s in SIGNALS})
wiggle = {s: np.median([stable(k, s).quantile(.9) - stable(k, s).quantile(.1) for k in ids]) for s in SIGNALS}
chance = (grade_of.nunique() - 1) / (len(ids) - 1)
follow = {s: 1 - (level_med[s] - level_med[s].groupby(grade_of).transform("mean")).var() / level_med[s].var() for s in SIGNALS}
unit = lambda s: "%p" if s.startswith("OP") else "℃"
level_rows = []
for s in [s for s in SIGNALS if follow[s] > chance]:
    for g, members in grade_of.groupby(grade_of):
        ks = members.index.tolist()
        if len(ks) == 2:
            d = level_med.loc[ks[1], s] - level_med.loc[ks[0], s]
            if abs(d) > wiggle[s]:
                level_rows.append({"판정": "2 LOT 차이(판단 불가)", "강종": g, "LOT": " · ".join(map(str, ks)), "신호": s, "차이": abs(d), "단위": unit(s),
                                   "다른 LOT끼리 폭": np.nan, "LOT 안 흔들림": wiggle[s]})
        for k in (ks if len(ks) >= 3 else []):
            others = level_med.loc[[o for o in ks if o != k], s]
            nearest = others.loc[(others - level_med.loc[k, s]).abs().idxmin()]
            if abs(level_med.loc[k, s] - nearest) > others.max() - others.min() and abs(level_med.loc[k, s] - nearest) > wiggle[s]:
                level_rows.append({"판정": "혼자 떨어짐", "강종": g, "LOT": k, "신호": s, "차이": level_med.loc[k, s] - nearest, "단위": unit(s),
                                   "다른 LOT끼리 폭": others.max() - others.min(), "LOT 안 흔들림": wiggle[s]})
level_issues = pd.DataFrame(level_rows)
signal_rule = pd.DataFrame({"강종으로 설명되는 정도": follow, "LOT 안 흔들림(10~90%)": wiggle}).round(2)
signal_rule["수준 비교"] = np.where(signal_rule["강종으로 설명되는 정도"] > chance, "같은 강종끼리 비교", "비교 안 함 (강종과 무관)")
table(signal_rule.reset_index().rename(columns={"index": "신호"}))
table(level_issues.round(2))
note(f"- 우연 수준은 {chance:.2f}다. 14 LOT을 {grade_of.nunique()}개 강종으로 나누면 강종과 관계없는 값도 평균 이만큼은 설명되는 것처럼 나온다.\n"
     f"- 강종과 무관해 수준을 비교하지 않는 신호: {', '.join(signal_rule.index[signal_rule['수준 비교'].str.startswith('비교 안 함')])}.\n"
     f"- 세 조건을 모두 넘은 것: " + ", ".join(f"{r.LOT} {r.신호} {r.차이:+.1f}{r.단위}" for r in level_issues.loc[level_issues.판정.eq('혼자 떨어짐')].itertuples()) + ".")


In [ ]:
# 탐지용 1분 통계를 만든다. 온도 예측 목표나 정답 라벨을 만들지 않는다.
# 작업 REMARK·교체·사후 상태·후보 플래그·전체 LOT 통계는 온라인 특징에 넣지 않는다.
def features_for_lot(g,lot,grade,start):
    g=g.sort_values("MEAS_DT").copy()
    g["MINUTE"]=np.floor((g.MEAS_DT-start).dt.total_seconds()/60).astype(int)
    grid=pd.RangeIndex(g.MINUTE.max()+1,name="MINUTE")
    grouped=g.groupby("MINUTE")[SIGNALS]
    agg=grouped.agg(["median","min","max","count"]).reindex(grid)
    iqr=(grouped.quantile(.75)-grouped.quantile(.25)).reindex(grid)
    result={"LOT":np.full(len(grid),lot),"GRADE":np.full(len(grid),grade),"MINUTE":grid.to_numpy(),"WINDOW_END_TIME":start+pd.to_timedelta(grid+1,unit="min")}
    last=g.groupby("MINUTE").MEAS_DT.max().reindex(grid)
    result["MAX_SOURCE_TIME"]=last.to_numpy()
    result["GRADE_EVENT_TIME"]=start
    for s in SIGNALS:
        # 온라인에서는 닫힌 60초 창을 기다린다. 마지막 짧은 관측도 60초를 분모로 사용한다.
        q=agg[(s,"count")].fillna(0)/60
        y=agg[(s,"median")].where(q.ge(.9))
        result[s+"_CURRENT"]=y.to_numpy()
        result[s+"_IQR"]=iqr[s].where(q.ge(.9)).to_numpy()
        result[s+"_VALID_RATIO_60"]=q.to_numpy()
        result[s+"_DELTA1"]=y.diff().to_numpy()
        past=y.shift(1)
        result[s+"_PAST5_MEDIAN"]=past.rolling(5,min_periods=3).median().to_numpy()
        result[s+"_PAST5_STD"]=past.rolling(5,min_periods=3).std().to_numpy()
        result[s+"_PAST60_Q95_Q05"]=(past.rolling(60,min_periods=30).quantile(.95)-past.rolling(60,min_periods=30).quantile(.05)).to_numpy()
        if s in ["TC-Z9","TC-OUT1","TC-OUT2"]:
            result[s+"_PAST300_Q95_Q05"]=(past.rolling(300,min_periods=180).quantile(.95)-past.rolling(300,min_periods=180).quantile(.05)).to_numpy()
    return pd.DataFrame(result)

feature_frames=[features_for_lot(raw[int(r.LOT_NO)],int(r.LOT_NO),r.GRADE,r.START) for r in lots.itertuples()]
features=pd.concat(feature_frames,ignore_index=True).sort_values(["WINDOW_END_TIME","LOT"]).reset_index(drop=True)
feature_columns=[c for c in features.columns if c not in ["LOT","GRADE","MINUTE","WINDOW_END_TIME","MAX_SOURCE_TIME","GRADE_EVENT_TIME"]]


In [ ]:
# 4단계: 두 가지 탐지(범위 기준선·이동 Z-score)를 같은 1분 단위로 맞춘다.
Z_WINDOW, Z_MIN, Z_LIMIT = 30, 20, 4                  # 이전 30분(최소 20분)의 중앙값·표준편차, |Z| ≥ 4
SD_FLOOR = {"TC": .5, "OP": 2.0}                      # 아주 작은 표준편차로 Z가 커지지 않게 하는 최소값
Z_SIGNALS = NONCYCLIC + [f"OP-Z{z}" for z in range(1, 9)]   # 이동 Z를 계산하는 신호(구간마다 가장 크게 움직인 센서를 고를 때도 쓴다)

def change_features(g):
    """한 LOT의 1분 값에서 이전 30분 대비 Z를 만든다. 현재 분은 기준에서 뺀다."""
    g = g.sort_values("MINUTE").set_index("MINUTE")
    out = pd.DataFrame({"LOT": g.LOT.iloc[0]}, index=g.index)
    for s in Z_SIGNALS:
        current = g[s + "_CURRENT"]
        past = current.shift(1)
        center = past.rolling(Z_WINDOW, min_periods=Z_MIN).median()
        spread = past.rolling(Z_WINDOW, min_periods=Z_MIN).std().clip(lower=SD_FLOOR["TC" if s in TC else "OP"])
        out[s + "_Z30"] = (current - center) / spread
    return out.rename_axis("MINUTE").reset_index()

detect = pd.concat([change_features(g) for _, g in features.groupby("LOT")], ignore_index=True)

# (1) 조건별 범위 기준선: 같은 강종·안정 참고·같은 경과 분 대조 중앙값과 |차이| ≥ 3℃, 대조 2 LOT 이상, 3분 연속
detect["RULE_FLAG"] = False
for e in episodes.itertuples():
    detect.loc[detect.LOT.eq(e.LOT) & detect.MINUTE.between(e.START_MIN, e.END_MIN - 1), "RULE_FLAG"] = True
detect["RULE_SCORED"] = [any(comparison[lot, s].N_PEERS.get(m, 0) >= 2 and pd.notna(comparison[lot, s].DELTA.get(m))
                             for s in NONCYCLIC) for lot, m in zip(detect.LOT, detect.MINUTE)]

# (2) 이동 Z-score: 같은 비주기 TC에서 이전 30분 대비 |Z| ≥ 4
z_columns = [s + "_Z30" for s in NONCYCLIC]
detect["Z_FLAG"] = detect[z_columns].abs().ge(Z_LIMIT).any(axis=1)
detect["Z_SCORED"] = detect[z_columns].notna().any(axis=1)

# 분마다 알려진 상황: 사후 상태 → 실제 작업 → 강종변경 직후 30분 → 그 외 (앞의 것이 우선)
state_rows = pd.concat([pd.DataFrame({"LOT": k, "MINUTE": s.index, "STATE": s.to_numpy()}) for k, s in states.items()])
detect = detect.merge(state_rows, on=["LOT", "MINUTE"], how="left", validate="one_to_one")
detect["WORK"] = [any(m < b and m + 1 > a for a, b in work_ranges[lot]) for lot, m in zip(detect.LOT, detect.MINUTE)]
CONTEXTS = ["초기 승온", "종료 전환", "실제 작업 겹침", "강종변경 직후 30분", "그 외"]
detect["CONTEXT"] = np.select([detect.STATE.eq("초기 승온"), detect.STATE.eq("종료 전환"), detect.WORK, detect.MINUTE.lt(30)],
                              CONTEXTS[:4], default="그 외")
METHODS = {"RULE": "조건별 범위 기준선", "Z": "이동 Z-score"}

# 연속으로 표시된 분을 하나의 탐지 구간으로 묶는다. CP 후보는 기존 CP 규칙의 구간을 그대로 쓴다.
interval_rows = []
for key, name in METHODS.items():
    for lot, d in detect.groupby("LOT"):
        d = d.set_index("MINUTE")
        for a, b in runs(d[key + "_FLAG"]):
            block = d.loc[a:b - 1]
            if key == "RULE":
                sensors = episodes.loc[episodes.LOT.eq(lot) & episodes.START_MIN.lt(b) & episodes.END_MIN.gt(a), "SENSOR"].unique()
            else:
                strength = block[[s + "_Z30" for s in Z_SIGNALS]].abs().max().dropna().sort_values(ascending=False)
                sensors = [c.replace("_Z30", "") for c in strength.index[:2]]
            interval_rows.append({"METHOD": name, "LOT": lot, "START_MIN": a, "END_MIN": b, "MINUTES": b - a, "SENSORS": " · ".join(sensors)})
for c in cp_candidates.itertuples():
    interval_rows.append({"METHOD": "CP 규칙", "LOT": c.LOT, "START_MIN": int(np.floor(c.START_MIN)), "END_MIN": int(np.ceil(c.END_MIN)),
                          "MINUTES": int(np.ceil(c.END_MIN)) - int(np.floor(c.START_MIN)), "SENSORS": "CP-Z4"})
intervals = pd.DataFrame(interval_rows)
intervals["START"] = intervals.LOT.map(lot_meta.START) + pd.to_timedelta(intervals.START_MIN, unit="min")
intervals["END"] = intervals.LOT.map(lot_meta.START) + pd.to_timedelta(intervals.END_MIN, unit="min")
intervals["GRADE"] = intervals.LOT.map(lot_meta.GRADE)

In [ ]:
# 운전 상태별 급변 크기: 같은 LOT 직전 30분 대비 |Z|(6절 탐지 규칙)를 신호별 '안정 운전 99% 값'으로 나눈 배수로 본다.
#   대상 신호 17개 = 온도 9개(CLN-IN, TC-Z1~Z8) + 출력 8개(OP-Z1~Z8). Z9·냉각은 원래 반복 파형이라, CP는 02의 5절 규칙으로 따로 본다.
#   정지 = 정기수리 직전 노를 끄는 구간: 실제 온도 하강 시작(가열 Z4~Z6 평균이 안정 수준 -1℃ 위에 있던 마지막 분의 다음 분) 5분 전부터 관측 끝까지
#          (과시효 Z7·Z8은 가열 구역이 -1℃를 넘기 전에 먼저 내려가기 시작한다)
#   승온 = 정기수리 뒤 노를 다시 켜는 구간: 관측 시작부터 승온 끝 뒤 10분까지(과시효 Z7·Z8이 가열 구역보다 늦게 따라 오르는 꼬리 포함)
#   안정 운전 = 정지·승온을 뺀 14 LOT의 나머지 모든 시간(Z3는 평소에도 출렁여 따로 표시)
def real_stop_min(k):
    heat = profile[k][[(f"TC-Z{z}", "median") for z in (4, 5, 6)]].mean(axis=1)
    level = heat[states[k].eq("안정 참고")].median()
    return int(heat.index[heat.ge(level - 1)].max()) + 1
stop_from = {int(e.LOT): real_stop_min(int(e.LOT)) - 5 for e in expected.loc[expected.종류.eq("예정된 정지")].itertuples()}
warm_to = {int(e.LOT): e.END_MIN + 10 for e in expected.loc[expected.종류.eq("재가동 승온")].itertuples()}
zc = [c for c in detect.columns if c.endswith("_Z30")]
jump = detect.melt(id_vars=["LOT", "MINUTE"], value_vars=zc, var_name="SENSOR", value_name="Z").dropna()
jump["SENSOR"] = jump.SENSOR.str.replace("_Z30", "")
jump["상태"] = np.select([[k in stop_from and m >= stop_from[k] for k, m in zip(jump.LOT, jump.MINUTE)],
                         [k in warm_to and m < warm_to[k] for k, m in zip(jump.LOT, jump.MINUTE)],
                         jump.SENSOR.isin(["TC-Z3", "OP-Z3"])],
                        ["정지", "승온", "Z3 (안정 운전)"], default="그 밖 신호 (안정 운전)")
stable_jump = jump.loc[jump.상태.str.endswith("(안정 운전)")]
p99 = stable_jump.Z.abs().groupby(stable_jump.SENSOR).quantile(.99)       # 신호마다 안정 운전의 99% 값
jump["배수"] = jump.Z.abs() / jump.SENSOR.map(p99)
jump["시각"] = jump.LOT.map(lot_meta.START) + pd.to_timedelta(jump.MINUTE, unit="min")
at_min = lambda k, m: lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(m))

STATE_ORDER = ["정지", "승온", "Z3 (안정 운전)", "그 밖 신호 (안정 운전)"]
STATE_C = {"정지": "#c83745", "승온": "#e69f00", "Z3 (안정 운전)": "#0072b2", "그 밖 신호 (안정 운전)": "#7c8995"}
def state_jump_figure():
    """(1) 계산 예시 · (2) 99% 값 · (3) 상태 × 신호 · (4) 상위 15개. 그림 안 이름이 밖으로 새지 않게 함수로 묶는다."""
    fig = plt.figure(figsize=(30, 24))
    outer = fig.add_gridspec(2, 1, height_ratios=[1, 1.6], left=.06, right=.98, top=.875, bottom=.05, hspace=.3)
    top_gs = outer[0].subgridspec(1, 2, width_ratios=[1.7, 1], wspace=.16)
    low_gs = outer[1].subgridspec(1, 2, width_ratios=[1, 1.15], wspace=.42)
    ax, bx = fig.add_subplot(top_gs[0]), fig.add_subplot(top_gs[1])
    cx, dx = fig.add_subplot(low_gs[0]), fig.add_subplot(low_gs[1])

    # (1) 계산 예시: 급변이 가장 컸던 한 점을 직접 계산한다
    ex = jump.sort_values("배수").iloc[-1]; k, s, m = int(ex.LOT), ex.SENSOR, int(ex.MINUTE)
    y1 = features.loc[features.LOT.eq(k)].set_index("MINUTE")[s + "_CURRENT"]
    past = y1.loc[m - Z_WINDOW:m - 1]; center = past.median()
    floor = SD_FLOOR["TC" if s in TC else "OP"]; spread = max(past.std(), floor)
    unit = "℃" if s in TC else "%"
    show = y1.loc[m - 80:]
    ax.axvspan(at_min(k, stop_from[k]), at_min(k, show.index.max() + 1), color=STATE_C["정지"], alpha=.08, lw=0)
    ax.axvspan(at_min(k, m - Z_WINDOW), at_min(k, m), color="#0072b2", alpha=.12, lw=0)
    ax.fill_between([at_min(k, m - Z_WINDOW), at_min(k, m)], center - spread, center + spread, color="#0072b2", alpha=.18, lw=0)
    ax.plot([at_min(k, m - Z_WINDOW), at_min(k, m)], [center, center], color="#0072b2", lw=2.4)
    ax.plot([at_min(k, i) for i in show.index], show.values, color="#17344d", lw=1.6, marker="o", ms=3)
    ax.plot(at_min(k, m), y1.loc[m], "o", color=STATE_C["정지"], ms=14, zorder=5)
    ax.annotate("", xy=(at_min(k, m), y1.loc[m]), xytext=(at_min(k, m), center), arrowprops=dict(arrowstyle="->", color=STATE_C["정지"], lw=2))
    ax.text(at_min(k, m - Z_WINDOW / 2), center + spread + 2, f"직전 30분 {at_min(k, m - Z_WINDOW):%H:%M}–{at_min(k, m - 1):%H:%M}\n중앙값 {center:.1f}{unit} · 표준편차 {past.std():.1f} → 최소 {floor:g} 적용",
            ha="center", va="bottom", fontsize=12.5, color="#0072b2", fontweight="bold")
    ax.text(at_min(k, stop_from[k]) + pd.Timedelta(minutes=1), show.max() * 1.29, f"정지 시작 {at_min(k, stop_from[k]):%H:%M}\n(실제 온도 하강 5분 전)", ha="left", va="top", fontsize=12.5, color=STATE_C["정지"])
    z = (y1.loc[m] - center) / spread
    ax.text(.012, .05, "\n".join([f"① 1분 값 = 그 1분 1초 값의 중앙값: {at_min(k, m):%H:%M} {s} = {y1.loc[m]:.1f}{unit}",
                                  f"② 직전 30분 기준: 중앙값 {center:.1f}{unit}, 흔들림 {spread:.1f}",
                                  f"③ Z = ({y1.loc[m]:.1f} - {center:.1f}) ÷ {spread:.1f} = {z:+.1f}",
                                  f"④ 급변 크기 = |Z| ÷ {s} 안정 운전 99% 값 {p99[s]:.2f} = {abs(z) / p99[s]:.1f}배  → (2)"]),
            transform=ax.transAxes, ha="left", va="bottom", fontsize=13.5, color="#17344d", linespacing=1.6,
            bbox=dict(boxstyle="round,pad=.6", facecolor="white", edgecolor="#9aa6ae"))
    ax.set_ylim(0, show.max() * 1.32); ax.set_ylabel(f"{s} 1분 값[{unit}]", fontsize=13)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M")); style(ax, "실제 날짜·시각"); ax.tick_params(labelsize=12)
    ax.set_title(f"(1) 계산 예시 — {k} {s} {at_min(k, m):%m/%d %H:%M} · 파란 띠 = 직전 30분 기준(중앙값 ± 흔들림)", loc="left", fontsize=15, pad=14)

    # (2) 그 신호의 안정 운전 |Z| 분포와 99% 값
    zs = stable_jump.loc[stable_jump.SENSOR.eq(s), "Z"].abs()
    bx.hist(zs, bins=np.arange(0, abs(z) + 1, .25), color="#9aa6ae")
    bx.axvline(p99[s], color="#17344d", ls="--", lw=1.6)
    bx.axvline(abs(z), color=STATE_C["정지"], lw=2.4)
    ymax = bx.get_ylim()[1]
    bx.text(p99[s] + .15, ymax * .7, f"99% 값 = {p99[s]:.2f}\n안정 운전 {len(zs):,}분 중\n99%가 이 아래", ha="left", va="center", fontsize=12.5, color="#17344d")
    bx.text(abs(z) - .15, ymax * .45, f"정지 {at_min(k, m):%H:%M}\n|Z| = {abs(z):.1f}\n= 99% 값의 {abs(z) / p99[s]:.1f}배", ha="right", va="center", fontsize=13, color=STATE_C["정지"], fontweight="bold")
    style(bx, f"{s} |Z|"); bx.tick_params(labelsize=12); bx.set_ylabel("분 수", fontsize=13)
    bx.set_title(f"(2) {s}의 안정 운전 |Z| 분포 — 99% 값으로 나눠 신호끼리 비교", loc="left", fontsize=15, pad=14)

    # (3) 상태 × 신호: 그 상태에서 가장 큰 급변 크기
    cols = ["정지", "승온", "안정 운전"]
    mx = jump.assign(열=jump.상태.where(~jump.상태.str.endswith("(안정 운전)"), "안정 운전")).pivot_table(index="SENSOR", columns="열", values="배수", aggfunc="max")
    rows = [f"TC-Z{z}" for z in range(1, 9)] + ["CLN-IN"] + [f"OP-Z{z}" for z in range(1, 9)]
    mx = mx.reindex(index=rows, columns=cols)
    cx.imshow(mx.to_numpy(), cmap="Reds", vmin=0, vmax=8, aspect="auto")
    for i, sig in enumerate(rows):
        for j, c in enumerate(cols):
            v = mx.loc[sig, c]; z3 = c == "안정 운전" and sig in ("TC-Z3", "OP-Z3")
            cx.text(j, i, f"{v:.1f}" + (" (Z3)" if z3 else ""), ha="center", va="center", fontsize=13,
                    color="white" if v >= 4.5 else "#0072b2" if z3 else "#17344d", fontweight="bold" if v >= 3 else "normal")
    cx.axhline(8.5, color="#17344d", lw=2)
    minutes = jump.drop_duplicates(["LOT", "MINUTE"]).assign(열=lambda d: d.상태.where(~d.상태.str.endswith("(안정 운전)"), "안정 운전")).열.value_counts()
    heads = {"정지": "정지 · 정기수리 직전\n" + "\n".join(f"{q} {at_min(q, a):%m/%d %H:%M}–{lot_meta.loc[q, 'END']:%H:%M}" for q, a in stop_from.items()),
             "승온": "승온 · 정기수리 직후\n" + "\n".join(f"{q} {lot_meta.loc[q, 'START']:%m/%d %H:%M}–{at_min(q, b):%H:%M}" for q, b in warm_to.items()),
             "안정 운전": "안정 운전\n14 LOT의 나머지 시간\n(정지·승온 제외)"}
    cx.xaxis.tick_top(); cx.set_xticks(range(3), [heads[c] + f"\n{minutes[c]:,}분" for c in cols], fontsize=12.5)
    cx.set_yticks(range(len(rows)), [f"{r} {'온도' if r.startswith(('TC', 'CLN')) else '출력'}" for r in rows], fontsize=12.5)
    cx.tick_params(length=0)
    cx.set_title("(3) 상태 × 신호 17개 · 칸 = 그 상태에서 가장 큰 급변 크기[배] · 위 9줄 온도, 아래 8줄 출력", loc="left", fontsize=15, pad=16)

    # (4) 14 LOT 급변 상위 15개
    top = jump.sort_values("배수", ascending=False).drop_duplicates(["LOT", "SENSOR"]).head(15)
    y = np.arange(len(top))[::-1]
    dx.barh(y, top.배수, color=[STATE_C[s_] for s_ in top.상태], height=.7)
    dx.set_yticks(y, [f"{r.LOT} · {r.SENSOR} · {r.시각:%m/%d %H:%M}" for r in top.itertuples()], fontsize=12.5)
    for yy, r in zip(y, top.itertuples()):
        dx.text(r.배수 + .08, yy, f"{r.배수:.1f}배 · {r.상태}", va="center", fontsize=12.5, color=STATE_C[r.상태])
    dx.axvline(1, color="#455a64", ls="--", lw=1.3)
    dx.set_xlim(0, top.배수.max() * 1.45)
    style(dx, "급변 크기[배] · 점선 1배 = 안정 운전 100분 중 1분 나오는 크기"); dx.tick_params(axis="x", labelsize=12)
    n_sched = int(top.상태.isin(["정지", "승온"]).sum())
    dx.set_title(f"(4) 14 LOT 급변 상위 15개(LOT·신호마다 1개) — {n_sched}개가 정지·승온", loc="left", fontsize=15, pad=14)
    header(fig, "운전 상태별 급변 — 정지·승온을 따로 떼어야 하는 이유",
           "급변 크기[배] = |Z| ÷ 그 신호의 안정 운전 |Z| 99% 값 · Z = (이번 1분 값 - 같은 LOT 직전 30분 중앙값) ÷ 직전 30분 표준편차(최소 온도 0.5℃·출력 2%p)",
           [Patch(facecolor=STATE_C[s_], alpha=.8, label=s_) for s_ in STATE_ORDER], ncol=4)
    compact_header(fig, .955, .935, title_size=24, sub_size=13, legend_size=13)
    axis_tag(fig, compare_tag("운전 상태별 급변"))
    finish(fig, "02_근거범위/00_운전상태별_급변.png")
    other_max = jump.loc[jump.상태.eq("그 밖 신호 (안정 운전)")].sort_values("배수").iloc[-1]
    note("\n".join([
        f"- **(4)** 급변 상위 15개 중 {n_sched}개가 정지·승온 구간이다(정지 최대 {mx['정지'].max():.1f}배, 승온 최대 {mx['승온'].max():.1f}배). 이 구간을 떼지 않으면 어떤 급변 순위도 계획된 정지·재가동이 차지한다.",
        f"- **(3)** 정지에서는 출력 Z5~Z8과 과시효 온도 Z7·Z8이 3.6~7.4배로 크고, 승온에서는 과시효 온도 Z7·Z8만 3.6~3.8배다. 안정 운전에서는 Z3(최대 {mx.loc[['TC-Z3', 'OP-Z3'], '안정 운전'].max():.1f}배)를 빼면 모두 2배 이내이고, 가장 큰 것은 {other_max.LOT} {other_max.SENSOR} {other_max.시각:%m/%d %H:%M}의 {other_max.배수:.1f}배다. 따로 확대할 급변 구간은 없다.",
        f"- 정지로 본 시작(실제 온도 하강 5분 전): " + ", ".join(f"{q} {at_min(q, a):%m/%d %H:%M}" for q, a in stop_from.items())
        + " · 승온 끝 + 10분: " + ", ".join(f"{q} {at_min(q, b):%m/%d %H:%M}" for q, b in warm_to.items()) + "."]))


state_jump_figure()

In [ ]:
# 정비 일정 전후 정지·재가동 — '설명됨'의 근거
# (a) 14 LOT의 끝 5분 하강·처음 80~100분 상승 비교, (b)~(e) 정지·재가동 구간에서 구역별 온도·출력이 같은 방향으로 움직였는지
HEAT_OP = [f"OP-Z{z}" for z in range(3, 7)]
OVER = ["TC-Z7", "TC-Z8"]
role = dict(zip(expected.LOT, expected["수리 관계"]))
def med(p, cols):
    return p.loc[:, [(s, "median") for s in cols]]
judge_rows = []
for e in expected.itertuples():
    p = profile[e.LOT]
    if e.종류 == "예정된 정지":
        before, after = p.loc[e.START_MIN - 10:e.START_MIN - 1], p.loc[e.END_MIN - 5:]
    else:
        before, after = p.loc[:4], p.loc[e.END_MIN - 5:e.END_MIN - 1]
    heat = (med(after, HEAT).median() - med(before, HEAT).median()).to_numpy()
    over = (med(after, OVER).median() - med(before, OVER).median()).to_numpy()
    op0, op1 = med(before, HEAT_OP).median().median(), med(after, HEAT_OP).median().median()
    same = (heat < 0).all() and (over < 0).all() if e.종류 == "예정된 정지" else (heat > 0).all() and (over > 0).all()
    judge_rows.append({"LOT": e.LOT, "GRADE": e.GRADE, "종류": e.종류, "실제 시각": f"{e.시작:%m/%d %H:%M}–{e.끝:%H:%M}", "수리 관계": role[e.LOT],
                       "가열 Z3~Z6 온도 변화[℃]": f"{heat.min():+.1f} ~ {heat.max():+.1f}", "과시효 Z7·Z8 온도 변화[℃]": f"{over.min():+.1f} ~ {over.max():+.1f}",
                       "가열 출력 중앙값[%]": f"{op0:.0f} → {op1:.0f}", "판정": "모든 구역이 같은 방향 → 예정된 변화" if same else "다르게 움직인 구역 있음 → 확인 필요"})
stop_judge = pd.DataFrame(judge_rows)
level_rows = []
for k in lots.LOT_NO.astype(int):
    p = med(profile[k], HEAT); m = p.index.max()
    drop = (p.loc[m - 4:].median() - p.loc[m * .45:m * .6].median()).min()
    rise = (p.loc[80:100].median() - p.loc[:4].median()).max() if m >= 100 else np.nan
    level_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "끝 5분 하강[℃]": drop, "처음 80~100분 상승[℃]": rise})
end_rise = pd.DataFrame(level_rows).set_index("LOT")
big = end_rise.index[end_rise["끝 5분 하강[℃]"].le(-10) | end_rise["처음 80~100분 상승[℃]"].ge(20)].tolist()

fig = plt.figure(figsize=(30, 15))
gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1, 1], left=.085, right=.955, top=.83, bottom=.07, hspace=.62, wspace=.36)
ax = fig.add_subplot(gs[:, 0]); y = np.arange(len(end_rise))
ax.barh(y, end_rise["끝 5분 하강[℃]"], color="#0072b2", height=.6)
ax.barh(y, end_rise["처음 80~100분 상승[℃]"].fillna(0), color="#e69f00", height=.6)
for lim in [-10, 20]:
    ax.axvline(lim, color="#526d80", ls=":", lw=1.4)
ax.axvline(0, color="#9fb3c3", lw=1)
for yy, lo, hi in zip(y, end_rise["끝 5분 하강[℃]"], end_rise["처음 80~100분 상승[℃]"]):
    if lo <= -10: ax.text(lo - 1, yy, f"{lo:.0f}", ha="right", va="center", fontsize=13, color="#0072b2", fontweight="bold")
    if hi >= 20: ax.text(hi + 1, yy, f"+{hi:.0f}", ha="left", va="center", fontsize=13, color="#ac7500", fontweight="bold")
ax.set_yticks(y, [f"{k} · {g}" + (f"\n({role[k]})" if k in role else "") for k, g in zip(end_rise.index, end_rise.GRADE)])
ax.invert_yaxis(); ax.set_xlim(-58, 58); style(ax, "가열 TC-Z3~Z6 중 가장 크게 바뀐 구역[℃] · 점선 = -10℃ / +20℃"); ax.tick_params(labelsize=12)
ax.set_title("(a) 14 LOT: 끝 5분 하강(파랑) · 처음 80~100분 상승(노랑)\n" + f"점선을 넘는 LOT은 {', '.join(map(str, big))}뿐 — 모두 계획수리 바로 앞·뒤", loc="left", fontsize=15, pad=10)
for idx, e in enumerate(expected.itertuples()):
    a2 = fig.add_subplot(gs[idx // 2, 1 + idx % 2]); p = profile[e.LOT]; jr = stop_judge.iloc[idx]
    w0, w1 = (max(0, e.START_MIN - 40), lot_meta.loc[e.LOT, "MINUTES"]) if e.종류 == "예정된 정지" else (0, e.END_MIN + 30)
    seg = p.loc[w0:w1]; tx = lot_meta.loc[e.LOT, "START"] + pd.to_timedelta(seg.index, unit="min")
    h, o = med(seg, HEAT).mean(axis=1), med(seg, OVER).mean(axis=1)
    a2.plot(tx, h - h.iloc[0], color="#17344d", lw=2.2); a2.plot(tx, o - o.iloc[0], color="#8c6d31", lw=1.6, ls="--")
    b2 = a2.twinx(); b2.plot(tx, med(seg, HEAT_OP).median(axis=1), color="#d55e00", lw=1.3); b2.set_ylim(-5, 105)
    a2.axvspan(e.시작, e.끝, facecolor="none", edgecolor=EXPECTED_COLOR, hatch="///", lw=0)
    a2.set_title(f"({'bcde'[idx]}) {e.LOT} · {e.종류} ({e.시작:%m/%d})\n가열 {jr['가열 Z3~Z6 온도 변화[℃]']}℃ · 과시효 {jr['과시효 Z7·Z8 온도 변화[℃]']}℃ · 출력 {jr['가열 출력 중앙값[%]']}%", loc="left", fontsize=13.5, pad=8)
    clock_axis(a2); a2.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=2, maxticks=4)); a2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    a2.set_xlabel("실제 시각[시:분]", fontsize=11.5); a2.set_ylabel("왼쪽 끝 대비 온도 변화[℃]", fontsize=11.5)
    b2.set_ylabel("가열 출력[%]", fontsize=11.5, color="#d55e00"); a2.tick_params(labelsize=11); b2.tick_params(labelsize=11, colors="#d55e00"); b2.spines[["top"]].set_visible(False)
header(fig, "정비 일정 전후 정지·재가동 — 설명됨의 근거",
       "(a) 14 LOT 비교  |  (b)~(e) 정지·재가동 구간: 두 온도선은 그림 왼쪽 끝을 0으로 맞춘 변화, 주황 = 가열 OP-Z3~Z6 중앙값, 회색 빗금 = 예정된 변화 구간",
       [Line2D([0], [0], color="#17344d", lw=2.2, label="가열 TC-Z3~Z6 평균"), Line2D([0], [0], color="#8c6d31", lw=1.6, ls="--", label="과시효 TC-Z7·Z8 평균"),
        Line2D([0], [0], color="#d55e00", lw=1.3, label="가열 출력"), Patch(facecolor="white", edgecolor=EXPECTED_COLOR, hatch="///", label="예정된 정지·재가동")], ncol=4)
compact_header(fig, .935, .905, title_size=26, sub_size=14, legend_size=14)
axis_tag(fig, "(a) " + compare_tag("14 LOT") + "  |  (b)~(e) " + TAG_REAL)
finish(fig, "02_근거범위/01_예정된변화_근거.png")
table(stop_judge)
others = end_rise.drop(index=big)
restart_op = np.median([float(v.split(" → ")[1]) for v, kind in zip(stop_judge["가열 출력 중앙값[%]"], stop_judge["종류"]) if kind != "예정된 정지"])
note("\n".join([
    f"- **(a)** 끝 5분에 10℃ 넘게 내려가거나 처음 80~100분에 20℃ 넘게 오른 LOT은 {', '.join(map(str, big))}뿐이고, 모두 계획수리 바로 앞(마지막 관측)이나 뒤(첫 관측)다. "
    f"나머지 {len(others)} LOT은 끝 하강이 가장 커도 {others['끝 5분 하강[℃]'].min():+.1f}℃, 처음 상승이 가장 커도 {others['처음 80~100분 상승[℃]'].max():+.1f}℃다.",
    "- **(b)(d) 정지:** 가열·과시효 온도가 함께 내려가고, 가열 출력도 끝부분에서 0 쪽으로 내려간다(표의 가열 출력 중앙값).",
    f"- **(c)(e) 재가동:** 가열 출력은 평소 수준(약 {restart_op:.0f}%)을 유지한 채 가열·과시효 온도가 함께 올라간다. 온도만 혼자 오르거나 한 구역만 다르게 움직이지 않는다.",
    "- 표의 '판정'처럼 네 구간 모두 다르게 움직인 구역이 없다. 그래서 이 구간과 정지 앞 15분에 잡힌 탐지는 **예정된 변화**로 빼고, 이상 후보로 보지 않는다."]))


In [ ]:
# 탐지 구간마다 ①~③ 가운데 어떤 이유로 설명되는지 붙인다(앞 단계에서 설명되면 뒤 단계는 보지 않는다).
z_cols = [c for c in detect.columns if c.endswith("_Z30")]
level_pairs = set(zip(overage.LOT, overage.SENSOR))
def top_sensor(k, a, b):
    d = detect.loc[detect.LOT.eq(k) & detect.MINUTE.between(a, b - 1), z_cols].abs().max()
    return d.idxmax().replace("_Z30", "") if d.notna().any() else ""
def is_expected(r):
    return any(e.START_MIN - 15 < r.END_MIN and e.END_MIN > r.START_MIN for e in expected.loc[expected.LOT.eq(r.LOT)].itertuples())
def is_level(r):
    return r.METHOD == "조건별 범위 기준선" and all((r.LOT, s) in level_pairs or s == "CLN-IN" for s in r.SENSORS.split(" · "))
def is_z3(r):
    """이동 Z: 가장 크게 움직인 센서가 Z3 / 범위 기준선: TC-Z3·OP-Z3와 ②로 설명된 센서만 잡힘"""
    if r.METHOD == "CP 규칙":
        return False
    sens = set(r.SENSORS.split(" · ")); explained = {"TC-Z3", "OP-Z3", "CLN-IN"} | {s for kk, s in level_pairs if kk == r.LOT}
    return r.TOP_SENSOR in ["TC-Z3", "OP-Z3"] or (r.METHOD == "조건별 범위 기준선" and bool(sens & {"TC-Z3", "OP-Z3"}) and sens <= explained)
scope = intervals.copy()
scope["TOP_SENSOR"] = [top_sensor(k, a, b) for k, a, b in zip(scope.LOT, scope.START_MIN, scope.END_MIN)]
scope["이유"] = ["① 예정된 변화" if is_expected(r) else "② LOT 수준 차이" if is_level(r) else "③ Z3 반복" if is_z3(r) else "남음" for r in scope.itertuples()]
REASONS = ["① 예정된 변화", "② LOT 수준 차이", "③ Z3 반복", "남음"]
count = pd.crosstab(scope.METHOD, scope["이유"]).reindex(columns=REASONS, fill_value=0)
table(count.assign(합계=count.sum(axis=1)))
by_lot = pd.crosstab(scope.LOT, scope["이유"]).reindex(columns=REASONS, fill_value=0)
table(by_lot.assign(합계=by_lot.sum(axis=1)))
def nearby_work(k, a, b, hours=6):
    t0, t1 = lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)), lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(b))
    w = cal_work.loc[cal_work.STRT_DT.lt(t1 + pd.Timedelta(hours=hours)) & cal_work.END_DT.gt(t0 - pd.Timedelta(hours=hours))]
    return " / ".join(f"{x.MNT_TYPE} {x.STRT_DT:%m/%d %H:%M}–{x.END_DT:%H:%M} {remark_text(x)}" for x in w.itertuples()) or f"앞뒤 {hours}시간 작업 없음"
left = count["남음"]
note(f"- {len(scope)}개 가운데 ① {count['① 예정된 변화'].sum()}개 · ② {count['② LOT 수준 차이'].sum()}개 · ③ {count['③ Z3 반복'].sum()}개는 해당 관측 형태로 분류한 구간이다.\n"
     f"- 이 분류에 속하지 않은 {left.sum()}개는 CP 규칙 {left.get('CP 규칙', 0)}개와 이동 Z {left.get('이동 Z-score', 0)}개다. CP 규칙은 LOT마다 가장 큰 펄스 하나만 남기므로, CP는 아래에서 14 LOT 전체를 다시 센다.")


In [ ]:
# 14 LOT 전체에서 CP-Z4가 LOT 중앙값보다 0.05 넘게 벗어난 경우를 모두 센다(1초 원본).
WATCH = "#b8863b"   # 한 번 벗어난 LOT 색
cp_rows = []
for k in lots.LOT_NO.astype(int):
    s = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = s - s.median()
    off = dev.abs().gt(.02); grp = off.ne(off.shift()).cumsum()
    for _, part in dev[off].groupby(grp[off]):
        peak = part.abs().idxmax()
        if abs(part.loc[peak]) < .05:
            continue
        a, b, m = [(t - lot_meta.loc[k, "START"]).total_seconds() / 60 for t in (part.index[0], part.index[-1], peak)]
        cp_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "PEAK": peak, "DEV": part.loc[peak], "A": a, "B": b, "M": m,
                        "초": int((part.index[-1] - part.index[0]).total_seconds()) + 1, "EXP": in_expected(k, m)})
cp_events = pd.DataFrame(cp_rows)
cp_real = cp_events.loc[~cp_events.EXP]
cp_count = cp_real.groupby("LOT").size()
CP_FOCUS = cp_count[cp_count.ge(2)].sort_values(ascending=False).index.tolist(); CP_WATCH = cp_count[cp_count.eq(1)].index.tolist()
ids = lots.sort_values("START").LOT_NO.astype(int).tolist()
# 14 LOT의 CP-Z4 1초 원본을 같은 시간축에 한 줄씩 놓는다.
# 세로 = CP-Z4 - LOT 중앙값 · 회색 띠 = ±0.05 · 점 = 0.05 넘게 벗어난 한 번의 꼭짓점(빨강 = 여러 번 벗어난 LOT, 주황 = 한 번, 빈 점 = 예정된 정지 중)
# 세로는 ±0.2까지만 그린다(작은 이탈도 보이게). 넘는 값은 점을 끝에 두고 숫자에 ↑·↓를 붙인다.
fig, axs = plt.subplots(len(ids), 1, figsize=(28, 1.2 * len(ids) + 3.4), sharex=True, sharey=True)
H = 1.2 * len(ids) + 3.4; LIM = .2
fig.subplots_adjust(left=.13, right=.91, top=1 - 3.0 / H, bottom=.6 / H, hspace=.3)
for ax, k in zip(axs, ids):
    g = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = g - g.median()
    hrs = (g.index - lot_meta.loc[k, "START"]).total_seconds() / 3600
    ax.fill_between([0, hrs.max()], -.05, .05, color="#e3e9ee", lw=0)   # 띠는 관측한 시간까지만
    ax.plot(hrs, dev, color="#526d80", lw=.6)
    ev = cp_events.loc[cp_events.LOT.eq(k)]
    for r in ev.itertuples():
        c = EXPECTED_COLOR if r.EXP else SELECT_COLOR if k in CP_FOCUS else WATCH
        ax.plot(r.M / 60, np.clip(r.DEV, -LIM * .93, LIM * .93), "o", ms=9, mfc="white" if r.EXP else c, mec=c, mew=2, zorder=4)
    if len(ev):   # 가장 크게 벗어난 한 번만 값을 적는다
        r = ev.loc[ev.DEV.abs().idxmax()]
        arrow = "↑" if r.DEV > LIM else "↓" if r.DEV < -LIM else ""
        ax.annotate(("예정된 정지 중 " if r.EXP else "") + f"{r.DEV:+.3f}{arrow}", (r.M / 60, np.clip(r.DEV, -LIM * .93, LIM * .93)), xytext=(8, 0), textcoords="offset points", va="center", fontsize=12,
                    color=EXPECTED_COLOR if r.EXP else SELECT_COLOR if k in CP_FOCUS else WATCH, fontweight="bold" if k in CP_FOCUS else "normal")
    n = int(cp_count.get(k, 0)); col = SELECT_COLOR if k in CP_FOCUS else WATCH if n else "#9aa6ae"
    ax.text(-.01, .5, f"{k} · {lot_meta.loc[k, 'GRADE']} · {lot_meta.loc[k, 'START']:%m/%d}", transform=ax.transAxes, ha="right", va="center", fontsize=13,
            color=col if k in CP_FOCUS else "#17344d", fontweight="bold" if k in CP_FOCUS else "normal")
    ax.text(1.008, .5, f"{n}번" if n else "없음", transform=ax.transAxes, ha="left", va="center", fontsize=15, color=col, fontweight="bold" if k in CP_FOCUS else "normal")
    style(ax, ""); ax.set_yticks([-.05, .05], ["", ""]); ax.tick_params(labelsize=10.5); ax.set_ylim(-LIM, LIM)
    ax.spines["bottom"].set_visible(ax is axs[-1])
    if ax is not axs[-1]:
        ax.tick_params(axis="x", length=0)
axs[-1].set_xlim(0, lot_meta.MINUTES.max() / 60 + .2); axs[-1].set_xlabel("관측 시작 후 시간[시간] · 모든 줄이 같은 눈금이라 관측 길이도 함께 비교된다", fontsize=13)
axs[-1].tick_params(axis="x", labelsize=12.5)
fig.text(.91 + .008, 1 - 2.75 / H, "벗어난\n횟수", ha="left", va="bottom", fontsize=13, color="#526d80")
header(fig, "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳",
       "한 줄 = 한 LOT의 관측 전체(1초 원본) · 세로 = CP-Z4 - LOT 중앙값(모든 LOT 약 0.45) · 회색 띠 = ±0.05 · 점 = 0.05 넘게 벗어난 한 번의 꼭짓점 · 세로는 ±0.2까지(넘으면 숫자에 ↑↓) · 오른쪽 = 횟수(예정된 정지 중 제외)",
       [Line2D([0], [0], marker="o", ls="None", color=SELECT_COLOR, ms=11, label="여러 번 벗어난 LOT"), Line2D([0], [0], marker="o", ls="None", color=WATCH, ms=11, label="한 번 벗어난 LOT"),
        Line2D([0], [0], marker="o", ls="None", mfc="white", mec=EXPECTED_COLOR, mew=2, ms=11, label="예정된 정지 중 (횟수에서 뺌)"),
        Patch(facecolor="#e3e9ee", label="LOT 중앙값 ±0.05")], ncol=4)
compact_header(fig, 1 - 1.05 / H, 1 - 1.55 / H, title_size=25, sub_size=14, legend_size=13)
axis_tag(fig, "TIME AXIS · 관측 시작 후 시간[시간]")
finish(fig, "02_근거범위/03_CP이탈_14LOT.png")
table(cp_real.assign(시각=cp_real.PEAK.dt.strftime("%m/%d %H:%M:%S"), 크기=cp_real.DEV.round(3))[["LOT", "GRADE", "시각", "크기", "초"]].sort_values(["LOT", "시각"]))

In [ ]:
finish, note, table = _finish, _note, _table      # 여기부터 다시 표시한다

<a id="funnel"></a>

## 결론 먼저 — 탐지된 구간이 어디로 갔나

01의 탐지 규칙(범위 기준선 · 이동 Z · CP 규칙)으로 잡힌 구간이 어떤 근거로 빠지고 무엇이 03으로 넘어갔는지 한 장으로 본다. 아래 표는 항목별 근거다.

In [ ]:
# 결론 그림: 01의 탐지 규칙으로 잡힌 구간이 어떤 근거로 빠지고 무엇이 남았나.
#   왼쪽 막대 = 탐지 구간 수(2절 분류 · 앞 단계에서 설명되면 뒤 단계는 보지 않음) · 오른쪽 = 왜 빼나 · 처리 → 어디서
import textwrap
lv0 = level_issues.assign(배수=level_issues["차이"].abs() / level_issues["LOT 안 흔들림"])
lv_keep = lv0.판정.eq("혼자 떨어짐") | lv0.배수.ge(4)
cp_n = cp_real.groupby("LOT").size()
cp_multi = cp_n[cp_n.ge(2)].index.tolist(); cp_once = [k for k in cp_n[cp_n.eq(1)].index if k != 240117]
left = scope.loc[scope.이유.eq("남음")]; other = left.loc[left.METHOD.ne("CP 규칙")]
n = scope.이유.value_counts()
OUT_C = {"제외": "#526d80", "03": SELECT_COLOR, "지켜볼": "#b8863b"}
rows = [
    ("① 정기수리 직전 정지·직후 승온", n["① 예정된 변화"], "계획된 운전 조작이다. 정지 때는 온도와 출력이 하강하고, 승온 때는 온도가 오르며 출력은 약 60%를 유지한다",
     [("제외", "제외 → 1절")]),
    ("③ Z3 반복", n["③ Z3 반복"], "14 LOT 모두 Z3만 출렁인다. 온도가 출력을 늦게 따라오는 Z3 구역의 제어 특성이다",
     [("제외", "제외 → 3절"), ("03", "예외 240117 11/13 12:55 → 03의 4절에서 확대")]),
    ("② LOT 전체 수준 차이", n["② LOT 수준 차이"], f"구간이 아니라 LOT 전체가 다른 것이다. LOT·신호 {len(lv0)}건을 평소 흔들림 대비 배수로 다시 본다",
     [("03", f"{lv_keep.sum()}건 비교: 과시효 Z7·Z8(240091·240121·240114), 출력 Z4(240061·240038) → 03의 6절"),
      ("제외", f"{(~lv_keep).sum()}건 제외: 평소 대비 1.3배 이하 → 8절")]),
    ("남은 CP 규칙", left.METHOD.eq("CP 규칙").sum(), "LOT마다 가장 큰 펄스 하나만 남기므로 14 LOT 전체를 1초 원본으로 다시 센다",
     [("03", f"확대: 여러 번 {'·'.join(map(str, cp_multi))} · 정비에 가까움 240117 → 03의 2~4절"),
      ("지켜볼", f"지켜볼: 한 번만 작게 {'·'.join(map(str, cp_once))} → 5절")]),
    ("남은 이동 Z", len(other), " · ".join(f"{int(r.LOT)} {r.TOP_SENSOR}" for r in other.itertuples()) + "의 1분 변화. 다른 LOT의 평소 오르내림에서도 나오는 크기다",
     [("제외", "제외 → 4절")]),
]
fig = plt.figure(figsize=(30, 12.5))
ax = fig.add_axes([.2, .08, .17, .7]); tx = fig.add_axes([.41, .08, .57, .7]); tx.axis("off")
y = np.arange(len(rows))
ax.barh(y, [r[1] for r in rows], color="#9aa6ae", height=.6)
for yy, r in zip(y, rows):
    ax.text(r[1] + 1.5, yy, f"{r[1]}구간", va="center", fontsize=14, color="#17344d", fontweight="bold")
ax.set_yticks(y, [r[0] for r in rows], fontsize=14); ax.invert_yaxis(); ax.set_xlim(0, max(r[1] for r in rows) * 1.35)
style(ax, "탐지 구간 수"); ax.tick_params(axis="x", labelsize=12)
ax.set_title(f"(1) 탐지 구간 {len(scope)}개 — 이유별 수", loc="left", fontsize=15, pad=14)
tx.set_ylim(len(rows) - .5, -.5); tx.set_xlim(0, 1)
tx.text(0, -.62, "왜 이상으로 보기 어려운가", fontsize=15, color="#17344d", fontweight="bold", va="bottom")
tx.text(.47, -.62, "처리 → 어디서", fontsize=15, color="#17344d", fontweight="bold", va="bottom")
for yy, r in zip(y, rows):
    tx.text(0, yy, textwrap.fill(r[2], 36), va="center", fontsize=13.5, color="#17344d", linespacing=1.4)
    outs = r[3]
    for j, (kind, text) in enumerate(outs):
        pos, va = (yy, "center") if len(outs) == 1 else (yy - .04, "bottom") if j == 0 else (yy + .04, "top")
        tx.text(.47, pos, textwrap.fill(text, 52), va=va, fontsize=13.5, color=OUT_C[kind], linespacing=1.4,
                fontweight="bold" if kind == "03" else "normal")
    if yy < len(rows) - 1:
        tx.axhline(yy + .5, color="#d5dce1", lw=1)
header(fig, "탐지된 구간이 어디로 갔나 — 근거가 있어 빼는 것과 03으로 넘기는 것",
       f"01의 탐지 규칙(범위 기준선 · 이동 Z · CP 규칙)으로 잡힌 {len(scope)}구간 · 앞 단계에서 설명되면 뒤 단계는 보지 않는다 · 빨강 = 03에서 확대·비교",
       [Patch(facecolor=OUT_C["제외"], label="제외(근거 있음)"), Patch(facecolor=OUT_C["03"], label="03에서 확대·비교"), Patch(facecolor=OUT_C["지켜볼"], label="지켜볼")], ncol=3)
compact_header(fig, .935, .905, title_size=24, sub_size=13, legend_size=13)
axis_tag(fig, compare_tag("탐지 구간 분류"))
finish(fig, "00_결론/00_탐지구간_어디로.png")

<a id="not-needed"></a>

### 안 봐도 되는 것 — 근거 표

아래 구간·LOT은 지우지 않고 근거와 함께 남기지만, **확대해서 파고들 대상에서는 뺀다.** "평소 대비"는 같은 LOT 직전 30분과 비교한 급변 크기를 그 신호의 안정 운전 99% 값으로 나눈 배수다(01의 운전 상태별 급변 그림).

| 구간 · LOT | 보인 것(크기) | 왜 이상으로 보기 어려운가 | 데이터 근거 | 처리 |
|---|---|---|---|---|
| **정기수리 직전 정지**<br>240058 06/09 08:08–09:02<br>240114 11/04 20:05–20:59 | 가열 Z3~Z6 −43℃<br>출력 57~60% → 4% | 계획된 운전 정지다. 온도가 먼저 내려가고 43~44분 뒤 출력이 함께 0 쪽으로 가는 정상 정지 순서다. 다른 LOT의 같은 시점은 정상 운전이라 비교할 기준이 없다 | 14 LOT 중 끝부분이 10℃ 넘게 떨어진 것은 이 둘뿐이고, 둘 다 계획수리 시작 3~15시간 전 마지막 관측이다. 모든 구역 온도·출력이 같은 방향이다. 떼지 않으면 14 LOT 급변 상위권을 정지·승온이 거의 모두 차지한다(01의 운전 상태별 급변 그림) | '정지'로 따로 표시(하강 5분 전부터), 이상 판단에서 제외(01의 운전 상태별 급변·1절) |
| **정기수리 직후 승온**<br>240061 06/15 01:04–02:50<br>240117 11/13 07:08–08:47 | 가열 Z3~Z6 +42~43℃<br>출력 55~61% 유지 | 계획된 재가동이다. 출력은 평소 수준인데 온도만 오르는 승온 운전이다 | 처음 80~100분에 20℃ 넘게 오른 것은 이 둘뿐이고 둘 다 계획수리 뒤 첫 관측이다. 과시효 Z7·Z8은 가열 구역보다 늦게 따라 올라 승온이 끝난 뒤에도 5~8분 더 오르므로 승온 끝 뒤 10분까지 승온으로 본다 | '승온'으로 따로 표시, 제외(01의 운전 상태별 급변·1절) |
| **Z3 출렁임**<br>14 LOT 전부 · 탐지 75구간 341분 | Z3만 ±10~16℃<br>출력 0~100% | Z3 구역의 제어 특성이다. 온도가 출력을 늦게 따라와 출력이 끝에 붙었다 돌아온다 | Z3 평소 흔들림(1분 값 IQR 1.5℃)이 Z4~Z6(0.3~0.4℃)의 4~5배, 출력은 17%p로 다른 구역(1~2%p)의 7배 이상이다. 같은 강종 대조 편차 상위 11개가 모두 Z3지만 Z3 자신의 99% 대비 최대 3.5배다(01 운전 상태별 급변 그림). 작은 배수 때문이 아니라 LOT마다 1~14번 반복되는 파형과 출력 반응을 근거로 제외한다 | 제외(3절). 예외: 240117 12:55–13:04는 CP로 이어져 [03의 4절](05_all_data_timeseries_jinny_03.ipynb#ev-240117)에서 본다 |
| **같은 경과 분 대조에서만 생긴 편차**<br>예: 240091 출력 Z4 −20%p<br>240069 Z7·Z8 +3℃ | 원본에서는 변화 없음 | LOT마다 관측 길이가 달라 대조 LOT이 중간에 빠지거나 바뀌면서 기준값이 뛴 착시다 | 같은 시각 원본 10분 값은 240091 출력 Z4 40~47%, 240069 Z7·Z8 329.7~330.2℃로 평평하다 | 시간 구간 판단에 같은 경과 분 대조를 쓰지 않는다 |
| **세정 입구 두 수준**<br>**240108 10/18 07:10** | 세정 입구 64.5~67.8℃ / 69.2~70.0℃ 두 무리<br>240108은 1분 가파른 변화 | 강종·정비와 무관한 LOT별 운전 단계, 평소 오르내림 중 한 순간이다 | 세정 입구를 강종으로 설명하는 정도는 우연 수준(0.43)이고 정비가 없던 LOT 사이에서도 바뀐다. 240108의 변화(이동 Z 4.0)는 다른 LOT에서도 3.5~3.8로 나온다 | 제외(4절) |
| **작은 CP 한 번**<br>240024 · 240038 · 240091 | CP 0.052~0.066<br>0.1~0.9분 | 함께 움직인 공정이 없고 기준 경계다 | 극값 앞뒤 3분의 Z4~Z6는 평소 범위다. 기준을 0.06으로 올리면 240038·240091은 빠진다 | 지켜볼(5절) |
| **Z9·냉각 차이**<br>240133 · 240044 | Z9 −4.7℃, +8.1℃ | 반복 파형의 일부만 관측해 생긴 차이다 | 안정 관측 108·75분이 두 주기(약 392분)보다 짧다 | 제외(6절) |
| **LOT 사이 정비 지표**<br>**LOT이 1~2개뿐인 강종의 작은 차이** | — | 비교 대상이 부족하다 | 7절 | 판단 보류 |

<a id="stop-restart"></a>

## 1. 정기수리 직전 정지·직후 승온 — 설명됨

(a)는 14 LOT 전체에서 끝부분 하강·처음 상승이 큰 LOT을 찾고, (b)~(e)는 그 구간에서 모든 구역의 온도와 출력이 같은 방향으로 움직였는지 본다. 같은 크기의 변화가 정비 일정 바로 앞뒤의 4 LOT에만 있어 "설명됨"으로 둔다. 지우지 않고 근거와 함께 남긴다.

![정지와 재가동의 온도 및 출력 근거](<../../figures/05_all_data_timeseries_jinny_02/02_근거범위/01_예정된변화_근거.png>)


In [ ]:
# 정비 일정 전후 정지·재가동 — '설명됨'의 근거
# (a) 14 LOT의 끝 5분 하강·처음 80~100분 상승 비교, (b)~(e) 정지·재가동 구간에서 구역별 온도·출력이 같은 방향으로 움직였는지
HEAT_OP = [f"OP-Z{z}" for z in range(3, 7)]
OVER = ["TC-Z7", "TC-Z8"]
role = dict(zip(expected.LOT, expected["수리 관계"]))
def med(p, cols):
    return p.loc[:, [(s, "median") for s in cols]]
judge_rows = []
for e in expected.itertuples():
    p = profile[e.LOT]
    if e.종류 == "예정된 정지":
        before, after = p.loc[e.START_MIN - 10:e.START_MIN - 1], p.loc[e.END_MIN - 5:]
    else:
        before, after = p.loc[:4], p.loc[e.END_MIN - 5:e.END_MIN - 1]
    heat = (med(after, HEAT).median() - med(before, HEAT).median()).to_numpy()
    over = (med(after, OVER).median() - med(before, OVER).median()).to_numpy()
    op0, op1 = med(before, HEAT_OP).median().median(), med(after, HEAT_OP).median().median()
    same = (heat < 0).all() and (over < 0).all() if e.종류 == "예정된 정지" else (heat > 0).all() and (over > 0).all()
    judge_rows.append({"LOT": e.LOT, "GRADE": e.GRADE, "종류": e.종류, "실제 시각": f"{e.시작:%m/%d %H:%M}–{e.끝:%H:%M}", "수리 관계": role[e.LOT],
                       "가열 Z3~Z6 온도 변화[℃]": f"{heat.min():+.1f} ~ {heat.max():+.1f}", "과시효 Z7·Z8 온도 변화[℃]": f"{over.min():+.1f} ~ {over.max():+.1f}",
                       "가열 출력 중앙값[%]": f"{op0:.0f} → {op1:.0f}", "판정": "모든 구역이 같은 방향 → 예정된 변화" if same else "다르게 움직인 구역 있음 → 확인 필요"})
stop_judge = pd.DataFrame(judge_rows)
level_rows = []
for k in lots.LOT_NO.astype(int):
    p = med(profile[k], HEAT); m = p.index.max()
    drop = (p.loc[m - 4:].median() - p.loc[m * .45:m * .6].median()).min()
    rise = (p.loc[80:100].median() - p.loc[:4].median()).max() if m >= 100 else np.nan
    level_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "끝 5분 하강[℃]": drop, "처음 80~100분 상승[℃]": rise})
end_rise = pd.DataFrame(level_rows).set_index("LOT")
big = end_rise.index[end_rise["끝 5분 하강[℃]"].le(-10) | end_rise["처음 80~100분 상승[℃]"].ge(20)].tolist()

fig = plt.figure(figsize=(30, 15))
gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1, 1], left=.085, right=.955, top=.83, bottom=.07, hspace=.62, wspace=.36)
ax = fig.add_subplot(gs[:, 0]); y = np.arange(len(end_rise))
ax.barh(y, end_rise["끝 5분 하강[℃]"], color="#0072b2", height=.6)
ax.barh(y, end_rise["처음 80~100분 상승[℃]"].fillna(0), color="#e69f00", height=.6)
for lim in [-10, 20]:
    ax.axvline(lim, color="#526d80", ls=":", lw=1.4)
ax.axvline(0, color="#9fb3c3", lw=1)
for yy, lo, hi in zip(y, end_rise["끝 5분 하강[℃]"], end_rise["처음 80~100분 상승[℃]"]):
    if lo <= -10: ax.text(lo - 1, yy, f"{lo:.0f}", ha="right", va="center", fontsize=13, color="#0072b2", fontweight="bold")
    if hi >= 20: ax.text(hi + 1, yy, f"+{hi:.0f}", ha="left", va="center", fontsize=13, color="#ac7500", fontweight="bold")
ax.set_yticks(y, [f"{k} · {g}" + (f"\n({role[k]})" if k in role else "") for k, g in zip(end_rise.index, end_rise.GRADE)])
ax.invert_yaxis(); ax.set_xlim(-58, 58); style(ax, "가열 TC-Z3~Z6 중 가장 크게 바뀐 구역[℃] · 점선 = -10℃ / +20℃"); ax.tick_params(labelsize=12)
ax.set_title("(a) 14 LOT: 끝 5분 하강(파랑) · 처음 80~100분 상승(노랑)\n" + f"점선을 넘는 LOT은 {', '.join(map(str, big))}뿐 — 모두 계획수리 바로 앞·뒤", loc="left", fontsize=15, pad=10)
for idx, e in enumerate(expected.itertuples()):
    a2 = fig.add_subplot(gs[idx // 2, 1 + idx % 2]); p = profile[e.LOT]; jr = stop_judge.iloc[idx]
    w0, w1 = (max(0, e.START_MIN - 40), lot_meta.loc[e.LOT, "MINUTES"]) if e.종류 == "예정된 정지" else (0, e.END_MIN + 30)
    seg = p.loc[w0:w1]; tx = lot_meta.loc[e.LOT, "START"] + pd.to_timedelta(seg.index, unit="min")
    h, o = med(seg, HEAT).mean(axis=1), med(seg, OVER).mean(axis=1)
    a2.plot(tx, h - h.iloc[0], color="#17344d", lw=2.2); a2.plot(tx, o - o.iloc[0], color="#8c6d31", lw=1.6, ls="--")
    b2 = a2.twinx(); b2.plot(tx, med(seg, HEAT_OP).median(axis=1), color="#d55e00", lw=1.3); b2.set_ylim(-5, 105)
    a2.axvspan(e.시작, e.끝, facecolor="none", edgecolor=EXPECTED_COLOR, hatch="///", lw=0)
    a2.set_title(f"({'bcde'[idx]}) {e.LOT} · {e.종류} ({e.시작:%m/%d})\n가열 {jr['가열 Z3~Z6 온도 변화[℃]']}℃ · 과시효 {jr['과시효 Z7·Z8 온도 변화[℃]']}℃ · 출력 {jr['가열 출력 중앙값[%]']}%", loc="left", fontsize=13.5, pad=8)
    clock_axis(a2); a2.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=2, maxticks=4)); a2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    a2.set_xlabel("실제 시각[시:분]", fontsize=11.5); a2.set_ylabel("왼쪽 끝 대비 온도 변화[℃]", fontsize=11.5)
    b2.set_ylabel("가열 출력[%]", fontsize=11.5, color="#d55e00"); a2.tick_params(labelsize=11); b2.tick_params(labelsize=11, colors="#d55e00"); b2.spines[["top"]].set_visible(False)
header(fig, "정비 일정 전후 정지·재가동 — 설명됨의 근거",
       "(a) 14 LOT 비교  |  (b)~(e) 정지·재가동 구간: 두 온도선은 그림 왼쪽 끝을 0으로 맞춘 변화, 주황 = 가열 OP-Z3~Z6 중앙값, 회색 빗금 = 예정된 변화 구간",
       [Line2D([0], [0], color="#17344d", lw=2.2, label="가열 TC-Z3~Z6 평균"), Line2D([0], [0], color="#8c6d31", lw=1.6, ls="--", label="과시효 TC-Z7·Z8 평균"),
        Line2D([0], [0], color="#d55e00", lw=1.3, label="가열 출력"), Patch(facecolor="white", edgecolor=EXPECTED_COLOR, hatch="///", label="예정된 정지·재가동")], ncol=4)
compact_header(fig, .935, .905, title_size=26, sub_size=14, legend_size=14)
axis_tag(fig, "(a) " + compare_tag("14 LOT") + "  |  (b)~(e) " + TAG_REAL)
finish(fig, "02_근거범위/01_예정된변화_근거.png")
table(stop_judge)
others = end_rise.drop(index=big)
restart_op = np.median([float(v.split(" → ")[1]) for v, kind in zip(stop_judge["가열 출력 중앙값[%]"], stop_judge["종류"]) if kind != "예정된 정지"])
note("\n".join([
    f"- **(a)** 끝 5분에 10℃ 넘게 내려가거나 처음 80~100분에 20℃ 넘게 오른 LOT은 {', '.join(map(str, big))}뿐이고, 모두 계획수리 바로 앞(마지막 관측)이나 뒤(첫 관측)다. "
    f"나머지 {len(others)} LOT은 끝 하강이 가장 커도 {others['끝 5분 하강[℃]'].min():+.1f}℃, 처음 상승이 가장 커도 {others['처음 80~100분 상승[℃]'].max():+.1f}℃다.",
    "- **(b)(d) 정지:** 가열·과시효 온도가 함께 내려가고, 가열 출력도 끝부분에서 0 쪽으로 내려간다(표의 가열 출력 중앙값).",
    f"- **(c)(e) 재가동:** 가열 출력은 평소 수준(약 {restart_op:.0f}%)을 유지한 채 가열·과시효 온도가 함께 올라간다. 온도만 혼자 오르거나 한 구역만 다르게 움직이지 않는다.",
    "- 표의 '판정'처럼 네 구간 모두 다르게 움직인 구역이 없다. 그래서 이 구간과 정지 앞 15분에 잡힌 탐지는 **예정된 변화**로 빼고, 이상 후보로 보지 않는다."]))


<a id="classify"></a>

## 2. 탐지 구간 분류 — 대부분 정비 전환·수준 차이·Z3 반복으로 설명됨

탐지 구간(범위 기준선·이동 Z-score·CP 규칙)마다 정비 전환(①)·LOT 수준 차이(②)·Z3 반복(③) 가운데 어디에 해당하는지 붙인다. 분류는 설명을 붙이는 것이지 목록에서 지우는 것이 아니다. LOT마다 몇 개씩인지는 아래 둘째 표에 있다.

In [ ]:
# 탐지 구간마다 ①~③ 가운데 어떤 이유로 설명되는지 붙인다(앞 단계에서 설명되면 뒤 단계는 보지 않는다).
z_cols = [c for c in detect.columns if c.endswith("_Z30")]
level_pairs = set(zip(overage.LOT, overage.SENSOR))
def top_sensor(k, a, b):
    d = detect.loc[detect.LOT.eq(k) & detect.MINUTE.between(a, b - 1), z_cols].abs().max()
    return d.idxmax().replace("_Z30", "") if d.notna().any() else ""
def is_expected(r):
    return any(e.START_MIN - 15 < r.END_MIN and e.END_MIN > r.START_MIN for e in expected.loc[expected.LOT.eq(r.LOT)].itertuples())
def is_level(r):
    return r.METHOD == "조건별 범위 기준선" and all((r.LOT, s) in level_pairs or s == "CLN-IN" for s in r.SENSORS.split(" · "))
def is_z3(r):
    """이동 Z: 가장 크게 움직인 센서가 Z3 / 범위 기준선: TC-Z3·OP-Z3와 ②로 설명된 센서만 잡힘"""
    if r.METHOD == "CP 규칙":
        return False
    sens = set(r.SENSORS.split(" · ")); explained = {"TC-Z3", "OP-Z3", "CLN-IN"} | {s for kk, s in level_pairs if kk == r.LOT}
    return r.TOP_SENSOR in ["TC-Z3", "OP-Z3"] or (r.METHOD == "조건별 범위 기준선" and bool(sens & {"TC-Z3", "OP-Z3"}) and sens <= explained)
scope = intervals.copy()
scope["TOP_SENSOR"] = [top_sensor(k, a, b) for k, a, b in zip(scope.LOT, scope.START_MIN, scope.END_MIN)]
scope["이유"] = ["① 예정된 변화" if is_expected(r) else "② LOT 수준 차이" if is_level(r) else "③ Z3 반복" if is_z3(r) else "남음" for r in scope.itertuples()]
REASONS = ["① 예정된 변화", "② LOT 수준 차이", "③ Z3 반복", "남음"]
count = pd.crosstab(scope.METHOD, scope["이유"]).reindex(columns=REASONS, fill_value=0)
table(count.assign(합계=count.sum(axis=1)))
by_lot = pd.crosstab(scope.LOT, scope["이유"]).reindex(columns=REASONS, fill_value=0)
table(by_lot.assign(합계=by_lot.sum(axis=1)))
def nearby_work(k, a, b, hours=6):
    t0, t1 = lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)), lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(b))
    w = cal_work.loc[cal_work.STRT_DT.lt(t1 + pd.Timedelta(hours=hours)) & cal_work.END_DT.gt(t0 - pd.Timedelta(hours=hours))]
    return " / ".join(f"{x.MNT_TYPE} {x.STRT_DT:%m/%d %H:%M}–{x.END_DT:%H:%M} {remark_text(x)}" for x in w.itertuples()) or f"앞뒤 {hours}시간 작업 없음"
left = count["남음"]
note(f"- {len(scope)}개 가운데 ① {count['① 예정된 변화'].sum()}개 · ② {count['② LOT 수준 차이'].sum()}개 · ③ {count['③ Z3 반복'].sum()}개는 해당 관측 형태로 분류한 구간이다.\n"
     f"- 이 분류에 속하지 않은 {left.sum()}개는 CP 규칙 {left.get('CP 규칙', 0)}개와 이동 Z {left.get('이동 Z-score', 0)}개다. CP 규칙은 LOT마다 가장 큰 펄스 하나만 남기므로, CP는 아래에서 14 LOT 전체를 다시 센다.")


<a id="z3"></a>

## 3. Z3 출렁임 — 14 LOT 공통

가열 초입 Z3는 온도가 출력을 늦게 따라와 출력이 0%·100%에 붙었다가 돌아오는 모양이 **14 LOT 모두에서** 반복된다. 특정 LOT의 이상이 아니라 Z3 구역의 특성으로 본다. 예외로 240117의 11/13 12:55 Z3 이탈은 바로 뒤 CP 하강과 이어져 [03의 4절](05_all_data_timeseries_jinny_03.ipynb#ev-240117)에서 확대한다.

<a id="level-variation"></a>

#### Z3 수준과 흐름 — 안정 참고 구간

In [ ]:
# 강종 그룹 색은 공통 범례를 따른다
GRADE_COLOR=report_viz.GRADE_COLORS.copy()
state_stats=[]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for state in states[k].unique():
        for s in SIGNALS:
            y=profile[k][(s,"median")].where(states[k].eq(state)&quality[k][s].ge(.9)).dropna()
            state_stats.append({"LOT":k,"GRADE":r.GRADE,"STATE":state,"SENSOR":s,"N_WINDOWS":len(y),"MEDIAN":y.median(),
                                "IQR":y.quantile(.75)-y.quantile(.25),
                                "FIRST30":y.head(30).median() if len(y)>=60 else np.nan,
                                "LAST30":y.tail(30).median() if len(y)>=60 else np.nan,
                                "FIRST_LAST30_DELTA":y.tail(30).median()-y.head(30).median() if len(y)>=60 else np.nan,
                                "SLOPE_PER_MIN":np.polyfit(y.index.to_numpy(),y.to_numpy(),1)[0] if len(y)>=20 else np.nan})
state_stats=pd.DataFrame(state_stats)

# 같은 강종을 붙여 놓고, 강종 사이에는 빈 줄을 둔다.
order=lots.assign(G=lots.GRADE.map(list(GRADE_COLOR).index)).sort_values(["G","LOT_NO"])
ypos,y,prev={},0,None
for r in order.itertuples():
    y+=.7 if prev not in [None,r.GRADE] else 0
    ypos[int(r.LOT_NO)]=y;y+=1;prev=r.GRADE

def stable_values(k,s):
    return profile[k][(s,"median")].where(states[k].eq("안정 참고")&quality[k][s].ge(.9)).dropna()

In [ ]:
# LOT별 Z3 수준: 점 = 안정 참고 구간 중앙값, 막대 = 가운데 50% (같은 강종끼리 묶음)
by_grade = order.LOT_NO.astype(int).tolist()
pos, x, prev = {}, 0, None
for k in by_grade:
    g = lot_meta.loc[k, "GRADE"]; x += .8 if prev not in [None, g] else 0; pos[k] = x; x += 1; prev = g
fig, axs = plt.subplots(1, 2, figsize=(30, 11)); fig.subplots_adjust(left=.05, right=.98, top=.75, bottom=.2, wspace=.15)
level_med = {}
for ax, (s, u) in zip(axs, [("TC-Z3", "℃"), ("OP-Z3", "%")]):
    for k in by_grade:
        v = stable_values(k, s); c = GRADE_COLOR[lot_meta.loc[k, "GRADE"]]; level_med[k, s] = v.median()
        ax.plot([pos[k], pos[k]], [v.quantile(.25), v.quantile(.75)], color=c, lw=14, alpha=.35, solid_capstyle="butt")
        ax.plot(pos[k], v.median(), "o", color=c, ms=12, zorder=3)
        ax.text(pos[k], v.quantile(.75), f"{v.median():.1f}", ha="center", va="bottom", fontsize=12, color=c)
    ax.set_xticks([pos[k] for k in by_grade], [str(k) for k in by_grade], rotation=35, ha="right")
    for t, k in zip(ax.get_xticklabels(), by_grade):
        t.set_color(GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
    for g in dict.fromkeys(lot_meta.loc[by_grade, "GRADE"]):
        xs_g = [pos[k] for k in by_grade if lot_meta.loc[k, "GRADE"] == g]
        ax.text(np.mean(xs_g), -.24, g, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=13, fontweight="bold", color=GRADE_COLOR[g])
    style(ax, ""); ax.set_ylabel(f"{'온도' if s == 'TC-Z3' else '제어출력'} {s} [{u}]", fontsize=13); ax.tick_params(labelsize=12)
    vals = pd.Series({k: level_med[k, s] for k in by_grade})
    if s == "TC-Z3":
        ax.set_ylim(857, 863)
        ax.set_title(f"(a) 온도: 14 LOT 중앙값이 {vals.min():.1f}~{vals.max():.1f}℃ — 차이 {vals.max() - vals.min():.1f}℃로 거의 같다", loc="left", fontsize=17, pad=12)
    else:
        sphc = vals[[k for k in by_grade if lot_meta.loc[k, "GRADE"] == "SPHC"]]; rest = vals.drop(sphc.index)
        ax.set_ylim(50, 100)
        ax.set_title(f"(b) 제어출력: SPHC는 {sphc.min():.0f}~{sphc.max():.0f}%, 나머지 강종은 {rest.min():.0f}~{rest.max():.0f}% — 강종마다 다르다", loc="left", fontsize=17, pad=12)
header(fig, "안정 참고 구간 · 가열·균열 Z3 수준", "점 = LOT 중앙값 · 굵은 막대 = 가운데 50% · 같은 강종끼리 묶음(아래 굵은 글씨 = 강종)", [], ncol=1)
compact_header(fig, .89, .86, title_size=26, sub_size=14)
axis_tag(fig, compare_tag("14 LOT 가열·균열 Z3"))
finish(fig, "01_수준변동/01_안정참고_Z3수준.png")
tc_v = pd.Series({k: level_med[k, "TC-Z3"] for k in by_grade}); op_v = pd.Series({k: level_med[k, "OP-Z3"] for k in by_grade})
note("\n".join([
    f"- (a) 온도 TC-Z3의 LOT 중앙값은 {tc_v.min():.1f}~{tc_v.max():.1f}℃로, 강종이 달라도 {tc_v.max() - tc_v.min():.1f}℃ 안에 있다. 온도는 강종과 관계없이 거의 같은 값으로 운전된다.",
    f"- (b) 같은 구역의 제어출력 OP-Z3는 {op_v.min():.0f}~{op_v.max():.0f}%로 넓게 퍼진다. 온도를 같게 맞추기 위해 강종마다 다른 출력을 쓴다."]))
table(state_stats)


In [ ]:
# LOT마다 한 칸: 왼쪽 축 = 온도 TC-Z3, 오른쪽 축 = 제어출력 OP-Z3. 둘 다 1분 값으로 그려 몇 분짜리 오르내림까지 같은 시점에서 본다.
date_order = lots.LOT_NO.astype(int).tolist()
OP_COLOR = TC_COLOR = report_viz.sensor_color("TC-Z3")   # TC 실선 · OP 점선
series = {k: (stable_values(k, "OP-Z3"), stable_values(k, "TC-Z3")) for k in date_order}
# 두 축 모두 LOT 중앙값을 가운데에 두고, 높이 1℃ = 출력 10%p로 맞춘다(선의 위아래 위치가 아니라 움직인 방향만 비교하도록).
# 10은 14 LOT에서 1분 값의 출력이 보통 움직이는 폭이 온도 폭의 약 9배라서다(1분 값의 10~90% 폭, LOT별 출력/온도 비 중앙값 9.3).
OP_PER_C = 10
tc_half = max(4, np.ceil(max((t - stable_values(k, "TC-Z3").median()).abs().max() for k, (o, t) in series.items())),
              np.ceil(max((o - stable_values(k, "OP-Z3").median()).abs().max() for k, (o, t) in series.items()) / OP_PER_C))
fig, axs = plt.subplots(7, 2, figsize=(30, 30)); fig.subplots_adjust(left=.05, right=.95, top=.875, bottom=.04, hspace=.62, wspace=.2)
flow_rows = []
for ax, k in zip(axs.T.ravel(), date_order):
    o, t = series[k]; m0 = min(o.index.min(), t.index.min())
    tm = stable_values(k, "TC-Z3").median()
    bx = ax.twinx()
    ax.axhspan(tm - 1, tm + 1, color="#e8edf1", lw=0, zorder=0)
    ax.plot(t.index - m0, t, color=TC_COLOR, lw=1.3, label="TC-Z3")
    bx.plot(o.index - m0, o, color=OP_COLOR, lw=1.1, ls="--", alpha=.85, label="OP-Z3")
    om = stable_values(k, "OP-Z3").median(); oh = tc_half * OP_PER_C
    ax.axhline(tm, color="#9fb3c3", lw=.8, ls=":")
    ax.set_ylim(tm - tc_half, tm + tc_half); bx.set_ylim(om - oh, om + oh)
    bx.set_yticks([v for v in range(0, 101, 25) if om - oh <= v <= om + oh])
    n = len(t); both = pd.concat([o, t], axis=1, keys=["OP", "TC"]).dropna()
    rho = both.OP.corr(both.TC, method="spearman")
    end_t = (t.iloc[int(n * .9):] - tm).median(); end_o = (o.iloc[int(len(o) * .9):] - stable_values(k, "OP-Z3").median()).median()
    together = "반대로 움직임" if rho <= -.4 else "같이 움직임" if rho >= .4 else "관계 뚜렷하지 않음"
    big = abs(end_t) >= 2
    ax.set_title(f"{k} · {lot_meta.loc[k, 'GRADE']} · {lot_meta.loc[k, 'START']:%m/%d} · 비교에 쓴 온도값 {n}개(1분 단위)\n"
                 f"{together} ({rho:+.2f}) · 표시된 값 중 마지막 10%: 온도 {end_t:+.1f}℃ / 출력 {end_o:+.1f}%p",
                 loc="left", fontsize=14, pad=8, color=GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
    if big:   # 안정 참고 유효값 끝부분의 수준 차이이며 종료 전환을 포함하지 않는다.
        ax.annotate("★", (0, 1), xycoords="axes fraction", xytext=(-8, 25), textcoords="offset points", ha="right", va="baseline", fontsize=20, color="#3f5a70")
    style(ax, ""); ax.tick_params(labelsize=11); ax.tick_params(axis="y", colors=TC_COLOR); bx.tick_params(labelsize=11, colors=OP_COLOR, labelbottom=False)   # 출력 축은 가로 눈금 숫자를 쓰지 않는다
    bx.spines[["top"]].set_visible(False); bx.spines["right"].set_color("#a9bac7")
    ax.set_ylabel("온도[℃]", fontsize=11.5, color=TC_COLOR); bx.set_ylabel("제어출력[%]", fontsize=11.5, color=OP_COLOR)
    ax.set_xlim(0, max(t.index.max(), o.index.max()) - m0)
    for s, y_, e_ in [("TC-Z3", stable_values(k, "TC-Z3"), end_t), ("OP-Z3", stable_values(k, "OP-Z3"), end_o)]:
        flow_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "SENSOR": s, "안정 구간[분]": len(y_),
                          "기울기[100분당]": np.polyfit(y_.index.to_numpy(), y_.to_numpy(), 1)[0] * 100, "표시된 값 중 마지막 10% 값": e_, "온도·출력 관계(순위상관)": rho})
for ax in axs[-1]:
    ax.set_xlabel("안정 참고 구간 시작 후[분]", fontsize=12)
flow_table = pd.DataFrame(flow_rows)
header(fig, "안정 참고 구간 · Z3 제어출력과 온도를 같이 보기",
       "LOT마다 한 칸 · 왼쪽 실선 = 온도 TC-Z3 · 오른쪽 점선 = 제어출력 OP-Z3 (같은 존 색 · 1분 값) · 두 축 모두 LOT 중앙값을 가운데(점선)에, 높이 1℃ = 10%p · 회색 띠 = ±1℃(= ±10%p) · 강종은 칸 제목에 표기",
       [Line2D([0], [0], color=TC_COLOR, lw=2.2, label="온도 TC-Z3 (왼쪽 축)"), Line2D([0], [0], color=OP_COLOR, lw=1.8, label="제어출력 OP-Z3 (오른쪽 축)"),
        Patch(facecolor="#e8edf1", label="중앙값 ±1℃ (= 출력 ±10%p)"),
        Line2D([0], [0], marker="*", ls="None", color="#3f5a70", ms=18, label="안정 참고 끝부분 온도 차이 2℃ 이상 (종료 전환 제외)")], ncol=4)
compact_header(fig, .955, .935, title_size=26, sub_size=14, legend_size=14)
axis_tag(fig, "TIME AXIS · 안정 참고 구간 시작 후[분]")
fig.text(.035, .006, "칸 제목 읽는 법: 표시된 값 중 마지막 10% = 각 신호의 마지막 10% 중앙값과 표시된 전체 값 중앙값의 차이, ★ = 온도 차이 2℃ 이상, 종료 전환 제외",
         ha="left", va="bottom", fontsize=12.5, color="#526d80")
finish(fig, "01_수준변동/02_안정참고_Z3출력과온도.png")
ft = flow_table.loc[flow_table.SENSOR.eq("TC-Z3")].set_index("LOT"); fo = flow_table.loc[flow_table.SENSOR.eq("OP-Z3")].set_index("LOT")
opp = ft.loc[ft["온도·출력 관계(순위상관)"].le(-.4), "온도·출력 관계(순위상관)"].sort_values()
endbig = ft.index[ft["표시된 값 중 마지막 10% 값"].abs().ge(2)].tolist()
note("\n".join([
    "- 같은 칸에서 보라 점선(출력)과 보라 실선(온도)이 **반대로** 움직이면, 온도가 내려갈 때 제어가 출력을 올려 보충한 것이다. 출력은 그대로인데 온도만 움직이면 출력 말고 다른 이유를 찾아야 한다.",
    f"- 반대로 움직인 정도가 뚜렷한 LOT(순위상관 -0.4 이하)은 " + ", ".join(f"{k}({r:+.2f})" for k, r in opp.items()) + f"이다 ({len(opp)}/{len(ft)} LOT)." + ("" if len(opp) == len(ft) else " 나머지 LOT은 관계가 뚜렷하지 않다.") if len(opp) else "- 반대로 움직인 정도가 뚜렷한 LOT은 없다.",
    f"- 300분 넘게 관측된 LOT의 기울기는 온도 100분당 ±{ft.loc[ft['안정 구간[분]'].ge(300), '기울기[100분당]'].abs().max():.1f}℃, 출력 100분당 ±{fo.loc[fo['안정 구간[분]'].ge(300), '기울기[100분당]'].abs().max():.1f}%p 이하라, 시간이 지나며 서서히 바뀐 LOT은 없다."]
    + [f"- **{k}**(★)은 표시된 값 중 마지막 10%에서 안정 참고 전체 중앙값보다 온도 {ft.loc[k, '표시된 값 중 마지막 10% 값']:+.1f}℃, 출력 {fo.loc[k, '표시된 값 중 마지막 10% 값']:+.1f}%p 달랐다. "
       + ("출력 중앙값 차이는 작다. 이것만으로 온도 변화의 원인을 확정하지 않는다." if abs(fo.loc[k, '표시된 값 중 마지막 10% 값']) < 5 else "온도와 반대 방향으로 출력이 바뀌었다. 종료 전환을 제외한 구간이며 정지 원인을 판단하는 값은 아니다.") for k in endbig]))
table(flow_table)


#### 수준·흐름 그림을 읽는 기준

![Z3 안정 참고 온도와 출력, 전체 경과시간과 표시된 값 중 마지막 10%](<../../figures/05_all_data_timeseries_jinny_02/01_수준변동/02_안정참고_Z3출력과온도.png>)

- **첫 그림 (a)(b):** 점 하나가 LOT 하나의 안정 참고 구간 중앙값, 굵은 막대가 가운데 50%다. 온도는 14 LOT이 거의 같은 값에 모여 있고, 제어출력은 강종마다 다르다.
- **둘째 그림:** LOT마다 한 칸이다. 왼쪽 보라 실선은 온도 TC-Z3, 오른쪽 보라 점선은 제어출력 OP-Z3이고, 둘 다 1분 값이다. 30분 이동 중앙값으로 그리면 몇 분짜리 오르내림이 지워져서 1분 값을 그대로 쓴다.
- **두 축 맞춤:** 두 축 모두 그 LOT의 중앙값을 가운데(점선)에 두고, 높이 1℃ = 출력 10%p로 맞췄다(1분 값에서 출력이 보통 움직이는 폭이 온도 폭의 약 9배라서, 평소 움직임이 두 선에서 비슷한 높이가 되게 했다). 그래서 두 선이 위아래 어디에 있는지는 뜻이 없고, **같은 시점에 어느 방향으로 움직였는지**만 비교한다.
- **읽는 법:** 온도가 내려갈 때 출력이 올라가면(반대로 움직임) 제어가 보충한 것이다. 출력은 그대로인데 온도만 움직이면 출력 말고 다른 이유를 찾아야 한다. 칸 제목의 "반대로 움직임"은 두 선의 순위상관이 -0.4 이하일 때다. 1분 값에서는 14 LOT 모두 반대로 움직인다.
- **표시된 값 중 마지막 10%:** 각 신호의 안정 참고, 유효율 90% 이상인 1분 값 중 마지막 10%의 중앙값과 그 신호의 안정 참고 전체 중앙값 차이다. 온도 차이가 2℃ 이상이면 ★를 붙인다. 종료 전환은 제외하므로 전체 LOT 끝부분의 정지 변화를 뜻하지 않는다. 240058의 정지와 240061의 재가동 판정은 1절의 전체 시계열 근거를 따른다.
- **강종:** 각 칸 제목의 강종 코드를 읽는다. TC와 OP는 같은 Z3 색을 쓰고 실선과 점선으로 구분한다.
- 안정 참고는 사후 형태 구분이며 정상 정답이 아니다. 22개 센서 전수 수치는 아래 표에 있다.

<a id="response-patterns"></a>

#### 출력이 0%·100%에 붙을 때의 온도

In [ ]:
# 출력이 0%·100%에 5분 이상 붙은 구간을, 출력이 끝에 닿은 순간(0분)에 맞춰 겹친다.
# 중앙값(굵은 선)과 가운데 50%(띠)로 보여주고, 읽을 지점에 직접 글을 단다. 회복 시간은 60분까지 본다(04와 같은 기준).
# 0 cases is distinct from zero recovery time or no temperature response.
NO_RECOVERY_CASE_LABEL = "조건 충족 사례 0회\n회복 시간 미산출"
RECOVERY_CASE_NOTE = "미산출 · 선정 조건을 충족한 출력 100% 사례가 없어 회복 시간을 계산하지 않음. 0분·온도 무반응·개선을 뜻하지 않음."

lot_ids = lots.LOT_NO.astype(int).tolist()
main_opp = opposite.loc[~opposite["예정된 변화 중"]]
sat = pd.DataFrame({f"Z{z}": [(lambda y: (y.ge(98) | y.le(2)).mean() * 100)(profile[k][(f"OP-Z{z}", "median")].where(states[k].eq("안정 참고") & quality[k][f"OP-Z{z}"].ge(.9)).dropna())
                              for k in lot_ids] for z in range(1, 9)}, index=lot_ids)
ep_rows = []
for k in lot_ids:
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9)); tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on in [("100%", op.ge(98)), ("0%", op.le(2))]:
            for a, b in runs(on.fillna(False), 5):
                if states[k].loc[a] != "안정 참고":
                    continue
                ep_rows.append({"LOT": k, "ZONE": f"Z{z}", "출력": kind, "START_MIN": a, "길이[분]": b - a,
                                "tc": (tc.reindex(range(a - 10, a + 61)) - tc.loc[a]).to_numpy(), "op": op.reindex(range(a - 10, a + 61)).to_numpy()})
saturation = pd.DataFrame(ep_rows); tt = np.arange(-10, 61)
zone_count = saturation.groupby(["ZONE", "출력"]).size().unstack(fill_value=0).reindex([f"Z{z}" for z in range(1, 9)], fill_value=0).reindex(columns=["100%", "0%"], fill_value=0)
only = zone_count.index[zone_count.sum(axis=1).gt(0)].tolist()
z3 = saturation.loc[saturation.ZONE.eq("Z3")]
fig = plt.figure(figsize=(30, 19))
gs = fig.add_gridspec(3, 2, height_ratios=[1.45, .55, 1.15], left=.06, right=.97, top=.82, bottom=.1, hspace=.42, wspace=.16)
response = {}; arrow = dict(arrowstyle="->", color="#17344d", lw=1.4)
for col, kind in enumerate(["100%", "0%"]):
    d = z3.loc[z3["출력"].eq(kind)]
    if d.empty:
        continue
    T, O = np.vstack(d.tc), np.vstack(d.op); color = "#d55e00" if kind == "100%" else "#0072b2"
    m, om = np.nanmedian(T, axis=0), np.nanmedian(O, axis=0)
    a1 = fig.add_subplot(gs[0, col]); a2 = fig.add_subplot(gs[1, col], sharex=a1)
    if len(d) <= 8:
        for t_, o_ in zip(T, O):
            a1.plot(tt, t_, color=color, lw=.9, alpha=.3); a2.plot(tt, o_, color=color, lw=.9, alpha=.3)
    else:
        a1.fill_between(tt, np.nanpercentile(T, 25, axis=0), np.nanpercentile(T, 75, axis=0), color=color, alpha=.18, lw=0)
        a2.fill_between(tt, np.nanpercentile(O, 25, axis=0), np.nanpercentile(O, 75, axis=0), color=color, alpha=.18, lw=0)
    a1.plot(tt, m, color=color, lw=3.5); a2.plot(tt, om, color=color, lw=3.5)
    for a in [a1, a2]:
        a.axvline(0, color="#17344d", lw=1.2, ls="--"); style(a, ""); a.tick_params(labelsize=12.5)
    a1.axhline(0, color="#9fb3c3", lw=1); a1.set_xlim(-10, 30); a1.set_ylim(-18, 15); a2.set_ylim(-5, 105)
    after = m[10:]
    if kind == "100%":
        i_ext = int(np.nanargmin(after[:26])); back = next((i for i in range(i_ext, len(after)) if after[i] >= -.5), None)
        response[kind] = {"사례": len(d), "더 움직인 시간[분]": i_ext, "최대 변화[℃]": after[i_ext], "원래 온도로[분]": back}
        a1.annotate(f"출력은 이미 100%인데\n{i_ext}분 더 내려가 {after[i_ext]:+.1f}℃", (i_ext, after[i_ext]), xytext=(i_ext + 5, after[i_ext] - 6), fontsize=14, color="#17344d", arrowprops=arrow)
        a1.annotate(f"약 {back}분 뒤 원래 온도", (back, after[back]), xytext=(back + 4, 7), fontsize=14, color="#17344d", arrowprops=arrow)
        a1.set_title(f"(a) 출력이 100%에 닿으면 — 온도는 바로 오르지 않고 {i_ext}분 더 내려간 뒤 돌아온다 ({len(d)}회 · 띠 = 가운데 50%)", loc="left", fontsize=16, pad=12)
    else:
        i_ext = int(np.nanargmax(after[:15])); j_low = int(np.nanargmin(after[i_ext:31])) + i_ext
        op_back = next((i for i in range(len(after)) if om[10 + i] >= 50), None)
        response[kind] = {"사례": len(d), "더 움직인 시간[분]": i_ext, "최대 변화[℃]": after[i_ext], "원래 온도로[분]": np.nan}
        a1.annotate(f"출력은 이미 0%인데\n{i_ext}분 더 올라 {after[i_ext]:+.1f}℃", (i_ext, after[i_ext]), xytext=(i_ext + 5, after[i_ext] + 4), fontsize=14, color="#17344d", arrowprops=arrow)
        a1.annotate(f"출력이 돌아온 뒤({op_back}분)에도\n{j_low}분까지 {after[j_low]:+.1f}℃로 내려감", (j_low, after[j_low]), xytext=(j_low + 5, -14.5), fontsize=14, color="#17344d", arrowprops=arrow, va="center")
        a1.set_title(f"(b) 출력이 0%에 닿으면 — 온도는 {i_ext}분 더 오르고, 출력이 돌아온 뒤에는 크게 내려간다 ({len(d)}회 · 가는 선 = 한 번)", loc="left", fontsize=16, pad=12)
    a1.set_ylabel("온도 TC-Z3 변화[℃] (0분 = 0)", fontsize=13); a2.set_ylabel("제어출력 OP-Z3[%]", fontsize=13)
    a2.set_xlabel("출력이 끝에 닿은 시점 기준 시간[분]", fontsize=13); a1.tick_params(labelbottom=False)
rec_rows = []
for r in z3.loc[z3["출력"].eq("100%")].itertuples():
    after = r.tc[10:]; i_ext = int(np.nanargmin(after[:26]))
    rec_rows.append({"LOT": r.LOT, "GRADE": lot_meta.loc[r.LOT, "GRADE"], "시작": lot_meta.loc[r.LOT, "START"] + pd.Timedelta(minutes=r.START_MIN),
                     "최대 하강[℃]": after[i_ext], "원래 온도로[분]": next((i for i in range(i_ext, len(after)) if after[i] >= -.5), np.nan)})
recovery = pd.DataFrame(rec_rows)
# (c) LOT별 회복 시간: 색 대신 아래 글자로 LOT·강종·날짜를 적는다(관측 날짜 순서)
ax = fig.add_subplot(gs[2, :]); rng = np.random.default_rng(0); DOT = "#17344d"
for i, k in enumerate(lot_ids):
    r_ = recovery.loc[recovery.LOT.eq(k), "원래 온도로[분]"]
    if r_.empty:
        ax.text(i, 2, NO_RECOVERY_CASE_LABEL, ha="center", va="bottom", fontsize=11.5, color="#61798b", linespacing=1.35)
        continue
    ax.plot(i + rng.uniform(-.12, .12, len(r_)), r_, "o", color=DOT, ms=10, alpha=.75)
    ax.plot([i - .3, i + .3], [r_.median()] * 2, color=DOT, lw=3)
    ax.text(i + .33, r_.median(), f"{r_.median():.0f}분", va="center", fontsize=11.5, color="#526d80")
med_all = recovery["원래 온도로[분]"].median()
ax.axhline(med_all, color="#9fb3c3", ls=":", lw=1.5)
rho = pd.Series([lot_ids.index(k) for k in recovery.LOT]).corr(recovery["원래 온도로[분]"].reset_index(drop=True), method="spearman")
ax.set_xticks(range(len(lot_ids)), [f"{k}\n{lot_meta.loc[k, 'GRADE']}\n{lot_meta.loc[k, 'START']:%m/%d}" for k in lot_ids])
ax.set_xlim(-.6, len(lot_ids) - .2); ax.set_ylim(0, recovery["원래 온도로[분]"].max() + 6)
style(ax, ""); ax.set_ylabel("원래 온도로 돌아오기까지[분]", fontsize=13); ax.tick_params(labelsize=12)
ax.set_title(f"(c) 출력을 100%로 올린 뒤 원래 온도로 돌아오기까지 걸린 시간 — Z3가 열을 더 낼 여유가 줄면 이 시간이 점점 길어진다\n"
             f"     관측 날짜 순서로 놓아도 길어지지 않는다 (순위상관 {rho:+.2f}) · 점 = 한 번, 가로선 = LOT 중앙값, 점선 = 전체 중앙값 {med_all:.0f}분",
             loc="left", fontsize=16, pad=12)
header(fig, "출력이 0%·100%에 붙을 때 Z3 온도는 어떻게 움직이나",
       f"출력이 끝에 5분 이상 붙는 일은 Z1~Z8 가운데 {', '.join(only)}에서만 {int(zone_count.loc[only].sum().sum())}회(100% {int(zone_count.loc[only, '100%'].sum())}회 · 0% {int(zone_count.loc[only, '0%'].sum())}회) · 0분 = 출력이 끝에 닿은 시점 · 1분 중앙값",
       [Patch(facecolor="#d55e00", label="출력이 100%(≥98%)에 붙은 경우"), Patch(facecolor="#0072b2", label="출력이 0%(≤2%)에 붙은 경우")], ncol=2)
compact_header(fig, .935, .905, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, "(a)(b) 출력이 끝에 닿은 시점 기준[분] | (c) LOT 관측 날짜 순서")
fig.text(.035, .012, RECOVERY_CASE_NOTE, ha="left", va="bottom", fontsize=12, color="#526d80")
finish(fig, "02_출력응답/02_출력끝값_온도반응.png")
r100 = response.get("100%", {}); r0 = response.get("0%", {})
note("\n".join([
    f"- 출력이 0% 또는 100%에 5분 이상 붙는 일은 Z1~Z8 중 {', '.join(only)}에서만 일어난다. 다른 구역의 출력은 끝까지 가지 않고 중간에서 조절된다.",
    f"- (a) Z3 출력이 100%에 닿은 뒤에도 온도는 {r100.get('더 움직인 시간[분]')}분 동안 {r100.get('최대 변화[℃]', np.nan):+.1f}℃ 더 떨어지고, 약 {r100.get('원래 온도로[분]')}분 뒤 원래 온도로 돌아온다. 출력을 최대로 올려도 온도가 바로 따라오지 않는다.",
    f"- (b) 반대로 출력이 0%에 닿은 뒤에도 온도는 {r0.get('더 움직인 시간[분]')}분 동안 {r0.get('최대 변화[℃]', np.nan):+.1f}℃ 더 오른다. 출력이 다시 올라온 뒤에도 온도는 한동안 크게 내려간다." if r0 else "- (b) 출력 0% 사례는 없다.",
    f"- (c) {len(recovery)}회 모두 60분 안에 돌아왔고 중앙값은 {med_all:.0f}분이다. 관측 날짜 순서와의 순위상관은 {rho:+.2f}로, 날짜가 지나며 길어지지 않았다. 조건 충족 사례가 없는 LOT은 '조건 충족 사례 0회 · 회복 시간 미산출'로 표시한다. 온도가 반응하지 않았거나 회복 시간이 0분이라는 뜻이 아니며, 이 지표로 정비 전후 개선을 비교할 수 없다."]))
table(recovery); table(sat)


#### 출력·온도 그림으로 알 수 있는 것

출력이 끝(0%·100%)에 붙는 장면은 **Z3에서만** 반복된다. 출력이 100%에 닿아도 온도는 바로 오르지 않고 몇 분 더 내려간 뒤 돌아온다. 온도가 출력을 늦게 따라오는 것이다.

**(c) 원래 온도로 돌아오기까지 걸린 시간을 보는 이유**

- **무엇을 보여주나:** Z3는 온도가 내려가면 출력을 100%까지 올려서 되돌린다. 이때 돌아오는 데 걸리는 시간은 그 구역이 열을 더 낼 여유가 얼마나 남았는지를 보여준다.
- **고장과의 관계:** 히터나 버너가 약해지면 같은 100%로 올려도 회복이 점점 늦어진다. 예지보전에서 보고 싶은 것이 이런 "서서히 나빠지는 흐름"이다.
- **04와의 관계:** 04에서 Z3 반복을 "설명됨"으로 두려면, 이 시간이 날짜에 따라 길어지지 않는다는 확인이 필요하다. 길어지고 있었다면 Z3 반복은 설명된 반복이 아니라 나빠지는 신호가 된다.
- **결과:** 31회 모두 60분 안에 돌아왔고 중앙값은 12분이다. 1월부터 12월까지 관측 날짜 순서로 놓아도 길어지지 않았다(순위상관 -0.14).
- **미산출 표시:** 안정 참고로 분류된 시점에 시작하고, 출력 OP-Z3의 1분 유효 기록량이 90% 이상인 값에서 98% 이상이 5분 이상 연속되는 사례가 없으면 **조건 충족 사례 0회 · 회복 시간 미산출**로 적는다. 온도 반응이 없거나 회복 시간이 0분이라는 뜻이 아니다.
- **정비 전후 비교:** 한쪽이라도 조건 충족 사례가 없으면 회복 시간으로 전후 개선을 비교할 수 없다.
- **한계:** 회복 시간은 소재 두께·통판 속도 같은 부하에 따라서도 달라지는데 그 자료가 없다. 그래서 "나빠지는 흐름은 보이지 않는다"까지만 말할 수 있고, 가열 능력이 그대로라고 확정하지는 않는다.

아래 표는 같은 강종에서 6월 정기수리 전후로 Z3 출력이 끝에 붙어 있던 시간 비율이다.

In [ ]:
june = plans.iloc[0].STRT_DT
trend_rows = []
for g in ["SPHC", "SPFH590", "SPCE", "SPCEN"]:
    ids = [k for k in lot_ids if lot_meta.loc[k, "GRADE"] == g]
    pre = [k for k in ids if lot_meta.loc[k, "END"] < june]; post = [k for k in ids if lot_meta.loc[k, "START"] > june]
    trend_rows.append({"강종": g, "6월 수리 전": ", ".join(f"{k}({lot_meta.loc[k, 'START']:%m/%d}) {sat.loc[k, 'Z3']:.0f}%" for k in pre) or "-",
                       "6월 수리 후": ", ".join(f"{k}({lot_meta.loc[k, 'START']:%m/%d}) {sat.loc[k, 'Z3']:.0f}%" for k in post) or "-",
                       "변화": "줄어듦" if pre and post and max(sat.loc[post, "Z3"]) < min(sat.loc[pre, "Z3"]) else "비슷하거나 섞임"})
z3_trend = pd.DataFrame(trend_rows)
table(z3_trend)
down = z3_trend.loc[z3_trend["변화"].eq("줄어듦"), "강종"].tolist()
note(f"**결과:** Z3 출력이 0%·100%에 붙어 있던 시간은 {', '.join(down) or '어느 강종도'}에서 6월 수리 뒤 줄었지만, "
     f"{', '.join(z3_trend.loc[z3_trend['변화'].ne('줄어듦'), '강종'])}은 수리 전후가 비슷하거나 섞여 있다. 강종마다 방향이 달라 수리 효과로 단정할 수 없고, "
     "14 LOT을 날짜 순서로 놓아도 꾸준히 늘거나 주는 추세는 없다. 따라서 지금 자료로는 ①을 고장 신호로 쓰지 않고, 앞으로 LOT이 쌓일 때 같은 강종 안에서 늘어나는지 보는 감시 지표로 남긴다.")

<a id="cln"></a>

## 4. 세정 입구 — 두 수준과 240108의 짧은 변화

<a id="process-levels"></a>

#### 세정 입구 두 수준 — 강종·정비와 무관한 LOT 단계

In [ ]:
# 위: LOT 중앙값을 날짜 순서로 · 아래: 같은 강종(SPCE) LOT의 1분 중앙값 곡선
hi_mean, lo_mean = cln[cln >= CLN_SPLIT].mean(), cln[cln < CLN_SPLIT].mean()
HI_C, LO_C = "#16a085", "#5d6d7e"
fig = plt.figure(figsize=(30, 17))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.05], left=.05, right=.97, top=.83, bottom=.06, hspace=.45)
ax = fig.add_subplot(gs[0])
ax.plot(range(len(lot_ids)), [cln[k] for k in lot_ids], color="#b9c4cc", lw=1.5, zorder=1)
for i, k in enumerate(lot_ids):
    hi = cln[k] >= CLN_SPLIT
    ax.plot(i, cln[k], "o", ms=16, color=HI_C if hi else LO_C, zorder=3)
    ax.text(i, cln[k] + (.35 if hi else -.35), f"{cln[k]:.1f}", ha="center", va="bottom" if hi else "top", fontsize=13, color="#17344d")
for m_, label, c in [(hi_mean, f"{CLN_SPLIT:.1f}℃ 이상", HI_C), (lo_mean, f"{CLN_SPLIT:.1f}℃ 미만", LO_C)]:
    ax.axhline(m_, color=c, ls="--", lw=1.3)
    ax.text(len(lot_ids) - .4, m_ + (.15 if c == HI_C else -.15), f" {label} 평균 {m_:.1f}℃", va="bottom" if c == HI_C else "top", fontsize=13, color=c, fontweight="bold")
switch_rows = []
for i in range(len(lot_ids) - 1):
    a, b = lots.iloc[i], lots.iloc[i + 1]
    n = int(cal_work.STRT_DT.between(a.END, b.START).sum()); sw = (cln[lot_ids[i]] >= CLN_SPLIT) != (cln[lot_ids[i + 1]] >= CLN_SPLIT)
    switch_rows.append({"앞 LOT": lot_ids[i], "뒤 LOT": lot_ids[i + 1], "그 사이 정비[WO]": n, "수준 바뀜": sw})
    ax.text(i + .5, 63.3, f"정비 {n}", ha="center", fontsize=12, color=SELECT_COLOR if (sw and n == 0) else "#7c8995", fontweight="bold" if (sw and n == 0) else "normal")
switches = pd.DataFrame(switch_rows); zero_switch = int((switches["수준 바뀜"] & switches["그 사이 정비[WO]"].eq(0)).sum())
ax.set_ylim(62.8, 71.5); ax.set_xlim(-.5, len(lot_ids) + .9)
ax.set_xticks(range(len(lot_ids)), [f"{k}\n{lot_meta.loc[k, 'GRADE']}" for k in lot_ids])
for t, k in zip(ax.get_xticklabels(), lot_ids):
    t.set_color(GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
style(ax, ""); ax.set_ylabel("CLN-IN LOT 중앙값[℃]", fontsize=13); ax.tick_params(labelsize=12.5)
hi_v, lo_v = cln[cln >= CLN_SPLIT], cln[cln < CLN_SPLIT]
ax.set_title(f"(a) 세정 입구 온도는 LOT마다 {cln.min():.1f}~{cln.max():.1f}℃로 다르고 강종을 따르지 않는다 — {len(hi_v)} LOT은 {hi_v.min():.1f}~{hi_v.max():.1f}℃에 모이고 나머지는 {lo_v.min():.1f}~{lo_v.max():.1f}℃에 흩어짐 · 정비 0건인 LOT 사이에서도 높고 낮음이 {zero_switch}번 바뀜 (빨간 '정비 0')", loc="left", fontsize=16, pad=12)
bx = fig.add_subplot(gs[1])
spce = [k for k in lot_ids if lot_meta.loc[k, "GRADE"] == "SPCE"]
for k, c in zip(spce, ["#5d6d7e", "#9aa6ae", "#7f8c8d"]):
    y = profile[k][("CLN-IN", "median")]; hi = cln[k] >= CLN_SPLIT
    bx.plot(y.index, y, lw=2.4 if hi else 1.8, color=LOT_COLOR[k], ls=report_viz.lot_linestyle(k), label=f"{k} · {lot_meta.loc[k, 'START']:%m/%d} · {'높은' if hi else '낮은'} 쪽 {cln[k]:.1f}℃")
bx.set_xlim(0, lots.loc[lots.GRADE.eq("SPCE"), "MINUTES"].max() + 5); style(bx, "관측 시작 후 경과시간[분]"); bx.set_ylabel("CLN-IN[℃] (1분 중앙값)", fontsize=13)
bx.legend(fontsize=13, frameon=False, loc="upper right"); bx.tick_params(labelsize=12.5)
low_spce = [k for k in spce if cln[k] < CLN_SPLIT]; high_spce = [k for k in spce if cln[k] >= CLN_SPLIT]
bx.set_title(f"(b) 같은 SPCE: 낮은 LOT({'·'.join(map(str, low_spce))})은 천천히 올랐다가 한 번에 떨어지는 반복(톱니)이 이어지고, 높은 LOT({'·'.join(map(str, high_spce))})은 처음부터 약 3℃ 위에 있다",
             loc="left", fontsize=16, pad=12)
header(fig, "세정 입구 온도 (CLN-IN)", "위: 점 = LOT 중앙값(날짜 순서), 점 사이 숫자 = 두 LOT 사이에 시작한 CAL01 정비 수 · 아래: 같은 강종 LOT의 1분 중앙값",
       [Line2D([0], [0], marker="o", color=HI_C, ls="None", ms=12, label=f"{CLN_SPLIT:.1f}℃ 이상 LOT ({int((cln >= CLN_SPLIT).sum())}개)"),
        Line2D([0], [0], marker="o", color=LO_C, ls="None", ms=12, label=f"{CLN_SPLIT:.1f}℃ 미만 LOT ({int((cln < CLN_SPLIT).sum())}개)")], ncol=2)
compact_header(fig, .935, .905, title_size=26, sub_size=15, legend_size=14)
axis_tag(fig, "(a) LOT 날짜 순서 | (b) " + TAG_ELAPSED)
finish(fig, "03_과시효_세정/04_세정입구.png")
table(switches)


#### 세정 입구 온도로 알 수 있는 것

- **LOT끼리 다름:** 14 LOT의 세정 입구 중앙값은 64.5~70.0℃다. 7 LOT은 69.2~70.0℃에 모이고 나머지 7 LOT은 64.5~67.8℃에 흩어져 있다. 같은 강종 안에서도 갈리고, 강종으로 설명되는 정도가 우연 수준이라(01의 6절, 0.43) 강종으로 정해지는 값이 아니다.
- **정비 이력과의 관계:** 위 표처럼 두 LOT 사이에 정비가 0건이어도 높고 낮음이 바뀐 경우가 있어, 정비로 설명되지 않는다.
- **LOT 안의 변동:** 천천히 오르다 한 번에 1~2℃ 떨어지는 톱니 모양이 높은 쪽·낮은 쪽 모두에서 반복된다. 세정 구역 자체의 운전 반복으로 보이며, 소둔로 상태를 알려주는 신호가 아니다.
- **그래서 어떻게 쓰나:** 세정 입구 온도는 LOT끼리 비교해 이상을 찾는 값이 아니라 **운전 조건**으로 다룬다. 04에서도 세정 입구 수준으로는 LOT을 고르지 않는다(선정 안 함).
- 세정 설정값·운전 기록이 없어 LOT끼리 다른 이유는 확인하지 않는다.

#### 240108 세정 입구의 짧은 변화 — 반복 파형과 대조

판정: **설명됨.** 세정 입구가 평소처럼 오르내리는 중 가장 가파른 1분이고(이동 Z 4.0), 다른 LOT도 3.5~3.8로 같은 모양이다.

![240108 탐지 구간과 반복 파형 대조](<../../figures/05_all_data_timeseries_jinny_02/02_근거범위/06_남은_짧은변화_240108.png>)


In [ ]:
# ①~③ 뒤에 남은 이동 Z 구간: LOT 전체에서의 위치와 1초 원본
remain_z = scope.loc[scope["이유"].eq("남음") & scope.METHOD.ne("CP 규칙")]
for r in remain_z.itertuples():
    k, a, b = int(r.LOT), float(r.START_MIN), float(r.END_MIN); r0 = lot_meta.loc[k]; sens = r.SENSORS.split(" · ")
    panels = sens + [s.replace("OP-", "TC-") for s in sens if s.startswith("OP-") and s.replace("OP-", "TC-") not in sens]
    d = detect.loc[detect.LOT.eq(k) & detect.MINUTE.between(a, b - 1)]; zmax = d[[s + "_Z30" for s in sens]].abs().max().max()
    w = raw_window(k, max(0, a - 15), min(r0.MINUTES, b + 15)); t_a, t_b = r0.START + pd.Timedelta(minutes=a), r0.START + pd.Timedelta(minutes=b)
    fig = plt.figure(figsize=(30, 12))
    gs = fig.add_gridspec(2, len(panels), height_ratios=[.75, 1.5], left=.05, right=.97, top=.75, bottom=.08, hspace=.55, wspace=.22)
    top = fig.add_subplot(gs[0, :]); s0 = sens[0]; p = profile[k][(s0, "median")]
    top.plot(r0.START + pd.to_timedelta(p.index, unit="min"), p, color=sensor_color(s0), lw=1.3)
    for x0, x1 in work_ranges[k]:
        top.axvspan(r0.START + pd.Timedelta(minutes=x0), r0.START + pd.Timedelta(minutes=x1), color=COL["WORK"], alpha=.18, lw=0)
    top.axvspan(t_a - pd.Timedelta(minutes=4), t_b + pd.Timedelta(minutes=4), fill=False, ec=SELECT_COLOR, lw=2.5)
    top.set_xlim(r0.START, r0.END); clock_axis(top); top.set_ylabel(shown_unit(s0), fontsize=12); top.tick_params(labelsize=11.5)
    top.set_title(f"(a) {k} 전체 · {s0} 1분 중앙값 · 빨간 상자 = 아래 확대 위치 · 갈색 음영 = 실제 작업", loc="left", fontsize=14.5, pad=8)
    lines = []
    for j, s in enumerate(panels):
        ax = fig.add_subplot(gs[1, j]); v = w[s] * sensor_scale(s)
        ax.plot(w.index, v, color=sensor_color(s), lw=1.4); ax.axvspan(t_a, t_b, color=SELECT_COLOR, alpha=.12, lw=0)
        pre, dur = v.loc[:t_a - pd.Timedelta(seconds=1)].tail(600).median(), v.loc[t_a:t_b]
        lines.append(f"{s} 앞 10분 {pre:.1f} → 구간 중 {dur.min():.1f}~{dur.max():.1f}")
        ax.set_title(f"({'bcdef'[j]}) {s}: 앞 10분 중앙값 {pre:.1f} → 구간 중 {dur.min():.1f}~{dur.max():.1f}", loc="left", fontsize=13.5, pad=8)
        ax.set_xlim(w.index[0], w.index[-1]); clock_axis(ax); ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 5))); ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.set_xlabel("실제 시각[시:분]", fontsize=11.5); ax.set_ylabel(shown_unit(s), fontsize=11.5); ax.tick_params(labelsize=11)
    header(fig, f"남은 짧은 변화 · {k} · {r0.GRADE} · 최대 |Z| {zmax:.1f} · {int(r.MINUTES)}분", f"{when(k, a, b)} · {r.METHOD}가 잡았고 ①~③ 어느 이유와도 맞지 않음 · 빨간 음영 = 잡힌 구간 · 1초 원본",
           [Patch(facecolor=SELECT_COLOR, alpha=.25, label="잡힌 구간"), Patch(facecolor=COL["WORK"], alpha=.3, label="실제 작업")], ncol=2)
    compact_header(fig, .925, .89, title_size=25, sub_size=14, legend_size=13)
    axis_tag(fig, TAG_REAL)
    finish(fig, f"02_근거범위/06_남은_짧은변화_{k}.png")
    d1m = p.diff(); here = d1m.loc[a:b - 1]; here = here.loc[here.abs().idxmax()]
    note(f"**{k} · {when(k, a, b)}:** (a)에서 보듯 {s0} 값은 이 LOT 동안 {p.min():.1f}~{p.max():.1f} 사이를 천천히 오르내린다. 잡힌 1분은 그 가운데 한 하강에서 가장 가파른 1분"
         f"({here:+.2f}/분, LOT 안 1분 변화 크기 {int((d1m.abs() > abs(here)).sum()) + 1}위)이고, 최대 |Z|({zmax:.1f})는 기준({Z_LIMIT})과 거의 같다. "
         + " / ".join(lines) + f". 앞뒤 6시간 작업: {nearby_work(k, a, b)}. **반복 파형으로 설명되는 변화라 이상 후보에서 뺀다.**")
EXPLAINED_SHORT = remain_z.LOT.astype(int).tolist()


<a id="cp-small"></a>

## 5. CP — 한 번만 작게 벗어난 LOT은 지켜볼 것

14 LOT 전체에서 CP-Z4가 LOT 평소보다 0.05 넘게 벗어난 곳을 센다. **여러 번 벗어난 240069·240037과 정비 직후 Z3 이탈에 이어진 240117은 [03](05_all_data_timeseries_jinny_03.ipynb)에서 확대한다.** 한 번만 작게 벗어난 240024(−0.066)·240038(+0.056)·240091(+0.052)은 지켜볼 것으로 남긴다. 기준을 0.06으로 올리면 240038·240091은 빠진다.

<a id="cp-all"></a>

#### CP-Z4 — 14 LOT 원본에서 기준(±0.05)을 넘은 곳

CP-Z4가 **LOT 중앙값(모든 LOT 약 0.45)에서 0.05 넘게 벗어난 경우**를 1초 원본에서 모두 센다. 한 번은 중앙값에서 0.02 넘게 벗어나 있던 연속 구간이고, 크기는 그 구간에서 가장 크게 벗어난 값이다.

- **그림 읽는 법:** 한 줄이 한 LOT의 관측 전체다. 모든 줄이 같은 시간 눈금이라 관측 길이도 함께 비교된다. 회색 띠는 LOT 중앙값 ±0.05, 점은 벗어난 한 번의 꼭짓점이다(빨강 = 여러 번 벗어난 LOT, 주황 = 한 번). 오른쪽 끝은 횟수다.
- **세로 범위:** 작은 이탈도 보이게 ±0.2까지만 그렸다. 넘는 값은 숫자에 ↑·↓를 붙였다.
- **예정된 정지 중 변화**(240058 끝)는 빈 점으로만 두고 횟수에서 뺐다.
- 각 이탈의 실제 시각과 크기는 아래 표, 여러 번 벗어난 LOT의 파형은 이어지는 확대 그림에서 확인한다.

In [ ]:
# 14 LOT 전체에서 CP-Z4가 LOT 중앙값보다 0.05 넘게 벗어난 경우를 모두 센다(1초 원본).
WATCH = "#b8863b"   # 한 번 벗어난 LOT 색
cp_rows = []
for k in lots.LOT_NO.astype(int):
    s = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = s - s.median()
    off = dev.abs().gt(.02); grp = off.ne(off.shift()).cumsum()
    for _, part in dev[off].groupby(grp[off]):
        peak = part.abs().idxmax()
        if abs(part.loc[peak]) < .05:
            continue
        a, b, m = [(t - lot_meta.loc[k, "START"]).total_seconds() / 60 for t in (part.index[0], part.index[-1], peak)]
        cp_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "PEAK": peak, "DEV": part.loc[peak], "A": a, "B": b, "M": m,
                        "초": int((part.index[-1] - part.index[0]).total_seconds()) + 1, "EXP": in_expected(k, m)})
cp_events = pd.DataFrame(cp_rows)
cp_real = cp_events.loc[~cp_events.EXP]
cp_count = cp_real.groupby("LOT").size()
CP_FOCUS = cp_count[cp_count.ge(2)].sort_values(ascending=False).index.tolist(); CP_WATCH = cp_count[cp_count.eq(1)].index.tolist()
ids = lots.sort_values("START").LOT_NO.astype(int).tolist()
# 14 LOT의 CP-Z4 1초 원본을 같은 시간축에 한 줄씩 놓는다.
# 세로 = CP-Z4 - LOT 중앙값 · 회색 띠 = ±0.05 · 점 = 0.05 넘게 벗어난 한 번의 꼭짓점(빨강 = 여러 번 벗어난 LOT, 주황 = 한 번, 빈 점 = 예정된 정지 중)
# 세로는 ±0.2까지만 그린다(작은 이탈도 보이게). 넘는 값은 점을 끝에 두고 숫자에 ↑·↓를 붙인다.
fig, axs = plt.subplots(len(ids), 1, figsize=(28, 1.2 * len(ids) + 3.4), sharex=True, sharey=True)
H = 1.2 * len(ids) + 3.4; LIM = .2
fig.subplots_adjust(left=.13, right=.91, top=1 - 3.0 / H, bottom=.6 / H, hspace=.3)
for ax, k in zip(axs, ids):
    g = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = g - g.median()
    hrs = (g.index - lot_meta.loc[k, "START"]).total_seconds() / 3600
    ax.fill_between([0, hrs.max()], -.05, .05, color="#e3e9ee", lw=0)   # 띠는 관측한 시간까지만
    ax.plot(hrs, dev, color="#526d80", lw=.6)
    ev = cp_events.loc[cp_events.LOT.eq(k)]
    for r in ev.itertuples():
        c = EXPECTED_COLOR if r.EXP else SELECT_COLOR if k in CP_FOCUS else WATCH
        ax.plot(r.M / 60, np.clip(r.DEV, -LIM * .93, LIM * .93), "o", ms=9, mfc="white" if r.EXP else c, mec=c, mew=2, zorder=4)
    if len(ev):   # 가장 크게 벗어난 한 번만 값을 적는다
        r = ev.loc[ev.DEV.abs().idxmax()]
        arrow = "↑" if r.DEV > LIM else "↓" if r.DEV < -LIM else ""
        ax.annotate(("예정된 정지 중 " if r.EXP else "") + f"{r.DEV:+.3f}{arrow}", (r.M / 60, np.clip(r.DEV, -LIM * .93, LIM * .93)), xytext=(8, 0), textcoords="offset points", va="center", fontsize=12,
                    color=EXPECTED_COLOR if r.EXP else SELECT_COLOR if k in CP_FOCUS else WATCH, fontweight="bold" if k in CP_FOCUS else "normal")
    n = int(cp_count.get(k, 0)); col = SELECT_COLOR if k in CP_FOCUS else WATCH if n else "#9aa6ae"
    ax.text(-.01, .5, f"{k} · {lot_meta.loc[k, 'GRADE']} · {lot_meta.loc[k, 'START']:%m/%d}", transform=ax.transAxes, ha="right", va="center", fontsize=13,
            color=col if k in CP_FOCUS else "#17344d", fontweight="bold" if k in CP_FOCUS else "normal")
    ax.text(1.008, .5, f"{n}번" if n else "없음", transform=ax.transAxes, ha="left", va="center", fontsize=15, color=col, fontweight="bold" if k in CP_FOCUS else "normal")
    style(ax, ""); ax.set_yticks([-.05, .05], ["", ""]); ax.tick_params(labelsize=10.5); ax.set_ylim(-LIM, LIM)
    ax.spines["bottom"].set_visible(ax is axs[-1])
    if ax is not axs[-1]:
        ax.tick_params(axis="x", length=0)
axs[-1].set_xlim(0, lot_meta.MINUTES.max() / 60 + .2); axs[-1].set_xlabel("관측 시작 후 시간[시간] · 모든 줄이 같은 눈금이라 관측 길이도 함께 비교된다", fontsize=13)
axs[-1].tick_params(axis="x", labelsize=12.5)
fig.text(.91 + .008, 1 - 2.75 / H, "벗어난\n횟수", ha="left", va="bottom", fontsize=13, color="#526d80")
header(fig, "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳",
       "한 줄 = 한 LOT의 관측 전체(1초 원본) · 세로 = CP-Z4 - LOT 중앙값(모든 LOT 약 0.45) · 회색 띠 = ±0.05 · 점 = 0.05 넘게 벗어난 한 번의 꼭짓점 · 세로는 ±0.2까지(넘으면 숫자에 ↑↓) · 오른쪽 = 횟수(예정된 정지 중 제외)",
       [Line2D([0], [0], marker="o", ls="None", color=SELECT_COLOR, ms=11, label="여러 번 벗어난 LOT"), Line2D([0], [0], marker="o", ls="None", color=WATCH, ms=11, label="한 번 벗어난 LOT"),
        Line2D([0], [0], marker="o", ls="None", mfc="white", mec=EXPECTED_COLOR, mew=2, ms=11, label="예정된 정지 중 (횟수에서 뺌)"),
        Patch(facecolor="#e3e9ee", label="LOT 중앙값 ±0.05")], ncol=4)
compact_header(fig, 1 - 1.05 / H, 1 - 1.55 / H, title_size=25, sub_size=14, legend_size=13)
axis_tag(fig, "TIME AXIS · 관측 시작 후 시간[시간]")
finish(fig, "02_근거범위/03_CP이탈_14LOT.png")
table(cp_real.assign(시각=cp_real.PEAK.dt.strftime("%m/%d %H:%M:%S"), 크기=cp_real.DEV.round(3))[["LOT", "GRADE", "시각", "크기", "초"]].sort_values(["LOT", "시각"]))

<a id="cycle"></a>

## 6. Z9·냉각 차이 — 관측이 반복 주기보다 짧아 생긴 차이

Z9·냉각(TC-OUT1·OUT2)은 원래 오르내리는 반복 파형이다(Z9 주기 약 196분, 냉각 약 43~46분). 240133(SPCE, 안정 관측 108분)의 Z9 −4.7℃와 240044(SPHC, 75분)의 Z9 +8.1℃·OUT1 −2.7℃는 **관측이 두 주기보다 짧아 반복 파형의 일부만 본 결과**다. 수준 차이로 보지 않는다. 강종별 주기·위상 그림은 [05 부록](05_all_data_timeseries_jinny_05.ipynb)의 강종 전체 비교에 있다.

<a id="hold"></a>

## 7. 판단 보류 — LOT 사이 정비 지표와 LOT이 1~2개뿐인 강종

<a id="maintenance-flow"></a>

#### 정비 전후 흐름 — LOT 사이 정비와 LOT별 상태 지표

**목적:** 정비 뒤에 설비 상태가 달라졌는지 본다. 온도는 14 LOT 관측 구간에만 있어 정비하는 동안의 온도는 거의 없다(작업 중 온도 기록은 240117의 2건뿐, [01 4절 정비 이력 상세](05_all_data_timeseries_jinny_01.ipynb#maintenance-detail)). 그래서 **정비 앞뒤 LOT을 비교**한다.

- **가로축:** LOT 순서(관측 날짜 순, 같은 간격). **사이 칸**에는 앞 LOT 관측 끝 ~ 뒤 LOT 관측 시작 사이에 시작한 CAL01 정비를 종류별 개수로 적는다(BM ● · CBM ◆ · TBM ■ · 부품 교체 ★). 관측 중에 시작한 정비는 그 LOT 칸 위에 적는다. 회색 숫자는 두 LOT 사이 일수, 진한 사이 칸은 CR1 정기수리가 있었던 칸이다.
- **읽는 법:** 사이 칸의 왼쪽 LOT = 정비 전, 오른쪽 LOT = 정비 뒤.
- **지표:** 강종마다 수준이 다른 온도·출력 값 대신, 강종 영향을 덜 받는 세 지표를 쓴다.
  1. 출력 100% 뒤 원래 온도로 돌아오기까지 걸린 시간(위 그림 (c)와 같은 기준). **조건 충족 사례 0회이면 미산출이며, 한쪽이라도 미산출이면 전후 회복 시간 비교 불가**다.
  2. CP-Z4가 LOT 중앙값보다 0.05 넘게 벗어난 횟수(5절과 같은 기준)
  3. Z3 출력과 온도가 반대로 움직인 정도(3절 그림과 같은 1분 값 순위상관). -1에 가까울수록 출력이 온도를 바로 따라 움직인다.

In [ ]:
# 정비 전후 흐름: 가로축 = LOT 순서(같은 간격). 정비는 '두 LOT 사이' 칸에 모아, 왼쪽 LOT = 정비 전 · 오른쪽 LOT = 정비 뒤로 읽는다.
# 지표는 강종 영향을 덜 받는 것만 쓴다: 출력 100% 뒤 회복 시간(2절) · CP-Z4가 벗어난 횟수 · 출력과 온도가 반대로 움직인 정도(1절, 1분 값)
TYPE_C = {"BM": report_viz.MAINTENANCE_COLORS["BM"], "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"]}; TYPE_M = {"BM": "o", "CBM": "D", "TBM": "s"}
cp_n = {}
for k in lot_ids:   # CP-Z4가 LOT 중앙값보다 0.05 넘게 벗어난 횟수(1초 원본 · 0.02 넘게 벗어나 있던 연속 구간을 한 번으로 · 예정된 정지 중 제외)
    s = raw[k].set_index("MEAS_DT")["CP-Z4"].dropna(); dev = s - s.median(); off = dev.abs().gt(.02); grp = off.ne(off.shift()).cumsum()
    cp_n[k] = sum(1 for _, part in dev[off].groupby(grp[off]) if part.abs().max() >= .05
                  and not in_expected(k, (part.abs().idxmax() - lot_meta.loc[k, "START"]).total_seconds() / 60))
rho_of = flow_table.loc[flow_table.SENSOR.eq("TC-Z3")].set_index("LOT")["온도·출력 관계(순위상관)"]
# 정비를 칸에 배정: 두 LOT 사이(앞 LOT 끝 ~ 뒤 LOT 시작) 또는 LOT 관측 중 · 첫 LOT 전 · 마지막 LOT 뒤
ends = [lot_meta.loc[k, "END"] for k in lot_ids]; starts = [lot_meta.loc[k, "START"] for k in lot_ids]
def slot(t):
    for i, k in enumerate(lot_ids):
        if starts[i] <= t <= ends[i]: return float(i)          # 관측 중
    return next((i - .5 for i in range(len(lot_ids)) if t < starts[i]), len(lot_ids) - .5)
mnt = cal_work.assign(X=cal_work.STRT_DT.map(slot), PART=cal_work.WO_NO.isin(cal_parts.WO_NO))
fig = plt.figure(figsize=(30, 17.5))
gs = fig.add_gridspec(4, 1, height_ratios=[.85, 1, .7, .7], left=.075, right=.975, top=.84, bottom=.1, hspace=.3)
m = fig.add_subplot(gs[0]); a1 = fig.add_subplot(gs[1], sharex=m); a2 = fig.add_subplot(gs[2], sharex=m); a3 = fig.add_subplot(gs[3], sharex=m)
plan_gap = {}
for p in plans.itertuples():   # 정기수리가 들어 있는 사이 칸
    x = slot(p.STRT_DT); plan_gap[x] = f"CR1 정기수리 {p.STRT_DT:%m/%d}~{p.END_DT:%m/%d}"
for a in [m, a1, a2, a3]:
    for i in range(len(lot_ids)):
        a.axvspan(i - .18, i + .18, color="#f3f5f7", lw=0, zorder=0)            # LOT 칸
    for x in plan_gap:
        a.axvspan(x - .32, x + .32, color="#dfe9f2", lw=0, zorder=0)            # 정기수리 사이 칸
    style(a, ""); a.tick_params(labelsize=12)
# 위 칸: 사이 칸마다 정비 종류별 개수 · 부품 교체 · 두 LOT 간격
LANES = ["BM", "CBM", "TBM", "부품 교체"]
for x, d in mnt.groupby("X"):
    for y, kind in enumerate(LANES):
        n = int(d.PART.sum()) if kind == "부품 교체" else int(d.MNT_TYPE.eq(kind).sum())
        if n:
            mk, col = ("*", SELECT_COLOR) if kind == "부품 교체" else (TYPE_M[kind], TYPE_C[kind])
            m.plot(x - .07, y, mk, color=col, ms=15 if mk == "*" else 12, zorder=3)
            m.text(x + .04, y, f"{n}", va="center", fontsize=13, color=col, fontweight="bold")
for i in range(len(lot_ids) - 1):
    m.text(i + .5, -1.05, f"{(starts[i + 1] - ends[i]).days}일", ha="center", va="center", fontsize=11.5, color="#7c8995")
for x, label in plan_gap.items():
    m.text(x, -1.7, label, ha="center", va="center", fontsize=12, color="#3f5a70", fontweight="bold")
m.set_yticks(range(len(LANES)), [f"{k} {int(cal_work.MNT_TYPE.eq(k).sum())}건" if k != "부품 교체" else f"부품 교체 {int(mnt.PART.sum())}건" for k in LANES])
m.set_ylim(len(LANES) - .4, -2.1)
m.set_title("LOT 사이에 있었던 CAL01 정비 (사이 칸 = 앞 LOT 끝 ~ 뒤 LOT 시작 · 회색 숫자 = 두 LOT 사이 일수 · LOT 칸 위 = 관측 중 정비)", loc="left", fontsize=14.5, pad=8)
# 지표 1: 출력 100% 뒤 원래 온도로 회복 시간
for i, k in enumerate(lot_ids):
    cases = recovery.loc[recovery.LOT.eq(k), "원래 온도로[분]"]
    r_ = cases.dropna()
    if cases.empty:
        a1.text(i, 4, NO_RECOVERY_CASE_LABEL, ha="center", va="bottom", fontsize=11.5, color="#61798b", linespacing=1.35)
        continue
    if r_.empty:
        a1.text(i, 4, "회복 확인 안 됨\n회복 시간 미산출", ha="center", va="bottom", fontsize=11.5, color="#61798b")
        continue
    a1.plot([i] * len(r_), r_, "o", color="#9aa6ae", ms=8, alpha=.85, zorder=2)
    a1.plot([i - .16, i + .16], [r_.median()] * 2, color="#17344d", lw=3.5, zorder=3)
    a1.text(i + .2, r_.median(), f"{r_.median():.0f}분", va="center", fontsize=11.5, color="#17344d")
a1.set_ylim(0, recovery["원래 온도로[분]"].max() + 8); a1.set_ylabel("회복 시간[분]", fontsize=12.5)
a1.set_title("① 출력 100% 뒤 원래 온도로 돌아오기까지 (점 = 한 번 · 가로 막대 = LOT 중앙값 · 3절 출력 끝값 분석과 같은 기준) — 가열 여유가 줄면 길어진다", loc="left", fontsize=14.5, pad=8)
# 지표 2: CP-Z4가 벗어난 횟수
for i, k in enumerate(lot_ids):
    a2.bar(i, cp_n[k], width=.28, color="#b04482" if cp_n[k] else "#e7d3df", zorder=2)
    a2.text(i, cp_n[k] + .3, f"{cp_n[k]}", ha="center", va="bottom", fontsize=12, color="#b04482" if cp_n[k] else "#9aa6ae")
a2.set_ylim(0, max(cp_n.values()) + 2.2); a2.set_ylabel("횟수", fontsize=12.5)
a2.set_title("② CP-Z4가 LOT 중앙값보다 0.05 넘게 벗어난 횟수 (1초 원본 · 예정된 정지 중 제외)", loc="left", fontsize=14.5, pad=8)
# 지표 3: Z3 출력과 온도가 반대로 움직인 정도
for i, k in enumerate(lot_ids):
    a3.plot(i, rho_of[k], "o", color="#d55e00", ms=11, zorder=3)
    a3.text(i + .2, rho_of[k], f"{rho_of[k]:+.2f}", va="center", fontsize=11.5, color="#d55e00")
a3.set_ylim(-1, -.5); a3.set_ylabel("순위상관", fontsize=12.5)
a3.set_title("③ Z3 출력과 온도가 반대로 움직인 정도 (1분 값 순위상관 · -1에 가까울수록 출력이 온도를 바로 따라 움직임 · 3절 Z3 흐름과 같은 계산)", loc="left", fontsize=14.5, pad=8)
for a in [m, a1, a2]:
    a.tick_params(labelbottom=False)
a3.set_xticks(range(len(lot_ids)), [f"{k}\n{lot_meta.loc[k, 'GRADE']}\n{lot_meta.loc[k, 'START']:%m/%d}" for k in lot_ids])
for lab, k in zip(a3.get_xticklabels(), lot_ids):
    lab.set_color(GRADE_COLOR[lot_meta.loc[k, "GRADE"]])
a3.set_xlim(-.6, len(lot_ids) - .4)
header(fig, "정비 전후 흐름 · LOT 사이 정비와 LOT별 상태 지표",
       "가로 = LOT 순서(관측 날짜 순) · 사이 칸 = 두 LOT 사이에 있었던 정비 · 사이 칸의 왼쪽 LOT = 정비 전, 오른쪽 LOT = 정비 뒤 · 지표는 강종 영향을 덜 받는 3가지 · 정비 효과 확정 아님",
       [Line2D([0], [0], marker=TYPE_M[k], ls="None", color=TYPE_C[k], ms=11, label=k) for k in TYPE_C] +
       [Line2D([0], [0], marker="*", ls="None", color=SELECT_COLOR, ms=14, label="부품 교체"), Patch(facecolor="#dfe9f2", label="CR1 정기수리가 있었던 사이 칸")], ncol=5)
compact_header(fig, .93, .9, title_size=25, sub_size=14, legend_size=13)
axis_tag(fig, "LOT 순서 · 사이 칸 = 정비")
fig.text(.035, .012, RECOVERY_CASE_NOTE, ha="left", va="bottom", fontsize=12, color="#526d80")
finish(fig, "02_출력응답/03_정비전후_LOT지표흐름.png")


#### 정비 전후 흐름으로 알 수 있는 것

| 정비 | 정비 전 → 뒤 LOT | 사이 | ① 회복 시간 | ② CP 횟수 | ③ 반대 움직임 |
|---|---|---|---|---|---|
| 6월 정기수리 | 240058(DP590) → 240061(SPHC) | 5일 · TBM 6건 · 부품 교체 1건 | 41분(1회) → 미산출(조건 충족 사례 0회 · 전후 비교 불가) | 0 → 0 | -0.78 → -0.83 |
| 11월 정기수리 | 240114(SPCEN) → 240117(SPCD) | 8일 · TBM 5건 · 관측 중 CBM·TBM 1건씩 | 중앙값 16분 → 6분 | 0 → 1 | -0.68 → -0.74 |
| 7~9월 정비 | 240069(SPHC) → 240091(SPHC) | 62일 · BM 5건 · CBM 1건 · TBM 2건 · 부품 교체 3건 | 양쪽 모두 미산출(조건 충족 사례 0회 · 전후 비교 불가) | 7 → 1 | -0.70 → -0.70 |

- **1년 흐름:** 세 지표 모두 LOT 순서를 따라 나빠지는 흐름은 보이지 않는다. ③은 -0.83~-0.68 사이에 있다.
- **정비 앞뒤 차이:** 숫자가 달라진 곳은 있지만, 앞뒤 LOT이 하나씩이고 강종이 다르거나(6월·11월) 사이에 정비가 여러 건 있어(7~9월) **정비 때문이라고 말할 수 없다.** "정비 전후에 이런 차이가 있었다"까지만 기록한다.
- **한계:** 정비 기록의 설비 이름이 CAL01 전체 단위라 어느 구역을 손봤는지 모른다. 정비하는 동안의 온도는 240117의 2건 말고는 없다. 회복 시간은 선정 조건을 충족한 출력 100% 사례가 있는 LOT만 잴 수 있다. **240044·240061·240069·240091은 조건 충족 사례 0회로 회복 시간이 미산출**이다. 6월의 `41분 → 미산출`은 회복이 빨라졌다는 뜻이 아니며, 7~9월도 양쪽 모두 미산출이므로 이 지표의 정비 전후 비교는 불가하다.

#### LOT이 1~2개뿐인 강종

- **1 LOT 강종(SGCC·DP590·SPCD):** 같은 강종 비교 대상이 없어 수준을 비교하지 않는다.
- **2 LOT 강종(SPCEN·SPFH590):** 비교 대상이 하나라 어느 쪽이 평소와 다른지 정할 수 없다. 작은 차이(출력 Z5·Z6·Z7, 평소 대비 1.0~1.3배)는 판단 보류로 둔다. 다만 SPCEN의 Z7·Z8(13~16배)과 출력 Z4(4.2배, 240038 70.5%는 14 LOT 중 최고)는 차이가 커서 8절에서 볼 것으로 올린다. 차이 표는 [03의 6-1](05_all_data_timeseries_jinny_03.ipynb#grade-diff)에 있다.

<a id="remaining"></a>

## 8. 남은 것 — 평소 흔들림 대비 편차 순위 → 03으로

안정 운전에서 세 가지 편차를 같은 잣대(그 신호의 평소 흔들림 대비 몇 배)로 나란히 놓는다. (1) LOT 전체 수준을 같은 강종과 비교, (2) CP가 LOT 평소에서 벗어난 크기, (3) 같은 LOT 직전 30분 대비 급변이다.

In [ ]:
# 8절: 세 가지 편차를 '그 신호의 평소 흔들림 대비 몇 배'로 바꿔 나란히 놓는다(빨강 = 볼 것).
#   (1) 수준: 01의 6절 원칙 2의 차이 ÷ LOT 안 흔들림(안정 참고 1분 값 10~90% 폭, 14 LOT 중앙값)
#   (2) CP: LOT 평소에서 가장 크게 벗어난 값 ÷ CP-Z4 1초 값 10~90% 폭(안정 참고, 14 LOT 중앙값)
#   (3) 급변: 01의 운전 상태별 급변의 배수 중 정지·승온·Z3를 뺀 LOT별 최대
lv = level_issues.assign(배수=level_issues["차이"].abs() / level_issues["LOT 안 흔들림"]).sort_values("배수")
lv["볼 것"] = lv.판정.eq("혼자 떨어짐") | lv.배수.ge(4)          # LOT 2개 강종은 4배 넘게 벌어진 것만
cp_width = np.median([(lambda v: v.quantile(.9) - v.quantile(.1))(raw[k].loc[raw[k].MINUTE.map(states[k]).eq("안정 참고"), "CP-Z4"].dropna()) for k in raw])
cpv = cp_real.groupby("LOT").DEV.agg(횟수="size", 최대=lambda s: s.loc[s.abs().idxmax()])
cpv["배수"] = cpv.최대.abs() / cp_width
cpv["볼 것"] = cpv.횟수.ge(2) | cpv.index.isin([240117])        # 240117: 정비 경계에 우연보다 가까운 유일한 CP(03의 1절)
cpv = cpv.sort_values("배수")
rest = jump.loc[jump.상태.eq("그 밖 신호 (안정 운전)")]
jv = rest.loc[rest.groupby("LOT").배수.idxmax()].set_index("LOT").sort_values("배수")

fig, axs = plt.subplots(1, 3, figsize=(30, 10.5))
fig.subplots_adjust(left=.13, right=.985, top=.75, bottom=.1, wspace=.62)
panels = [
    (axs[0], [f"{r.LOT} · {r.신호} {abs(r.차이):.1f}{r.단위}" for r in lv.itertuples()], lv.배수, lv["볼 것"],
     "(1) 같은 강종 수준 차이 ÷ LOT 안 흔들림", "점선 = LOT 안 흔들림 폭"),
    (axs[1], [f"{k} · {lot_meta.loc[k, 'GRADE']} · {r.횟수}번 · {r.최대:+.3f}" for k, r in cpv.iterrows()], cpv.배수, cpv["볼 것"],
     "(2) CP가 LOT 평소에서 벗어난 최대 ÷ CP 1초 흔들림", "점선 = CP 1초 흔들림 폭"),
    (axs[2], [f"{k} · {r.SENSOR} · {r.시각:%m/%d %H:%M}" for k, r in jv.iterrows()], jv.배수, pd.Series(False, index=jv.index),
     "(3) 안정 운전 급변 최대(정지·승온·Z3 제외)", "점선 = 안정 운전 99% 값"),
]
for ax, labels, values, pick, title, ref in panels:
    y = np.arange(len(values))
    ax.barh(y, values, color=[SELECT_COLOR if p else "#b6c0c8" for p in pick], height=.7)
    ax.set_yticks(y, labels, fontsize=12)
    for t, p in zip(ax.get_yticklabels(), pick):
        t.set_color(SELECT_COLOR if p else "#526d80"); t.set_fontweight("bold" if p else "normal")
    for yy, v in zip(y, values):
        ax.text(v * 1.03 + .05, yy, f"{v:.1f}배", va="center", fontsize=12, color="#17344d")
    ax.axvline(1, color="#455a64", ls="--", lw=1.3)
    ax.set_xlim(0, float(values.max()) * 1.3)
    style(ax, f"평소 대비[배] · {ref}"); ax.tick_params(axis="x", labelsize=12)
    ax.set_title(title, loc="left", fontsize=15, pad=14)
header(fig, "볼 것 고르기 — 평소 흔들림 대비 편차",
       "℃·%를 그대로 비교하면 원래 크게 흔들리는 Z3가 모든 순위를 차지한다 · 신호마다 평소 흔들림으로 나눈 배수로 비교 · 빨강 = 볼 것",
       [Patch(facecolor=SELECT_COLOR, label="볼 것"), Patch(facecolor="#b6c0c8", label="지켜볼·설명됨")], ncol=2)
compact_header(fig, .935, .905, title_size=24, sub_size=13, legend_size=13)
axis_tag(fig, compare_tag("평소 대비 편차"))
finish(fig, "02_근거범위/09_평소대비_편차순위.png")
table(lv.sort_values("배수", ascending=False).round(2))
table(cpv.sort_values("배수", ascending=False).round(3))
note("\n".join([
    f"- **(1) 수준:** " + ", ".join(f"{r.LOT} {r.신호} {abs(r.차이):.1f}{r.단위}({r.배수:.1f}배)" for r in lv.loc[lv['볼 것']].sort_values('배수', ascending=False).itertuples())
    + f"가 볼 것이다. 나머지는 {lv.loc[~lv['볼 것'], '배수'].max():.1f}배 이하다.",
    f"- **(2) CP:** 0.05 넘게 벗어난 펄스는 모두 CP 1초 흔들림 폭({cp_width:.4f})의 {cpv.배수.min():.0f}배 이상이라 크기만으로는 가를 수 없다. "
    f"여러 번 벗어난 LOT과 정비에 우연보다 가까운 240117을 보고, 한 번만 작게 벗어난 " + ", ".join(map(str, cpv.index[~cpv['볼 것']])) + "은 지켜본다.",
    f"- **(3) 급변:** 정지·승온·Z3를 빼면 LOT별 최대가 {jv.배수.min():.1f}~{jv.배수.max():.1f}배다. 따로 확대할 급변 구간은 없다."]))

#### 8절 해석 — 무엇을 볼까

- **(1) 수준:** 과시효 Z7·Z8의 +3℃가 LOT 안 흔들림의 13~16배로 가장 크다(240091·240121, 같은 333℃인 240114). 출력 Z4는 240038(70.5%)과 240114(53.2%)의 차이가 4.2배, 240061이 가장 가까운 SPHC보다 +9.8%p로 2.4배다. 나머지 수준 차이는 1.3배 이하다.
- **(2) CP:** 0.05 넘게 벗어난 펄스는 모두 CP 1초 흔들림의 9배 이상이라 크기만으로는 가를 수 없다. 여러 번 벗어난 240069(7번)·240037(3번)과, 정비 경계에 우연보다 가까운 유일한 240117을 본다.
- **(3) 급변:** 정지·승온·Z3를 빼면 LOT별 최대가 안정 운전 99% 값의 2배 이내다. 따로 확대할 급변 구간은 없다.

| 볼 것 | LOT · 구간 | 평소 대비 | 어디서 |
|---|---|---|---|
| CP가 여러 번 벗어남 · CP가 노 온도와 따로 흔들림 | **240069** 07/07 18:59–19:43 · 07/08 02:30–02:40 | 7번, 최대 약 79배 | [03의 2절](05_all_data_timeseries_jinny_03.ipynb#ev-240069) |
| CP가 여러 번 벗어남 | **240037** 04/13 23:20–04/14 00:00 · 04/14 03:10–03:12 | 3번, 최대 약 27배 | [03의 3절](05_all_data_timeseries_jinny_03.ipynb#ev-240037) |
| 정비 직후 Z3 이탈 → CP 하강 | **240117** 11/13 12:44–13:07 | 약 14배 · 정비에 우연보다 가까운 유일한 CP | [03의 4절](05_all_data_timeseries_jinny_03.ipynb#ev-240117) |
| 과시효 Z7·Z8 333℃ 단계 | **240091 · 240121 · 240114** | 13~16배 | [03의 6-2](05_all_data_timeseries_jinny_03.ipynb#z78) |
| 같은 온도에 출력 Z4가 다름 | **240038**(70.5%, LOT 2개 강종) · **240061**(+9.8%p) | 4.2배 · 2.4배 | [03의 6-3](05_all_data_timeseries_jinny_03.ipynb#op-240061) |
| 6월·11월 정기수리 전후의 안정 운전 수준 | 240058→240061 · 240114→240117 | 정지·승온은 빼고 비교 | [03의 7절](05_all_data_timeseries_jinny_03.ipynb#mnt-0609) |
| 판정·가설·다음 확인 기록 | 전체 | — | [04](05_all_data_timeseries_jinny_04.ipynb) |

<a id="appendix-02"></a>

## 부록 — 확인용 표

본문 판단에 쓰지 않지만 근거로 남기는 전수 표다.

<a id="relationships"></a>

### 부록 A. 온도·출력 연관성 — 확인용 표

In [ ]:
# 같은 구역 온도(TC)와 출력(OP)의 상관: 제어가 정상적으로 작동하는지 확인하는 표로만 남긴다.
corr_rows = []
for r in lots.itertuples():
    k = int(r.LOT_NO); p = profile[k]
    for z in range(1, 9):
        tc, op = f"TC-Z{z}", f"OP-Z{z}"
        valid = states[k].eq("안정 참고") & quality[k][tc].ge(.9) & quality[k][op].ge(.9)
        pair = pd.DataFrame({"TC": p[(tc, "median")].where(valid), "OP": p[(op, "median")].where(valid)}).dropna()
        d = pair.diff().dropna()
        corr_rows.append({"LOT": k, "GRADE": r.GRADE, "ZONE": f"Z{z}",
                          "수준 상관": pair.corr(method="spearman").iloc[0, 1] if len(pair) >= 30 else np.nan,
                          "1분 변화 상관": d.corr(method="spearman").iloc[0, 1] if len(d) >= 30 else np.nan})
corr_table = pd.DataFrame(corr_rows)
corr_zone = corr_table.groupby("ZONE")[["수준 상관", "1분 변화 상관"]].agg(["min", "median", "max"]).round(2)
table(corr_zone)
note(f"**결과:** 14 LOT × 8구역 모두에서 온도와 출력은 반대로 움직인다(수준 상관 {corr_table['수준 상관'].min():+.2f}~{corr_table['수준 상관'].max():+.2f}, "
     f"1분 변화 상관 {corr_table['1분 변화 상관'].min():+.2f}~{corr_table['1분 변화 상관'].max():+.2f}). 온도가 오르면 출력을 줄이는 제어가 모든 LOT·구역에서 정상적으로 작동한다는 확인이며, "
     "다르게 움직이는 LOT이 없어 따로 볼 구간은 없다. 그래서 그림 없이 표로만 남긴다.")
table(corr_table)


<a id="all-raw-evidence"></a>

### 부록 B. 원본 후보의 전수 확인

In [ ]:
raw_delta,temperature_rows={},[]
def seconds_series(k,s):
    g=raw[k];ix=np.rint(g.ELAPSED.to_numpy()*60).astype(int)
    grid=pd.RangeIndex(int(np.floor(lot_meta.loc[k,"MINUTES"]*60))+1)
    y=pd.Series(g[s].to_numpy(),index=ix).reindex(grid)
    minute=grid.to_numpy()//60
    valid=states[k].reindex(minute).to_numpy()=="안정 참고"
    valid &= quality[k][s].reindex(minute).to_numpy()>=.9
    return y.where(valid)
for e in episodes.itertuples():
    key=e.LOT,e.SENSOR
    if key not in raw_delta:
        ids=lots.loc[lots.GRADE.eq(e.GRADE),"LOT_NO"].astype(int).tolist()
        target=seconds_series(e.LOT,e.SENSOR)
        peers=pd.DataFrame({k:seconds_series(k,e.SENSOR).reindex(target.index) for k in ids if k!=e.LOT})
        n=peers.count(axis=1)
        d=(target-peers.median(axis=1)).where(n.ge(2))
        raw_delta[key]=pd.DataFrame({"DELTA":d,"N_PEERS":n})
    block=raw_delta[key].loc[int(e.START_MIN*60):int(e.END_MIN*60)-1]
    v=block.DELTA.dropna();hit=block.DELTA.abs().ge(3)
    run=runs(hit);maxrun=max([b-a for a,b in run],default=0)
    ratio=hit.sum()/len(v) if len(v) else np.nan
    temperature_rows.append({"EPISODE_ID":e.Index+1,"LOT":e.LOT,"GRADE":e.GRADE,"SENSOR":e.SENSOR,
                             "START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"DURATION_MIN":e.DURATION_MIN,
                             "RAW_COMPARE_SECONDS":len(v),"RAW_EXCEED_SECONDS":int(hit.sum()),"RAW_EXCEED_RATIO":ratio,
                             "MAX_CONTIGUOUS_RECORDED_S":maxrun,"RAW_MAX_ABS_DELTA_C":v.abs().max(),
                             "VERDICT":"원본 반복 확인" if len(v)>=60 and ratio>=.5 else "비율 부족·보류"})
temperature_validation=pd.DataFrame(temperature_rows)
window_rows=[]
for k,sub in episodes.groupby("LOT"):
    current=None
    for e in sub.sort_values(["START_MIN","END_MIN"]).itertuples():
        if current is None:
            current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
        elif e.START_MIN<=current["END_MIN"]+5:
            current["END_MIN"]=max(current["END_MIN"],e.END_MIN);current["SENSORS"].add(e.SENSOR)
        else:
            window_rows.append(current);current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
    if current is not None:window_rows.append(current)
temperature_windows=pd.DataFrame(window_rows)
note(f"**전수 결과:** {len(temperature_validation)}개 센서 후보 중 {(temperature_validation.VERDICT=='원본 반복 확인').sum()}개는 원본 비교 초의 절반 이상에서 3℃ 차이가 반복된다. 보류는 {temperature_validation.VERDICT.ne('원본 반복 확인').sum()}개다. 총 {len(temperature_windows)}개의 확대창에서 원본·해당 OP·개별 후보를 아래에 모두 보여준다.")

## 다음 — 03에서 확대·비교

8절에서 남은 것(CP 240069·240037·240117, 과시효 Z7·Z8, 출력 Z4, 정기수리 전후)은 [03 주요 이상 의심 구간 상세 분석 및 비교](05_all_data_timeseries_jinny_03.ipynb)에서 본다. 판정과 다음 확인 기록은 [04](05_all_data_timeseries_jinny_04.ipynb)에 있다.